# D2: P2 katmanı (stride 4) ile küçük hedef tespiti

**Sorun:** detector-v1'in küçük hedef recall'u düşük. E1a'da küçük (<8 px) GT kutularının %18'i IoU ≥ 0,3 ile bile eşleşmiyor; D1'de <8×8 recall ≈ 0,59 (IoU ≥ 0,5). YOLOv12l'nin en ince çıkışı P3 (stride 8); 8 px'lik bir hedef P3 grid'inde yaklaşık tek hücreye düşüyor.

**Mekanizma:** Ultralytics'in atayıcısı (TaskAlignedAssigner) yalnızca merkezi GT kutusunun **içine** düşen anchor'ları aday alır. Anchor merkezleri P3'te 8 px, P2'de 4 px aralıklıdır; kenarı stride'dan küçük bir hedefte P3'te hiç aday anchor oluşmayabilir, yani o hedef P3'te hiç pozitif örnek üretmez. E1a'daki "%18 hiç eşleşmiyor" bulgusu bununla uyumlu. Bu notebook, val GT'leri üzerinde aday anchor'ı olmayan küçük hedeflerin oranını P3 ve P2 için ayrıca ölçer (Adım 4.2, yaklaşık; augment yok).

**Amaç:** detector-v1'e stride 4'te çalışan bir P2 çıkış dalı eklemek, küçük hedef tespitini ve bunun tracker'a yansımasını ölçmek.

**Model:** detector-v1 (`common_detector_bsb-s0`; SOT → MOT fine-tune edilmiş YOLOv12l). Katman 0–20 ve Detect'in P3/P4/P5 dalları detector-v1'den **birebir** aktarılır. Yeni katmanlar:
- 21: `nn.Upsample` (P3 çıkışından, katman 14)
- 22: `Concat` (backbone P2/4, katman 2)
- 23: `A2C2f` (256 kanal; katman 14 ile aynı yapı)
- Detect'in P2 dalı (Detect artık `[23, 14, 17, 20]` girişli, 4 seviye)

P2 dalı 256 kanal seçildi; Detect'in ara kanal sayıları ilk girişe bağlı olduğundan, böylece P3–P5 dallarının ağırlık şekilleri değişmez ve aktarım kayıpsız olur.

**Protokol (D1 ile birebir):** 4 epoch, batch 16, imgsz 640, AdamW lr0 0,001, freeze 10 (backbone donuk), seed 0, aynı `dataset.yaml`, `last.pt`. Tespit: imgsz 640, conf 0,05. Tracker: SB BoT-SORT, IoU maliyeti, match_thresh 0,7, sparseOptFlow, K = 0 (D1'in karar protokolü).

**Kollar:**
- `v1_ref`: detector-v1 (fine-tune yok)
- `ctrl`: D1'in CIoU kontrol fine-tune'u (aynı epoch, lr, freeze); ağırlıklar D1'den alınır, tespit ve tracker bu oturumda yeniden koşulur
- `p2`: bu deney

**Önceden yazılmış karar kuralı** (eğitimden önce `WORK/D2/D2_prereg.json` olarak kilitlenir; sonradan değiştirilemez):
- **Birincil:** DetA farkı (p2 − ctrl), sekans düzeyinde bootstrap (B = 10.000), %95 CI. CI alt sınırı > 0 ise detector-v1-P2 kabul edilir; değilse detector-v1 kalır.
- **İkincil** (Holm düzeltmeli, kararı değiştirmez): <8 px recall, HOTA, AssA, IDSW (p2 − ctrl).
- **Bilgi:** p2 − v1_ref; dondurulmuş tracker konfigürasyonu (E2: sparseOptFlow + NWD + K10) ile HOTA ve MOTA_seq; hız (ms/kare), parametre ve GFLOPs.

**P2 başlığının başlatılması:** Detect'in P2 dalı, P3 dalının ağırlıklarıyla başlatılır (giriş kanalları aynı). P2 sınıf biası stride 4 önseline göre (`bias_init`) bırakılır. Katman 21–23 rastgele başlar. Gerekçe: 4 epoch, sıfırdan bir başlık için kısa.

**İstatistik tanımları (ön kayda yazılır):**
- Bootstrap **eşleştirilmiş**: her örnekte aynı sekanslar seçilir; fark = sekans başına (p2 − ctrl) farklarının ortalaması. Bu, D1 ve E serisinde kullanılan tanımın aynısıdır (`compare`). TrackEval'in birleşik (combined) DetA'sı, sayımlar toplanarak hesaplandığı için sekans ortalamasından farklıdır; birleşik DetA farkı ayrıca **nokta tahmini** olarak raporlanır.
- Küçük hedef: √(w·h) < 8 px (E1a ve D1 ile aynı). Recall: IoU ≥ 0,5, skor ≥ 0,09, kare başına Hungarian eşleşme. Küçük GT'si olmayan sekanslar bu karşılaştırmanın dışındadır.
- D1 tekrar üretilebilirlik toleransı: MOTA/HOTA/DetA/AssA için |Δ| ≤ 0,003; IDSW için ±%2.

**Sınırlar:**
- Tek seed; val 16 sekans (geliştirme seti). Seed varyansı CI'ya girmez. `ctrl − v1_ref` farkı, aynı mimaride fine-tune gürültüsünün kaba bir referansı olarak raporlanır.
- 4 epoch yeni katmanlar için kısa olabilir. P2 ile anchor sayısı 8.400'den ~34.000'e çıkar (640 px); çoğu negatif olduğundan sınıflandırma kaybının dengesi ve atamalar değişir. Olumsuz sonuç "P2 işe yaramaz" değil, "bu eğitim bütçesinde fayda göstermedi" anlamına gelir.
- Freeze, yalnız gradyanı durdurur; donuk backbone'daki BN running istatistikleri train modunda güncellenir. ctrl'de de aynısı geçerli olduğu için karşılaştırma adildir.

**Görsel kayıt:** boyut sınıfına göre recall, eğitim eğrileri (ctrl vs p2), sekans başına DetA farkı, küçük hedefi en çok olan sekansta ctrl ve p2 videosu (yan yana).

**Tahmini süre:** veri kurulumu 10–20 dk, eğitim yaklaşık 30–45 dk (D1: ~5 dk/epoch; P2 daha ağır), tespit + tracker + değerlendirme 30–45 dk.

**Paralel çalışma güvenliği:** Bu notebook yalnızca `WORK/D2/` altına yazar. `E0_status.json`, D1 ve E2 dosyalarını yalnızca okur. `run_patched.py` ve MLflow veritabanı yerel diskte (`/content/local`) tutulur; ana notebook ile çakışmaz.

**Çalıştırma sırası:** Adım 0 (ana notebook'ta, bir kez) → Adım 1 → 2 → 3 → 4 → 5. Hücreler yeniden çalıştırılabilir: tamamlanmış eğitim ve tespitler tekrar yapılmaz.

## Adım 0: Eğitim verisinin arşivi (ANA notebook'ta, bir kez)

D1 eğitim verisi Drive'da değil, ana Colab oturumunun yerel diskinde (`/content/work-data/bsb-95df10193a85/yolo`, 63.925 train + 1.188 val görüntüsü). Yeni bir Colab oturumunda bu klasör yoktur.

**Bu hücreyi ana notebook'a kopyalayıp orada çalıştırın** (D1'i eğittiğiniz, `/content/work-data` klasörünün bulunduğu oturum). Hücre, `dataset.yaml`'ın işaret ettiği tüm görüntü ve etiketleri Drive'a tek bir arşiv olarak yazar (`WORK/D2_inputs/yolo_data.tar` + manifest). P2 notebook'u bu arşivi **aynı yola** açar ve manifest ile birebir doğrular. Böylece D1 ile aynı veri kullanılır.

- Süre: birkaç GB, 5–15 dakika.
- Ana oturum yeniden başladıysa ve `/content/work-data` yoksa, önce ana notebook'taki veri hazırlama adımını çalıştırın.
- Arşiv güncelse hücre yeniden yazmaz.

In [ ]:
# ===== ADIM 0: BU HÜCREYİ ANA NOTEBOOK'TA ÇALIŞTIRIN (D1'i eğittiğiniz, /content/work-data'nın olduğu oturum) =====
import os, json, tarfile, hashlib, time
from pathlib import Path
import yaml

DATA_YAML_D1 = Path('/content/drive/MyDrive/tir-jde-work/runs/common_detector_bsb-s0/detector_data/dataset.yaml')
ARCHIVE_DIR = Path('/content/drive/MyDrive/tir-jde/work/D2_inputs')
ARCHIVE = ARCHIVE_DIR / 'yolo_data.tar'
MANIFEST = ARCHIVE_DIR / 'yolo_data_manifest.json'
IMG_FORMATS = {'bmp', 'dng', 'jpeg', 'jpg', 'mpo', 'png', 'tif', 'tiff', 'webp', 'pfm', 'heic'}


def yolo_split_files(data_yaml):
    """Ultralytics'in okuyacağı dosyalar: train/val görüntüleri, etiketleri ve liste (.txt) dosyaları."""
    d = yaml.safe_load(open(data_yaml))
    base = Path(d.get('path') or Path(data_yaml).parent)
    if not base.is_absolute():
        raise ValueError(f"dataset.yaml 'path' göreli: {base}. Mutlak yol bekleniyor.")
    out = {}
    for split in ('train', 'val'):
        entries = d[split] if isinstance(d[split], (list, tuple)) else [d[split]]
        imgs, lists = [], []
        for e in entries:
            p = Path(e) if Path(e).is_absolute() else base / e
            if p.is_dir():
                imgs += [str(x) for x in p.rglob('*.*')]
            elif p.is_file():
                lists.append(str(p))
                parent = str(p.parent) + os.sep
                imgs += [x.replace('./', parent) if x.startswith('./') else x
                         for x in p.read_text().strip().splitlines() if x.strip()]
            else:
                raise FileNotFoundError(f'{split} girdisi bulunamadı: {p}')
        imgs = sorted(x.strip() for x in imgs if x.strip().split('.')[-1].lower() in IMG_FORMATS)
        sa, sb = f'{os.sep}images{os.sep}', f'{os.sep}labels{os.sep}'
        labels = [sb.join(x.rsplit(sa, 1)).rsplit('.', 1)[0] + '.txt' for x in imgs]
        out[split] = {'images': imgs, 'labels': labels, 'lists': lists}
    return d, out


def manifest_of(files):
    h, total = hashlib.sha1(), 0
    for f in sorted(files):
        s = os.path.getsize(f)
        total += s
        h.update(f'{f}\t{s}\n'.encode())
    return {'n': len(files), 'bytes': total, 'sha1': h.hexdigest()}


def local_data_files(sp):
    """Drive dışındaki (yerel) görüntü, etiket ve liste dosyaları. Etiketi olmayan görüntü = arka plan (normal)."""
    files, missing = [], []
    for v in sp.values():
        for f in v['images']:
            (files if os.path.exists(f) else missing).append(f)
        files += [f for f in v['labels'] if os.path.exists(f)]
        files += v['lists']
    files = sorted({f for f in files if not f.startswith('/content/drive/')})
    return files, missing


d_yaml, sp = yolo_split_files(DATA_YAML_D1)
files, missing = local_data_files(sp)
counts = {s: {'images': len(v['images']), 'labels': sum(os.path.exists(x) for x in v['labels'])} for s, v in sp.items()}
print('dataset.yaml:', {k: d_yaml[k] for k in ('path', 'train', 'val') if k in d_yaml})
print('Sayılar:', counts, '| eksik görüntü:', len(missing))
assert not missing, f'{len(missing)} görüntü yok (örn. {missing[:3]}). Bu hücre, D1 verisinin bulunduğu ana oturumda çalışmalı.'
assert files and all(f.startswith('/content/') for f in files), 'Beklenmeyen yol: dosyalar /content altında olmalı.'

man = {'data_yaml': str(DATA_YAML_D1), 'created': time.strftime('%Y-%m-%d %H:%M:%S'), 'counts': counts,
       'files': manifest_of(files)}
print(f"Arşivlenecek: {man['files']['n']} dosya, {man['files']['bytes'] / 1e9:.2f} GB")

if ARCHIVE.exists() and MANIFEST.exists() and json.load(open(MANIFEST)).get('files') == man['files']:
    print('Arşiv zaten güncel; yeniden yazılmadı:', ARCHIVE)
else:
    ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
    tmp = ARCHIVE.with_name(ARCHIVE.name + '.part')
    t0 = time.time()
    with tarfile.open(tmp, 'w', dereference=True) as tf:   # sembolik bağlar gerçek dosya olarak yazılır
        for i, f in enumerate(files):
            tf.add(f, arcname=f.lstrip('/'), recursive=False)
            if i % 20000 == 0:
                print(f'  {i}/{len(files)} ({time.time() - t0:.0f} s)')
    os.replace(tmp, ARCHIVE)
    with open(MANIFEST, 'w') as fh:
        json.dump(man, fh, indent=1, ensure_ascii=False)
    print(f'Arşiv yazıldı: {ARCHIVE} ({time.time() - t0:.0f} s)')
print('Manifest:', MANIFEST)

## Adım 1: Kurulum (bu notebook)

**Runtime:** GPU (D1 ile aynı sınıf tercih edilir). Hücreleri sırayla çalıştırın.

- **1.1** Ayarlar ve yollar. Hiperparametreler D1 eğitim logundan alındı; değiştirmeyin.
- **1.2** Paketler, YOLOv12 fork'u (`a97bba3`) ve NumPy 2 yaması (`np.trapz` → `np.trapezoid`; D1'de yaşanan doğrulama hatası).
- **1.3** Val girdileri: detector-v1 val (dets/gt/val.txt), val kareleri (yalnız 16 val sekansı Train.zip'ten açılır), tracker betikleri.
- **1.4** Eğitim verisi: Adım 0 arşivinin açılması ve manifest ile doğrulama.
- **1.5** Ortak kütüphane (ana notebook 0.3 ile birebir).
- **1.6** E0-c kapısı ve ön kayıt (karar kuralının kilitlenmesi).

In [ ]:
# 1.1) Ayarlar, yollar ve Drive
import os, sys, json, glob, time, shutil, hashlib, subprocess, re, zipfile, tarfile, base64, copy, math
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/tir-jde')
WORK = BASE / 'work'                      # ana notebook ile aynı; bu notebook yalnız WORK/D2 altına yazar
D2_DIR = WORK / 'D2'
LOCAL = Path('/content/local')
for _p in (D2_DIR, LOCAL):
    _p.mkdir(parents=True, exist_ok=True)

# Model ve veri (D1 ile birebir aynı)
DETECTOR_WEIGHTS = Path('/content/drive/MyDrive/tir-jde-work/runs/common_detector_bsb-s0/detector/train/weights/best.pt')
DATA_YAML = Path('/content/drive/MyDrive/tir-jde-work/runs/common_detector_bsb-s0/detector_data/dataset.yaml')
D1_DIR = WORK / 'D1'
D1_CTRL_WEIGHTS = D1_DIR / 'ctrl' / 'weights' / 'last.pt'
D1_CTRL_ARGS = D1_DIR / 'ctrl' / 'args.yaml'
D1_CTRL_RESULTS = D1_DIR / 'ctrl' / 'results.csv'
FROZEN_CONFIG = WORK / 'E2' / 'frozen_config.json'

# D1 hiperparametreleri (D1 eğitim logundan; değiştirmeyin)
EPOCHS, IMGSZ, BATCH, LR0, OPT, FREEZE, SEED = 4, 640, 16, 0.001, 'AdamW', 10, 0
DET_CONF = 0.05                           # D1 detect_full_model ile aynı

# Kod sürümü
SB_REPO_URL = 'https://github.com/wish44165/YOLOv12-BoT-SORT-ReID'
SB_COMMIT = 'a97bba3252814543a48286b74ef182e731f38221'
SB_ROOT = Path('/content/baseline_src/YOLOv12-BoT-SORT-ReID')
os.environ['SB_BOTSORT'] = str(SB_ROOT / 'BoT-SORT')
YOLO_FORK = SB_ROOT / 'BoT-SORT' / 'yolov12'

# Girdiler
DATA_ARCHIVE = WORK / 'D2_inputs' / 'yolo_data.tar'
DATA_MANIFEST = WORK / 'D2_inputs' / 'yolo_data_manifest.json'
TRAIN_ZIP_CANDIDATES = [BASE / 'datasets/BSB/Train.zip', Path('/content/drive/MyDrive/datasets/BSB/Train.zip')]
DV1_CANDIDATES = [BASE / 'inputs/detector-v1_val.zip', Path('/content/drive/MyDrive/tir-jde-work/detections/detector-v1_val')]
SCRIPTS_OVERRIDE = None                   # track_7a.py ve trackeval_runner.py'nin Drive klasörü (biliyorsanız yazın)
SCRIPTS_CANDIDATES = [BASE / 'scripts', BASE / 'inputs', BASE / 'inputs/scripts', BASE,
                      Path('/content/drive/MyDrive/tir-jde-work/scripts'), Path('/content/drive/MyDrive/tir-jde-work')]
VAL_FRAMES_TOTAL = 11868                  # val 16 sekans toplam kare (E0-E2 raporu)

# D1 sonuçları (Confluence D1 sayfası): tekrar üretilebilirlik kontrolü için
D1_REF = {'v1_ref': {'MOTA': 0.6990, 'HOTA': 0.5108, 'DetA': 0.5907, 'AssA': 0.4447, 'IDSW': 2383},
          'ctrl':   {'MOTA': 0.6976, 'HOTA': 0.5189, 'DetA': 0.5941, 'AssA': 0.4560, 'IDSW': 2127}}
REPRO_TOL = 0.003

FORCE_RETRAIN = False                     # True: tamamlanmış p2 eğitimini silmeden baştan eğitir

_need = [(DETECTOR_WEIGHTS, 'detector-v1 best.pt'), (DATA_YAML, 'dataset.yaml'), (D1_CTRL_WEIGHTS, 'D1 ctrl last.pt'),
         (D1_CTRL_ARGS, 'D1 ctrl args.yaml'), (D1_CTRL_RESULTS, 'D1 ctrl results.csv'),
         (WORK / 'E0_status.json', 'E0_status.json')]
for _p, _ad in _need:
    print(('OK   ' if _p.exists() else 'YOK  ') + f'{_ad}: {_p}')
_eksik_girdi = [a for p, a in _need if not p.exists()]
assert not _eksik_girdi, f'Eksik girdi: {_eksik_girdi}'
print('\nFROZEN_CONFIG:', 'var' if FROZEN_CONFIG.exists() else 'yok (E2 sabitleri kullanılacak)')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=False)

In [ ]:
# 1.2) Paketler, YOLOv12 fork'u (a97bba3) ve NumPy 2 uyumluluk yaması
%pip install -q "git+https://github.com/JonathonLuiten/TrackEval.git" lap py-cpuinfo ultralytics-thop

if not (SB_ROOT / '.git').exists():
    SB_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '-q', SB_REPO_URL, str(SB_ROOT)], check=True)
subprocess.run(['git', '-C', str(SB_ROOT), 'checkout', '-q', SB_COMMIT], check=True)
_head = subprocess.run(['git', '-C', str(SB_ROOT), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert _head == SB_COMMIT, f'Fork commit farklı: {_head}'

import numpy as np
METRICS_PY = YOLO_FORK / 'ultralytics' / 'utils' / 'metrics.py'
_src = METRICS_PY.read_text()
if not hasattr(np, 'trapz') and 'np.trapz(' in _src:
    assert _src.count('np.trapz(') == 1, 'metrics.py içinde beklenmeyen sayıda np.trapz'
    METRICS_PY.write_text(_src.replace('np.trapz(', 'np.trapezoid('))
    print('metrics.py yamalandı: np.trapz -> np.trapezoid')
assert hasattr(np, 'trapz') or 'np.trapz(' not in METRICS_PY.read_text()

# Doğru ultralytics: fork (stok paket değil)
for _k in [k for k in list(sys.modules) if k == 'ultralytics' or k.startswith('ultralytics.')]:
    del sys.modules[_k]
if str(YOLO_FORK) not in sys.path:
    sys.path.insert(0, str(YOLO_FORK))
import torch
import ultralytics
from ultralytics import YOLO
assert str(YOLO_FORK) in ultralytics.__file__, f'Yanlış ultralytics: {ultralytics.__file__}'
assert ultralytics.__version__ == '8.3.63', ultralytics.__version__
assert torch.cuda.is_available(), 'GPU yok: Runtime > Change runtime type > GPU'
print('ultralytics:', ultralytics.__file__, ultralytics.__version__)
print('torch:', torch.__version__, '| numpy:', np.__version__, '| GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 1.3) Val girdileri: detector-v1 val (dets/gt/val.txt), val kareleri, tracker betikleri

# (a) detector-v1 val klasörü
def find_dv1():
    for c in DV1_CANDIDATES:
        if c.suffix == '.zip' and c.exists():
            dest = LOCAL / 'detv1'
            if not (dest.exists() and any(dest.rglob('val.txt'))):
                dest.mkdir(parents=True, exist_ok=True)
                zipfile.ZipFile(c).extractall(dest)
            roots = [p.parent for p in dest.rglob('val.txt') if (p.parent / 'gt').is_dir()]
        elif c.is_dir():
            roots = [c] if (c / 'val.txt').exists() and (c / 'gt').is_dir() else []
        else:
            continue
        if len(roots) == 1:
            return roots[0], c
        if len(roots) > 1:
            raise RuntimeError(f'Birden fazla detector-v1 val kökü: {roots}')
    raise FileNotFoundError(f'detector-v1 val bulunamadı. Adaylar: {DV1_CANDIDATES}')

DV1, DV1_SRC = find_dv1()
VAL_TXT = DV1 / 'val.txt'
VAL_SEQS = [l.strip() for l in open(VAL_TXT) if l.strip() and not l.startswith('#')]
assert len(VAL_SEQS) == 16, f'Val sekans sayısı 16 olmalı: {len(VAL_SEQS)}'
assert all((DV1 / 'gt' / f'{s}.txt').exists() for s in VAL_SEQS), 'Bazı val GT dosyaları yok'
print('DV1:', DV1, '(kaynak:', DV1_SRC, ')')

# (b) val kareleri: yalnız 16 val sekansı Train.zip'ten açılır
FRAMES_ROOT = LOCAL / 'BSB' / 'TrainImages'
_need = [s for s in VAL_SEQS if len(glob.glob(f'{FRAMES_ROOT}/{s}/*.jpg')) == 0]
if _need:
    TRAIN_ZIP = next((p for p in TRAIN_ZIP_CANDIDATES if p.exists()), None)
    assert TRAIN_ZIP is not None, f'Train.zip bulunamadı. Adaylar: {TRAIN_ZIP_CANDIDATES}'
    _pat = re.compile(r'(?:^|/)TrainImages/([^/]+)/([^/]+\.jpg)$')
    with zipfile.ZipFile(TRAIN_ZIP) as zf:
        _members = [(n, m.group(1), m.group(2)) for n in zf.namelist()
                    for m in [_pat.search(n)] if m and m.group(1) in _need]
        assert _members, f'Train.zip içinde TrainImages/<seq>/*.jpg yok. İlk adlar: {zf.namelist()[:5]}'
        for n, seq, fn in _members:
            dst = FRAMES_ROOT / seq / fn
            dst.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(n) as src, open(dst, 'wb') as out:
                shutil.copyfileobj(src, out)
    print('Train.zip:', TRAIN_ZIP, '| açılan kare:', len(_members))

_n_frames = {s: len(glob.glob(f'{FRAMES_ROOT}/{s}/*.jpg')) for s in VAL_SEQS}
for s in VAL_SEQS:
    _g = np.loadtxt(DV1 / 'gt' / f'{s}.txt', delimiter=',', ndmin=2)
    assert _n_frames[s] >= int(_g[:, 0].max()), f'{s}: kare sayısı ({_n_frames[s]}) GT son karesinden az'
if sum(_n_frames.values()) != VAL_FRAMES_TOTAL:
    print(f'UYARI: val kare toplamı {sum(_n_frames.values())}, raporlanan {VAL_FRAMES_TOTAL}. '
          'Ana notebook farklı bir kare kaynağı kullanıyor olabilir; 4.2\'deki D1 tekrar üretilebilirlik kontrolüne bakın.')
print('Val kareleri:', sum(_n_frames.values()), '(16 sekans) |', FRAMES_ROOT)

# (c) tracker betikleri: Drive'da varsa o (ana notebook'un kullandığı), yoksa gömülü kopya (9 Kasım sürümü)
EMBEDDED_SCRIPTS = {
    'track_7a.py': ('e4fde2fcb79a62a49855cdce18822d18', 'IiIiRGVuZXkgNWEgLyA3YTogU0InbmluIEJvVC1TT1JUJ3UgKHRyYWNrZXIubWNfYm90X3NvcnQsIHByZWRpY3RfdHJhY2szIGlsZSBheW5pKSBkb2t1bG11cyBkZXRlY3Rvci12MSB0ZXNwaXRsZXJpbmRlbi4KCi0gSGVyIHNla2FucyBpY2luIHllbmkgdHJhY2tlcjsgMS4ga2FyZWRlIEdUIGt1dHVsYXJpIFt4MSx5MSx4Mix5MiwxLDBdIChTQidkZWtpIGdpYmkgdGVzcGl0IHllcmluZSkuCi0gU29ucmFraSBrYXJlbGVyZGUgdGVzcGl0bGVyIHNjb3JlID4gY29uZl90aHJlcyAoU0I6IE5NUyBjb25mIDAuMDkpLiBUZXNwaXRzaXoga2FyZSBTQidkZWtpIGdpYmkgYXRsYW5pci4KLSAtLWVtYl9kaXIgdmVyaWxpcnNlIHdpdGhfcmVpZDogU0InbmluIEZhc3RSZUlEJ2kgeWVyaW5lIGJpemltIDdhIHZla3RvcmxlcmltaXogKGt1dHV5bGEgZXNsZXNlbiBzYXRpcikuCi0gQ2lrdGk6IDxvdXQ+LzxzZXE+LnR4dCAgImthcmUsaWQseCx5LHcsaCxza29yLC0xLC0xLC0xIiAgKyAgPG91dD4vcnVuLmpzb24gKGF5YXJsYXIsIEZQUykuCiIiIgppbXBvcnQgYXJncGFyc2UsIGdsb2IsIGpzb24sIG9zLCBzeXMsIHRpbWUsIHR5cGVzCmltcG9ydCBudW1weSBhcyBucAoKCmNsYXNzIEJhbms6CiAgICBib3hlcyA9IG5wLnplcm9zKCgwLCA0KSk7IGVtYnMgPSBucC56ZXJvcygoMCwgMTI4KSwgbnAuZmxvYXQzMikKCgpkZWYgaW5zdGFsbF9zdHVicygpOgogICAgIiIiZmFzdF9yZWlkJ2kgKGZhc3RyZWlkIGJhZ2ltbGlsaWtsYXJpKSB2ZSBnZXJla2lyc2UgY3l0aG9uX2Jib3gnaSB5dWtsZW1lZGVuIHRyYWNrZXInaSBpY2UgYWt0YXJtYWsgaWNpbi4iIiIKICAgIGNsYXNzIEZhc3RSZUlESW50ZXJmYWNlOgogICAgICAgIGRlZiBfX2luaXRfXyhzZWxmLCAqYSwgKiprKToKICAgICAgICAgICAgcGFzcwoKICAgICAgICBkZWYgaW5mZXJlbmNlKHNlbGYsIGltZywgZGV0cyk6CiAgICAgICAgICAgIGRldHMgPSBucC5hc2FycmF5KGRldHMsIGR0eXBlPW5wLmZsb2F0NjQpLnJlc2hhcGUoLTEsIDQpCiAgICAgICAgICAgIGlmIGxlbihkZXRzKSA9PSAwOgogICAgICAgICAgICAgICAgcmV0dXJuIG5wLnplcm9zKCgwLCBCYW5rLmVtYnMuc2hhcGVbMV0pLCBucC5mbG9hdDMyKQogICAgICAgICAgICBkID0gbnAuYWJzKGRldHNbOiwgTm9uZSwgOl0gLSBCYW5rLmJveGVzW05vbmUsIDosIDo0XSkubWF4KDIpCiAgICAgICAgICAgIGogPSBkLmFyZ21pbigxKQogICAgICAgICAgICBpZiBub3QgKGRbbnAuYXJhbmdlKGxlbihqKSksIGpdIDwgMWUtNCkuYWxsKCk6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoInRlc3BpdCBrdXR1c3UgdmVrdG9yIGJhbmthc2luZGEgeW9rIikKICAgICAgICAgICAgcmV0dXJuIEJhbmsuZW1ic1tqXQoKICAgIHBrZyA9IHR5cGVzLk1vZHVsZVR5cGUoImZhc3RfcmVpZCIpOyBtb2QgPSB0eXBlcy5Nb2R1bGVUeXBlKCJmYXN0X3JlaWQuZmFzdF9yZWlkX2ludGVyZmVjZSIpCiAgICBtb2QuRmFzdFJlSURJbnRlcmZhY2UgPSBGYXN0UmVJREludGVyZmFjZTsgcGtnLmZhc3RfcmVpZF9pbnRlcmZlY2UgPSBtb2QKICAgIHN5cy5tb2R1bGVzWyJmYXN0X3JlaWQiXSA9IHBrZzsgc3lzLm1vZHVsZXNbImZhc3RfcmVpZC5mYXN0X3JlaWRfaW50ZXJmZWNlIl0gPSBtb2QKICAgIHRyeToKICAgICAgICBpbXBvcnQgY3l0aG9uX2Jib3ggICMgbm9xYTogRjQwMQogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIGRlZiBiYm94X292ZXJsYXBzKGEsIGIpOgogICAgICAgICAgICBhID0gbnAuYXNhcnJheShhLCBmbG9hdCk7IGIgPSBucC5hc2FycmF5KGIsIGZsb2F0KQogICAgICAgICAgICBpdyA9IG5wLmNsaXAobnAubWluaW11bShhWzosIE5vbmUsIDJdLCBiW05vbmUsIDosIDJdKSAtIG5wLm1heGltdW0oYVs6LCBOb25lLCAwXSwgYltOb25lLCA6LCAwXSkgKyAxLCAwLCBOb25lKQogICAgICAgICAgICBpaCA9IG5wLmNsaXAobnAubWluaW11bShhWzosIE5vbmUsIDNdLCBiW05vbmUsIDosIDNdKSAtIG5wLm1heGltdW0oYVs6LCBOb25lLCAxXSwgYltOb25lLCA6LCAxXSkgKyAxLCAwLCBOb25lKQogICAgICAgICAgICBpbnRlciA9IGl3ICogaWgKICAgICAgICAgICAgYWEgPSAoKGFbOiwgMl0gLSBhWzosIDBdICsgMSkgKiAoYVs6LCAzXSAtIGFbOiwgMV0gKyAxKSlbOiwgTm9uZV0KICAgICAgICAgICAgYmIgPSAoKGJbOiwgMl0gLSBiWzosIDBdICsgMSkgKiAoYls6LCAzXSAtIGJbOiwgMV0gKyAxKSlbTm9uZSwgOl0KICAgICAgICAgICAgcmV0dXJuIGludGVyIC8gKGFhICsgYmIgLSBpbnRlcikKICAgICAgICBtID0gdHlwZXMuTW9kdWxlVHlwZSgiY3l0aG9uX2Jib3giKTsgbS5iYm94X292ZXJsYXBzID0gYmJveF9vdmVybGFwczsgc3lzLm1vZHVsZXNbImN5dGhvbl9iYm94Il0gPSBtCgoKZGVmIGxvYWQocCwgY29scyk6CiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMocCkgb3Igb3MucGF0aC5nZXRzaXplKHApID09IDA6CiAgICAgICAgcmV0dXJuIG5wLnplcm9zKCgwLCBjb2xzKSkKICAgIHJldHVybiBucC5sb2FkdHh0KHAsIGRlbGltaXRlcj0iLCIsIG5kbWluPTIpCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWJvdF9yb290IiwgcmVxdWlyZWQ9VHJ1ZSwgaGVscD0iLi4uL0JvVC1TT1JUIChTQiBrb2R1KSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGF0YV9kaXIiLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJkZXRlY3Rvci12MV92YWw6IGRldHMvLCBndC8sIHZhbC50eHQiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWltYWdlcyIsIGRlZmF1bHQ9Ii9jb250ZW50L2RhdGEvQlNCL1RyYWluSW1hZ2VzIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1lbWJfZGlyIiwgZGVmYXVsdD0iIiwgaGVscD0iYm9zOiA1YSAoUmVJRCBrYXBhbGkpOyBkb2x1OiA3YSB2ZWt0b3JsZXJpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWNtYyIsIGRlZmF1bHQ9InNwYXJzZU9wdEZsb3ciLCBoZWxwPSJzcGFyc2VPcHRGbG93IHwgbm9uZSB8IG9yYiB8IGVjYyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY29uZl90aHJlcyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wOSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10cmFja19oaWdoX3RocmVzaCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4zKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRyYWNrX2xvd190aHJlc2giLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMDUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbmV3X3RyYWNrX3RocmVzaCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC40KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRyYWNrX2J1ZmZlciIsIHR5cGU9aW50LCBkZWZhdWx0PTYwLCBoZWxwPSJTQiBSRUFETUUnZGVraSBUcmFjayAzIGtvbXV0dTogNjAiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1hdGNoX3RocmVzaCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC43KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXByb3hpbWl0eV90aHJlc2giLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuNSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1hcHBlYXJhbmNlX3RocmVzaCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4yNSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1taW5fYm94X2FyZWEiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTEwKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1vdDIwIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwgaGVscD0iU0IgLS1mdXNlLXNjb3JlICh2YXJzYXlpbGFuIGthcGFsaSkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNlcXMiLCBkZWZhdWx0PSIiLCBoZWxwPSJ2aXJndWxsZTsgYm9zID0gdmFsLnR4dCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY29zdF9tb2RlIiwgZGVmYXVsdD0iaW91IiwgaGVscD0iaW91IHwgbndkIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1ud2RfQyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MTIuMCwgaGVscD0iTldEIHNhYml0aSAocHgpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jb2FzdF9rIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MCwgaGVscD0ia2F5aXAgaXogdGFobWluaSBrYXJlIHNheWlzaSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY29uZmlybV9vbmx5IiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1rZl9xIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0xLjApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0ta2ZfciIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MS4wKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWdtY19kb3duc2NhbGUiLCB0eXBlPWludCwgZGVmYXVsdD0yLCBoZWxwPSJHTUMgZG93bnNjYWxlOiAxIHZleWEgMiIpCiAgICBhID0gYXAucGFyc2VfYXJncygpCgogICAgaW5zdGFsbF9zdHVicygpCiAgICBzeXMucGF0aC5pbnNlcnQoMCwgYS5ib3Rfcm9vdCkKICAgIGltcG9ydCBjdjIKICAgIGZyb20gdHJhY2tlci5tY19ib3Rfc29ydCBpbXBvcnQgQm9UU09SVAogICAgZnJvbSB0cmFja2VyIGltcG9ydCBtYXRjaGluZyBhcyBfbTsgX20uQ09TVFsibW9kZSJdID0gYS5jb3N0X21vZGU7IF9tLkNPU1RbIkMiXSA9IGEubndkX0MKCiAgICB0YXJncyA9IHR5cGVzLlNpbXBsZU5hbWVzcGFjZSgKICAgICAgICB0cmFja19oaWdoX3RocmVzaD1hLnRyYWNrX2hpZ2hfdGhyZXNoLCB0cmFja19sb3dfdGhyZXNoPWEudHJhY2tfbG93X3RocmVzaCwgbmV3X3RyYWNrX3RocmVzaD1hLm5ld190cmFja190aHJlc2gsCiAgICAgICAgdHJhY2tfYnVmZmVyPWEudHJhY2tfYnVmZmVyLCBtYXRjaF90aHJlc2g9YS5tYXRjaF90aHJlc2gsIHByb3hpbWl0eV90aHJlc2g9YS5wcm94aW1pdHlfdGhyZXNoLAogICAgICAgIGFwcGVhcmFuY2VfdGhyZXNoPWEuYXBwZWFyYW5jZV90aHJlc2gsIG1vdDIwPWEubW90MjAsIGNtY19tZXRob2Q9YS5jbWMsIHdpdGhfcmVpZD1ib29sKGEuZW1iX2RpciksCiAgICAgICAgZmFzdF9yZWlkX2NvbmZpZz0iIiwgZmFzdF9yZWlkX3dlaWdodHM9IiIsIGRldmljZT0iY3B1IiwgbmFtZT0iIiwgYWJsYXRpb249RmFsc2UpCgogICAgc2VxcyA9IGEuc2Vxcy5zcGxpdCgiLCIpIGlmIGEuc2VxcyBlbHNlIFtsLnN0cmlwKCkgZm9yIGwgaW4gb3Blbihvcy5wYXRoLmpvaW4oYS5kYXRhX2RpciwgInZhbC50eHQiKSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmIGwuc3RyaXAoKSBhbmQgbm90IGwuc3RhcnRzd2l0aCgiIyIpXQogICAgb3MubWFrZWRpcnMoYS5vdXQsIGV4aXN0X29rPVRydWUpCiAgICB0X3VwZCwgbl9mcmFtZXMsIHBlcl9zZXEgPSAwLjAsIDAsIHt9CiAgICBmb3IgcyBpbiBzZXFzOgogICAgICAgIGltZ3MgPSBzb3J0ZWQoZ2xvYi5nbG9iKG9zLnBhdGguam9pbihhLmltYWdlcywgcywgIiouanBnIikpKQogICAgICAgIGRldHMgPSBsb2FkKG9zLnBhdGguam9pbihhLmRhdGFfZGlyLCAiZGV0cyIsIHMgKyAiLnR4dCIpLCA2KQogICAgICAgIGd0ID0gbG9hZChvcy5wYXRoLmpvaW4oYS5kYXRhX2RpciwgImd0IiwgcyArICIudHh0IiksIDYpCiAgICAgICAgZzEgPSBndFtndFs6LCAwXSA9PSAxXQogICAgICAgIHByaW9yID0gbnAuc3RhY2soW2cxWzosIDJdLCBnMVs6LCAzXSwgZzFbOiwgMl0gKyBnMVs6LCA0XSwgZzFbOiwgM10gKyBnMVs6LCA1XSwKICAgICAgICAgICAgICAgICAgICAgICAgICBucC5vbmVzKGxlbihnMSkpLCBucC56ZXJvcyhsZW4oZzEpKV0sIDEpIGlmIGxlbihnMSkgZWxzZSBucC56ZXJvcygoMCwgNikpCiAgICAgICAgaWYgYS5lbWJfZGlyOgogICAgICAgICAgICB6ID0gbnAubG9hZChvcy5wYXRoLmpvaW4oYS5lbWJfZGlyLCBzICsgIi5ucHoiKSkKICAgICAgICAgICAgZGV0X2VtYiwgZ3QxX2VtYiA9IHpbImRldF9lbWIiXS5hc3R5cGUobnAuZmxvYXQzMiksIHpbImd0MV9lbWIiXS5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICAgICAgYXNzZXJ0IGxlbihkZXRfZW1iKSA9PSBsZW4oZGV0cykgYW5kIGxlbihndDFfZW1iKSA9PSBsZW4ocHJpb3IpLCBmIntzfTogdmVrdG9yIHNheWlzaSB1eW11eW9yIgogICAgICAgIGJ5X2ZyYW1lID0ge30KICAgICAgICBmb3IgaSwgZiBpbiBlbnVtZXJhdGUoZGV0c1s6LCAwXS5hc3R5cGUoaW50KSk6CiAgICAgICAgICAgIGJ5X2ZyYW1lLnNldGRlZmF1bHQoZiwgW10pLmFwcGVuZChpKQoKICAgICAgICB0cmFja2VyID0gQm9UU09SVCh0YXJncywgZnJhbWVfcmF0ZT0zMC4wKTsgZnJvbSB0cmFja2VyLmdtYyBpbXBvcnQgR01DOyB0cmFja2VyLmdtYyA9IEdNQyhtZXRob2Q9YS5jbWMsIGRvd25zY2FsZT1hLmdtY19kb3duc2NhbGUpCiAgICAgICAgdHJhY2tlci5jb2FzdF9rID0gYS5jb2FzdF9rCiAgICAgICAgdHJhY2tlci5jb25maXJtX29ubHkgPSBhLmNvbmZpcm1fb25seQogICAgICAgIGZyb20gdHJhY2tlciBpbXBvcnQga2FsbWFuX2ZpbHRlciBhcyBfa2Y7IF9rZi5LRl9TQ0FMRVsiUSJdID0gYS5rZl9xOyBfa2YuS0ZfU0NBTEVbIlIiXSA9IGEua2ZfcgogICAgICAgIGxpbmVzLCB0cyA9IFtdLCB0aW1lLnRpbWUoKQogICAgICAgIGZvciBmIGluIHJhbmdlKDEsIGxlbihpbWdzKSArIDEpOgogICAgICAgICAgICBpZiBmID09IDE6CiAgICAgICAgICAgICAgICBvdXQgPSBwcmlvci5jb3B5KCk7IGVtYiA9IGd0MV9lbWIgaWYgYS5lbWJfZGlyIGVsc2UgTm9uZQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcm93cyA9IG5wLmFycmF5KGJ5X2ZyYW1lLmdldChmLCBbXSksIGludCkKICAgICAgICAgICAgICAgIHJvd3MgPSByb3dzW2RldHNbcm93cywgNV0gPiBhLmNvbmZfdGhyZXNdIGlmIGxlbihyb3dzKSBlbHNlIHJvd3MKICAgICAgICAgICAgICAgIG91dCA9IG5wLmNvbmNhdGVuYXRlKFtkZXRzW3Jvd3MsIDE6Nl0sIG5wLnplcm9zKChsZW4ocm93cyksIDEpKV0sIDEpIGlmIGxlbihyb3dzKSBlbHNlIG5wLnplcm9zKCgwLCA2KSkKICAgICAgICAgICAgICAgIGVtYiA9IGRldF9lbWJbcm93c10gaWYgYS5lbWJfZGlyIGVsc2UgTm9uZQogICAgICAgICAgICBpZiBsZW4ob3V0KSA9PSAwOgogICAgICAgICAgICAgICAgY29udGludWUgICMgU0I6IHByZWQgYm9zc2EgdHJhY2tlci51cGRhdGUgY2FncmlsbWF6CiAgICAgICAgICAgIGltZyA9IGN2Mi5pbXJlYWQoaW1nc1tmIC0gMV0pIGlmIGEuY21jICE9ICJub25lIiBlbHNlIG5wLnplcm9zKCg1MTIsIDY0MCwgMyksIG5wLnVpbnQ4KQogICAgICAgICAgICBpZiBhLmVtYl9kaXI6CiAgICAgICAgICAgICAgICBCYW5rLmJveGVzLCBCYW5rLmVtYnMgPSBvdXRbOiwgOjRdLCBlbWIKICAgICAgICAgICAgdDAgPSB0aW1lLnRpbWUoKQogICAgICAgICAgICB0cmFja3MsIF8gPSB0cmFja2VyLnVwZGF0ZShvdXQsIGltZykKICAgICAgICAgICAgdF91cGQgKz0gdGltZS50aW1lKCkgLSB0MDsgbl9mcmFtZXMgKz0gMQogICAgICAgICAgICBmb3IgdCBpbiB0cmFja3M6CiAgICAgICAgICAgICAgICB4LCB5LCB3LCBoID0gdC50bHdoCiAgICAgICAgICAgICAgICBpZiB3ICogaCA+IGEubWluX2JveF9hcmVhOgogICAgICAgICAgICAgICAgICAgIGxpbmVzLmFwcGVuZChmIntmfSx7dC50cmFja19pZH0se3g6LjJmfSx7eTouMmZ9LHt3Oi4yZn0se2g6LjJmfSx7dC5zY29yZTouMmZ9LC0xLC0xLC0xXG4iKQogICAgICAgIG9wZW4ob3MucGF0aC5qb2luKGEub3V0LCBzICsgIi50eHQiKSwgInciKS53cml0ZWxpbmVzKGxpbmVzKQogICAgICAgIHBlcl9zZXFbc10gPSB7ImthcmUiOiBsZW4oaW1ncyksICJzYXRpciI6IGxlbihsaW5lcyksICJzbiI6IHJvdW5kKHRpbWUudGltZSgpIC0gdHMsIDEpfQogICAgICAgIHByaW50KGYie3N9OiB7bGVuKGltZ3MpfSBrYXJlLCB7bGVuKGxpbmVzKX0gc2F0aXIsIHt0aW1lLnRpbWUoKSAtIHRzOi4xZn0gc24iLCBmbHVzaD1UcnVlKQoKICAgIHJlYyA9IHsiYXlhcmxhciI6IHZhcnMoYSksICJ0cmFja2VyX2ZwcyI6IG5fZnJhbWVzIC8gbWF4KHRfdXBkLCAxZS05KSwgInNla2Fuc2xhciI6IHBlcl9zZXF9CiAgICBqc29uLmR1bXAocmVjLCBvcGVuKG9zLnBhdGguam9pbihhLm91dCwgInJ1bi5qc29uIiksICJ3IiksIGluZGVudD0xKQogICAgcHJpbnQoZiJbYml0dGldIHRyYWNrZXIgRlBTICh5YWxuaXogdXBkYXRlLCBHTUMgZGFoaWwpOiB7cmVjWyd0cmFja2VyX2ZwcyddOi4xZn0iLCBmbHVzaD1UcnVlKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'),
    'trackeval_runner.py': ('179349de9e01aaa31505488658e73cd7', 'IiIiT25lIGV2YWx1YXRvciBmb3IgZXZlcnkgbW9kZWw6IFRyYWNrRXZhbCAoSE9UQSwgQ0xFQVIsIElkZW50aXR5KSBvbiB0aGUgc2FtZSBzcGxpdC4KCklucHV0cyBhcmUgcGxhaW4gTU9UQ2hhbGxlbmdlIHRleHQgZmlsZXMsIG9uZSBwZXIgc2VxdWVuY2U6OgoKICAgIGd0X2Rpci88c2VxPi50eHQgICAgIGZyYW1lLCBpZCwgeCwgeSwgdywgaCwgMSwgMSwgMS4wICAgKCsgPHNlcT4ubGVuIHdpdGggdGhlIGZyYW1lIGNvdW50KQogICAgcHJlZF9kaXIvPHNlcT4udHh0ICAgZnJhbWUsIGlkLCB4LCB5LCB3LCBoLCBzY29yZSwgLTEsIC0xLCAtMQoKVGhleSBhcmUgbGFpZCBvdXQgdGhlIHdheSBUcmFja0V2YWwncyBNb3RDaGFsbGVuZ2UyREJveCByZWFkZXIgZXhwZWN0cyBpbiBhCnRlbXBvcmFyeSBmb2xkZXIsIHNvIG1vZGVscyBuZXZlciBuZWVkIHRvIGtub3cgYWJvdXQgVHJhY2tFdmFsLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0ZW1wZmlsZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKX0NPTUJJTkVEID0gIkNPTUJJTkVEX1NFUSIKX0NMQVNTID0gInBlZGVzdHJpYW4iICAjIFRyYWNrRXZhbCdzIE1PVCBjbGFzcyBpZCAxOyBvdXIgVUFWIGxhYmVscyB1c2UgY2xhc3MgMQoKCmRlZiBfbnVtcHlfY29tcGF0KCkgLT4gTm9uZToKICAgICIiIlRyYWNrRXZhbCBzdGlsbCB1c2VzIGFsaWFzZXMgcmVtb3ZlZCBpbiBOdW1QeSAxLjI0IChucC5mbG9hdCwgbnAuaW50LCBucC5ib29sKS4iIiIKICAgIGZvciBuYW1lLCB0eXAgaW4gKCgiZmxvYXQiLCBmbG9hdCksICgiaW50IiwgaW50KSwgKCJib29sIiwgYm9vbCkpOgogICAgICAgIGlmIG5hbWUgbm90IGluIG5wLl9fZGljdF9fOgogICAgICAgICAgICBzZXRhdHRyKG5wLCBuYW1lLCB0eXApCgoKZGVmIF9zZXFfbGVuZ3RoKGd0X2RpcjogUGF0aCwgc2VxOiBzdHIpIC0+IGludDoKICAgIGxlbmd0aF9maWxlID0gZ3RfZGlyIC8gZiJ7c2VxfS5sZW4iCiAgICBpZiBsZW5ndGhfZmlsZS5leGlzdHMoKToKICAgICAgICByZXR1cm4gaW50KGxlbmd0aF9maWxlLnJlYWRfdGV4dCgpLnN0cmlwKCkpCiAgICBmcmFtZXMgPSBucC5sb2FkdHh0KGd0X2RpciAvIGYie3NlcX0udHh0IiwgZGVsaW1pdGVyPSIsIiwgbmRtaW49MilbOiwgMF0KICAgIHJldHVybiBpbnQoZnJhbWVzLm1heCgpKSBpZiBsZW4oZnJhbWVzKSBlbHNlIDEKCgpkZWYgX2xheW91dChndF9kaXI6IFBhdGgsIHByZWRfZGlyOiBQYXRoLCBzZXFzOiBsaXN0W3N0cl0sIHRyYWNrZXI6IHN0ciwgcm9vdDogUGF0aCkgLT4gZGljdDoKICAgIGd0X3Jvb3QsIHRya19yb290ID0gcm9vdCAvICJndCIsIHJvb3QgLyAidHJhY2tlcnMiCiAgICBmb3Igc2VxIGluIHNlcXM6CiAgICAgICAgZCA9IGd0X3Jvb3QgLyBzZXEKICAgICAgICAoZCAvICJndCIpLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAoZCAvICJndCIgLyAiZ3QudHh0Iikud3JpdGVfdGV4dCgoZ3RfZGlyIC8gZiJ7c2VxfS50eHQiKS5yZWFkX3RleHQoKSkKICAgICAgICAoZCAvICJzZXFpbmZvLmluaSIpLndyaXRlX3RleHQoCiAgICAgICAgICAgIGYiW1NlcXVlbmNlXVxubmFtZT17c2VxfVxuc2VxTGVuZ3RoPXtfc2VxX2xlbmd0aChndF9kaXIsIHNlcSl9XG4iCiAgICAgICAgKQogICAgICAgIHByZWQgPSBwcmVkX2RpciAvIGYie3NlcX0udHh0IgogICAgICAgIG91dCA9IHRya19yb290IC8gdHJhY2tlciAvICJkYXRhIiAvIGYie3NlcX0udHh0IgogICAgICAgIG91dC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgICAgIG91dC53cml0ZV90ZXh0KHByZWQucmVhZF90ZXh0KCkgaWYgcHJlZC5leGlzdHMoKSBlbHNlICIiKSAgIyBtaXNzaW5nID0gbm8gZGV0ZWN0aW9ucwogICAgc2VxbWFwID0gcm9vdCAvICJzZXFtYXAudHh0IgogICAgc2VxbWFwLndyaXRlX3RleHQoIm5hbWVcbiIgKyAiXG4iLmpvaW4oc2VxcykgKyAiXG4iKQogICAgcmV0dXJuIHsiZ3QiOiBndF9yb290LCAidHJhY2tlcnMiOiB0cmtfcm9vdCwgInNlcW1hcCI6IHNlcW1hcH0KCgpkZWYgZXZhbHVhdGUoZ3RfZGlyOiBzdHIgfCBQYXRoLCBwcmVkX2Rpcjogc3RyIHwgUGF0aCwgc2VxczogbGlzdFtzdHJdIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgICAgICB0cmFja2VyOiBzdHIgPSAibW9kZWwiLCBtZXRyaWNzOiB0dXBsZVtzdHIsIC4uLl0gPSAoIkhPVEEiLCAiQ0xFQVIiLCAiSWRlbnRpdHkiKSkgLT4gZGljdDoKICAgICIiIlJldHVybnMgb3ZlcmFsbCBhbmQgcGVyLXNlcXVlbmNlIHRyYWNraW5nIG1ldHJpY3MuCgogICAgT3ZlcmFsbCBrZXlzOiBIT1RBLCBEZXRBLCBBc3NBLCBMb2NBLCBNT1RBLCBNT1RQLCBJREYxLCBJRFAsIElEUiwgSURTVywgRlAsIEZOLCBGcmFnLgogICAgIiIiCiAgICBfbnVtcHlfY29tcGF0KCkKICAgIGltcG9ydCB0cmFja2V2YWwKCiAgICBndF9kaXIsIHByZWRfZGlyID0gUGF0aChndF9kaXIpLCBQYXRoKHByZWRfZGlyKQogICAgc2VxcyA9IHNvcnRlZChzZXFzIG9yIFtwLnN0ZW0gZm9yIHAgaW4gZ3RfZGlyLmdsb2IoIioudHh0IildKQogICAgaWYgbm90IHNlcXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk5vIGdyb3VuZC10cnV0aCBmaWxlcyBpbiB7Z3RfZGlyfSIpCgogICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB0bXA6CiAgICAgICAgcGF0aHMgPSBfbGF5b3V0KGd0X2RpciwgcHJlZF9kaXIsIHNlcXMsIHRyYWNrZXIsIFBhdGgodG1wKSkKICAgICAgICBldmFsX2NmZyA9IHRyYWNrZXZhbC5FdmFsdWF0b3IuZ2V0X2RlZmF1bHRfZXZhbF9jb25maWcoKQogICAgICAgIGV2YWxfY2ZnLnVwZGF0ZShQUklOVF9SRVNVTFRTPUZhbHNlLCBQUklOVF9DT05GSUc9RmFsc2UsIFRJTUVfUFJPR1JFU1M9RmFsc2UsCiAgICAgICAgICAgICAgICAgICAgICAgIE9VVFBVVF9TVU1NQVJZPUZhbHNlLCBPVVRQVVRfREVUQUlMRUQ9RmFsc2UsIFBMT1RfQ1VSVkVTPUZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICBVU0VfUEFSQUxMRUw9RmFsc2UsIERJU1BMQVlfTEVTU19QUk9HUkVTUz1UcnVlKQogICAgICAgIGRzX2NmZyA9IHRyYWNrZXZhbC5kYXRhc2V0cy5Nb3RDaGFsbGVuZ2UyREJveC5nZXRfZGVmYXVsdF9kYXRhc2V0X2NvbmZpZygpCiAgICAgICAgZHNfY2ZnLnVwZGF0ZSgKICAgICAgICAgICAgR1RfRk9MREVSPXN0cihwYXRoc1siZ3QiXSksIFRSQUNLRVJTX0ZPTERFUj1zdHIocGF0aHNbInRyYWNrZXJzIl0pLAogICAgICAgICAgICBUUkFDS0VSU19UT19FVkFMPVt0cmFja2VyXSwgU0VRTUFQX0ZJTEU9c3RyKHBhdGhzWyJzZXFtYXAiXSksIFNLSVBfU1BMSVRfRk9MPVRydWUsCiAgICAgICAgICAgIFRSQUNLRVJfU1VCX0ZPTERFUj0iZGF0YSIsIE9VVFBVVF9GT0xERVI9c3RyKFBhdGgodG1wKSAvICJvdXQiKSwgQ0xBU1NFU19UT19FVkFMPVtfQ0xBU1NdLAogICAgICAgICAgICBCRU5DSE1BUks9IlVBViIsIFNQTElUX1RPX0VWQUw9InZhbCIsIERPX1BSRVBST0M9RmFsc2UsIFBSSU5UX0NPTkZJRz1GYWxzZSwKICAgICAgICAgICAgR1RfTE9DX0ZPUk1BVD0ie2d0X2ZvbGRlcn0ve3NlcX0vZ3QvZ3QudHh0IiwKICAgICAgICApCiAgICAgICAgbWV0cmljX29ianMgPSBbZ2V0YXR0cih0cmFja2V2YWwubWV0cmljcywgbSkoeyJQUklOVF9DT05GSUciOiBGYWxzZX0pIGZvciBtIGluIG1ldHJpY3NdCiAgICAgICAgZXZhbHVhdG9yID0gdHJhY2tldmFsLkV2YWx1YXRvcihldmFsX2NmZykKICAgICAgICByZXMsIG1zZyA9IGV2YWx1YXRvci5ldmFsdWF0ZShbdHJhY2tldmFsLmRhdGFzZXRzLk1vdENoYWxsZW5nZTJEQm94KGRzX2NmZyldLCBtZXRyaWNfb2JqcykKCiAgICBzdGF0dXMgPSBtc2dbIk1vdENoYWxsZW5nZTJEQm94Il1bdHJhY2tlcl0KICAgIGlmIHN0YXR1cyAhPSAiU3VjY2VzcyI6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiVHJhY2tFdmFsIGZhaWxlZDoge3N0YXR1c30iKQogICAgcGVyX3NlcSA9IHJlc1siTW90Q2hhbGxlbmdlMkRCb3giXVt0cmFja2VyXQogICAgcmV0dXJuIHsKICAgICAgICAib3ZlcmFsbCI6IF9zdW1tYXJpc2UocGVyX3NlcVtfQ09NQklORURdW19DTEFTU10pLAogICAgICAgICJwZXJfc2VxdWVuY2UiOiB7czogX3N1bW1hcmlzZShwZXJfc2VxW3NdW19DTEFTU10pIGZvciBzIGluIHNlcXN9LAogICAgfQoKCmRlZiBfc3VtbWFyaXNlKHI6IGRpY3QpIC0+IGRpY3Q6CiAgICBvdXQgPSB7fQogICAgaWYgIkhPVEEiIGluIHI6CiAgICAgICAgZm9yIGsgaW4gKCJIT1RBIiwgIkRldEEiLCAiQXNzQSIsICJMb2NBIik6CiAgICAgICAgICAgIG91dFtrXSA9IGZsb2F0KG5wLm1lYW4oclsiSE9UQSJdW2tdKSkKICAgIGlmICJDTEVBUiIgaW4gcjoKICAgICAgICBmb3IgayBpbiAoIk1PVEEiLCAiTU9UUCIpOgogICAgICAgICAgICBvdXRba10gPSBmbG9hdChyWyJDTEVBUiJdW2tdKQogICAgICAgIGZvciBrLCBuYW1lIGluICgoIklEU1ciLCAiSURTVyIpLCAoIkNMUl9GUCIsICJGUCIpLCAoIkNMUl9GTiIsICJGTiIpLCAoIkZyYWciLCAiRnJhZyIpKToKICAgICAgICAgICAgb3V0W25hbWVdID0gaW50KHJbIkNMRUFSIl1ba10pCiAgICBpZiAiSWRlbnRpdHkiIGluIHI6CiAgICAgICAgZm9yIGsgaW4gKCJJREYxIiwgIklEUCIsICJJRFIiKToKICAgICAgICAgICAgb3V0W2tdID0gZmxvYXQoclsiSWRlbnRpdHkiXVtrXSkKICAgIHJldHVybiBvdXQK'),
}

def _md5(p):
    return hashlib.md5(Path(p).read_bytes()).hexdigest()

SCRIPTS = LOCAL / 'scripts'
SCRIPTS.mkdir(parents=True, exist_ok=True)
_cands = ([Path(SCRIPTS_OVERRIDE)] if SCRIPTS_OVERRIDE else []) + SCRIPTS_CANDIDATES
_found = next((c for c in _cands if all((c / n).exists() for n in EMBEDDED_SCRIPTS)), None)
for _name, (_emd5, _b64) in EMBEDDED_SCRIPTS.items():
    if _found is not None:
        shutil.copy(_found / _name, SCRIPTS / _name)
    else:
        _raw = base64.b64decode(_b64)
        assert hashlib.md5(_raw).hexdigest() == _emd5, f'Gömülü {_name} bozuk'
        (SCRIPTS / _name).write_bytes(_raw)
    _m = _md5(SCRIPTS / _name)
    print(f'{_name}: md5 {_m} |', 'gömülüyle aynı' if _m == _emd5 else 'gömülüden FARKLI (Drive sürümü kullanılıyor)')
SCRIPTS_SRC = str(_found) if _found is not None else 'gömülü (9 Kasım sürümü)'
print('Betik kaynağı:', SCRIPTS_SRC)

_t7 = (SCRIPTS / 'track_7a.py').read_text()
_flags = ['--cost_mode', '--nwd_C', '--coast_k', '--match_thresh', '--seqs', '--cmc', '--bot_root', '--data_dir', '--images', '--out']
_missing_flags = [f for f in _flags if f'"{f}"' not in _t7 and f"'{f}'" not in _t7]
assert not _missing_flags, f'track_7a.py şu bayrakları desteklemiyor: {_missing_flags}'
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))
print('Tracker betikleri hazır:', SCRIPTS)

In [ ]:
# 1.4) Eğitim verisi: Adım 0 arşivini aynı yola aç ve manifest ile doğrula
import yaml

IMG_FORMATS = {'bmp', 'dng', 'jpeg', 'jpg', 'mpo', 'png', 'tif', 'tiff', 'webp', 'pfm', 'heic'}


def yolo_split_files(data_yaml):
    """Adım 0'daki ile aynı fonksiyon (aynı dosya listesi ve manifest)."""
    d = yaml.safe_load(open(data_yaml))
    base = Path(d.get('path') or Path(data_yaml).parent)
    if not base.is_absolute():
        raise ValueError(f"dataset.yaml 'path' göreli: {base}. Mutlak yol bekleniyor.")
    out = {}
    for split in ('train', 'val'):
        entries = d[split] if isinstance(d[split], (list, tuple)) else [d[split]]
        imgs, lists = [], []
        for e in entries:
            p = Path(e) if Path(e).is_absolute() else base / e
            if p.is_dir():
                imgs += [str(x) for x in p.rglob('*.*')]
            elif p.is_file():
                lists.append(str(p))
                parent = str(p.parent) + os.sep
                imgs += [x.replace('./', parent) if x.startswith('./') else x
                         for x in p.read_text().strip().splitlines() if x.strip()]
            else:
                raise FileNotFoundError(f'{split} girdisi bulunamadı: {p}')
        imgs = sorted(x.strip() for x in imgs if x.strip().split('.')[-1].lower() in IMG_FORMATS)
        sa, sb = f'{os.sep}images{os.sep}', f'{os.sep}labels{os.sep}'
        labels = [sb.join(x.rsplit(sa, 1)).rsplit('.', 1)[0] + '.txt' for x in imgs]
        out[split] = {'images': imgs, 'labels': labels, 'lists': lists}
    return d, out


def manifest_of(files):
    h, total = hashlib.sha1(), 0
    for f in sorted(files):
        s = os.path.getsize(f)
        total += s
        h.update(f'{f}\t{s}\n'.encode())
    return {'n': len(files), 'bytes': total, 'sha1': h.hexdigest()}


def local_data_files(sp):
    files, missing = [], []
    for v in sp.values():
        for f in v['images']:
            (files if os.path.exists(f) else missing).append(f)
        files += [f for f in v['labels'] if os.path.exists(f)]
        files += v['lists']
    files = sorted({f for f in files if not f.startswith('/content/drive/')})
    return files, missing


assert DATA_MANIFEST.exists() and DATA_ARCHIVE.exists(), (
    f'Arşiv yok: {DATA_ARCHIVE}\nÖnce Adım 0 hücresini ANA notebook\'ta çalıştırın.')
MAN = json.load(open(DATA_MANIFEST))


def inspect_data():
    """(splits, yerel dosyalar, eksikler). dataset.yaml yerel bir liste/klasör gösteriyorsa ve henüz yoksa eksik sayılır."""
    try:
        _, sp = yolo_split_files(DATA_YAML)
    except FileNotFoundError as e:
        return None, [], [str(e)]
    files, missing = local_data_files(sp)
    return sp, files, missing


SPLITS, _files, _missing = inspect_data()
if _missing:
    print(f'{len(_missing)} eksik girdi; arşiv açılıyor (birkaç dakika)... örn. {_missing[:1]}')
    _t0 = time.time()
    with tarfile.open(DATA_ARCHIVE) as tf:
        try:
            tf.extractall('/', filter='data')
        except TypeError:          # filter parametresi olmayan Python sürümleri
            tf.extractall('/')
    print(f'Açıldı ({time.time() - _t0:.0f} s)')
    SPLITS, _files, _missing = inspect_data()

assert SPLITS is not None and not _missing, f'Arşivden sonra da eksik girdi var: {len(_missing)} (örn. {_missing[:3]})'
_counts = {s: {'images': len(v['images']), 'labels': sum(os.path.exists(x) for x in v['labels'])} for s, v in SPLITS.items()}
assert _counts == MAN['counts'], f'Sayılar manifestten farklı: {_counts} != {MAN["counts"]}'
assert manifest_of(_files) == MAN['files'], 'Dosya manifesti farklı: veri D1 ile birebir aynı değil'
print('Eğitim verisi D1 ile birebir aynı (manifest doğrulandı):', _counts)

In [ ]:
# 1.5) Ortak kütüphane: ana notebook 0.3 hücresinin birebir kopyası
#      (tek fark: PATCH_FILE yerel diske yazılır, paralel çalışan ana notebook ile çakışmasın diye)
import os
os.environ['SB_BOTSORT'] = '/content/baseline_src/YOLOv12-BoT-SORT-ReID/BoT-SORT'
import os, sys, json, glob, subprocess, hashlib, time, zipfile, tarfile, itertools
from pathlib import Path
SB_BOTSORT = Path(os.environ['SB_BOTSORT'])
import numpy as np
import pandas as pd
import cv2
from scipy.optimize import linear_sum_assignment

# ---- E0 durum dosyası ve kapılar ----
STATUS_FILE = WORK / 'E0_status.json'

def status_load():
    return json.load(open(STATUS_FILE)) if STATUS_FILE.exists() else {}

def status_set(key, val):
    st = status_load(); st[key] = val
    json.dump(st, open(STATUS_FILE, 'w'), indent=1, ensure_ascii=False)

B_DURUMLAR = ('dogrulandi', 'celiski_aciklandi', 'celiski_suruyor', 'kismen_dogrulandi')

def require_gate(*keys):
    """Kapı kontrolü. Koşul sağlanmıyorsa RuntimeError."""
    st = status_load(); bad = []
    for k in keys:
        if k == 'E0a' and st.get('E0a', {}).get('mota_tanimi') not in ('sekans_ortalamasi', 'toplam'):
            bad.append('E0-a (resmî MOTA tanımı yazılmadı)')
        if k == 'E0b' and (st.get('E0b', {}).get('durum') not in B_DURUMLAR or not st.get('E0b', {}).get('sb_kaynak')
                           or not st.get('E0b', {}).get('dt_kaynak')):
            bad.append('E0-b (SB / Dist-Tracker kaynak notu tamamlanmadı)')
        if k == 'E0c' and st.get('E0c') != 'TEMIZ':
            bad.append(f"E0-c (karar: {st.get('E0c', 'çalıştırılmadı')}; TEMİZ gerekli)")
        if k == 'E0d' and st.get('E0d') != 'CIoU':
            bad.append(f"E0-d (detector-v1 kaybı: {st.get('E0d', 'doğrulanmadı')}; CIoU gerekli. Değilse D1'in gerekçesi yeniden değerlendirilir)")
    if bad:
        raise RuntimeError('Kapı geçilmedi: ' + ' | '.join(bad))

def official_mota_col():
    t = status_load().get('E0a', {}).get('mota_tanimi')
    if t not in ('sekans_ortalamasi', 'toplam'):
        raise RuntimeError('E0-a mota_tanimi yazilmadi; once E0-a hucresini calistirin')
    return 'MOTA_seq' if t == 'sekans_ortalamasi' else 'MOTA'

# ---- Temel yardımcılar ----
def load_mot(path):
    """GT: frame,id,x,y,w,h,conf,cls,vis | Tahmin: frame,id,x,y,w,h,score,-1,-1,-1 (10 sütun)."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return np.zeros((0, 10))
    return np.loadtxt(path, delimiter=",", ndmin=2)


def load_dets(path):
    """Dedektör çıktısı: frame,x1,y1,x2,y2,score."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return np.zeros((0, 6))
    return np.loadtxt(path, delimiter=",", ndmin=2)


def xywh2xyxy(b):
    b = np.asarray(b, dtype=float).reshape(-1, 4)
    return np.stack([b[:, 0], b[:, 1], b[:, 0] + b[:, 2], b[:, 1] + b[:, 3]], 1)


def iou_matrix(a, b):
    """a: (n,4) xyxy, b: (m,4) xyxy -> (n,m) IoU."""
    a = np.asarray(a, float).reshape(-1, 4)
    b = np.asarray(b, float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def match_frame(gt_xyxy, pr_xyxy, thr=0.5):
    """Macar Hungarian (IoU maksimum). Dönüş: [(gt_idx, pred_idx, iou)]."""
    M = iou_matrix(gt_xyxy, pr_xyxy)
    if M.size == 0:
        return []
    r, c = linear_sum_assignment(-M)
    return [(int(i), int(j), float(M[i, j])) for i, j in zip(r, c) if M[i, j] >= thr]


def bootstrap_diff(base, new, B=10000, seed=0):
    """Sekans düzeyi eşleştirilmiş fark (new - base).
    Dönüş: ortalama fark, %95 güven aralığı, iki yönlü bootstrap p-değeri."""
    base = np.asarray(base, float); new = np.asarray(new, float)
    d = new - base
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(d), size=(B, len(d)))
    boots = d[idx].mean(1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())
    return float(d.mean()), (float(lo), float(hi)), float(min(1.0, p))


def holm(pvals):
    """Holm-Bonferroni düzeltmesi. Girdi ve çıktı aynı sırada."""
    p = np.asarray(pvals, float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i])
        adj[i] = min(1.0, run)
    return adj


def peak_snr(img_gray, box_xyxy, ring=4):
    """Kutu içi tepe değerin, çevre halkasına göre SNR'ı. (E6 ve E8 alt kümeleri)"""
    x1, y1, x2, y2 = [int(round(v)) for v in box_xyxy]
    H, W = img_gray.shape
    x1, y1 = max(0, x1), max(0, y1); x2, y2 = min(W, x2), min(H, y2)
    if x2 <= x1 or y2 <= y1:
        return np.nan
    inner = img_gray[y1:y2, x1:x2].astype(np.float32)
    peak = inner.max()
    X1, Y1, X2, Y2 = max(0, x1 - ring), max(0, y1 - ring), min(W, x2 + ring), min(H, y2 + ring)
    ring_mask = np.ones((Y2 - Y1, X2 - X1), bool)
    ring_mask[y1 - Y1:y2 - Y1, x1 - X1:x2 - X1] = False
    bg = img_gray[Y1:Y2, X1:X2].astype(np.float32)[ring_mask]
    return float((peak - bg.mean()) / max(bg.std(), 1.0))  # std tabanı: düz arka planda sıfıra bölmeyi önler


def ap_and_recall(dets, gt, thr=0.5, small_side=8, recall_score_min=0.0):
    """Tek sınıf AP50 (VOC tüm nokta; tüm skorlarla) + recall (tüm GT ve küçük GT).
    recall yalnız skoru >= recall_score_min olan tespitlerle hesaplanır (iki yöntemi aynı eşikle karşılaştırın).
    small: sqrt(w*h) < small_side piksel."""
    gtf = {}
    for row in gt:
        gtf.setdefault(int(row[0]), []).append(row[2:6])
    gtb = {f: xywh2xyxy(np.array(v)) for f, v in gtf.items()}
    gt_side = {f: np.sqrt(np.array(v)[:, 2] * np.array(v)[:, 3]) for f, v in gtf.items()}
    npos = sum(len(v) for v in gtb.values())
    n_small = sum(int((s < small_side).sum()) for s in gt_side.values())
    d = dets[np.argsort(-dets[:, 5])] if len(dets) else np.zeros((0, 6))
    used = {f: np.zeros(len(v), bool) for f, v in gtb.items()}
    tp = np.zeros(len(d))
    for k in range(len(d)):
        f = int(d[k, 0])
        if f not in gtb:
            continue
        ious = iou_matrix(d[k:k + 1, 1:5], gtb[f])[0]
        ious[used[f]] = -1
        j = int(np.argmax(ious))
        if ious[j] >= thr:
            tp[k] = 1; used[f][j] = True
    ctp = np.cumsum(tp)
    rec = ctp / max(npos, 1)
    prec = ctp / np.arange(1, len(d) + 1) if len(d) else np.zeros(0)
    mrec = np.concatenate([[0], rec, [1]]); mpre = np.concatenate([[0], prec, [0]])
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))
    # recall: her GT için, skoru eşiği geçen tespitlerle eşleşme
    dr = dets[dets[:, 5] >= recall_score_min] if len(dets) else dets
    matched_small = 0; matched_all = 0
    for f, boxes in gtb.items():
        dd = dr[dr[:, 0] == f]
        pairs = match_frame(boxes, dd[:, 1:5], thr)
        mi = {p[0] for p in pairs}
        matched_all += len(mi)
        matched_small += sum(1 for i in mi if gt_side[f][i] < small_side)
    return {"AP50": ap, "recall_all": matched_all / max(npos, 1),
            "recall_small": matched_small / max(n_small, 1), "n_gt": npos, "n_small": n_small}


def tiles(H, W, nx=3, ny=2, ov=0.25):
    """SAHI tarzı dilimler: nx x ny, %ov örtüşme. Dönüş: [(x0,y0,x1,y1)]."""
    tw = int(np.ceil(W / (nx - (nx - 1) * ov)))
    th = int(np.ceil(H / (ny - (ny - 1) * ov)))
    sx = int(tw * (1 - ov)); sy = int(th * (1 - ov))
    out = []
    for j in range(ny):
        for i in range(nx):
            x0 = min(i * sx, W - tw); y0 = min(j * sy, H - th)
            out.append((x0, y0, x0 + tw, y0 + th))
    return out


# ---- GT-tahmin eşleşmesi, IDSW grupları, uzun boşluk yeniden bağlama ----
def centers(a):
    return np.c_[a[:, 2] + a[:, 4] / 2, a[:, 3] + a[:, 5] / 2] if len(a) else np.zeros((0, 2))


def nearest_other_gt(g):
    """Her GT kutusu için aynı karedeki en yakın diğer GT merkezine uzaklık (piksel)."""
    c = centers(g)
    if len(c) < 2:
        return np.full(len(c), np.inf)
    D = np.linalg.norm(c[:, None] - c[None], axis=-1)
    np.fill_diagonal(D, np.inf)
    return D.min(1)


def assignments(gt, pr, thr=0.5):
    """Her karede GT->tahmin eşleşmelerini çıkarır. Dönüş: DataFrame(f, gid, pid, nn_dist)."""
    rows = []
    frames = np.unique(gt[:, 0]).astype(int)
    for f in frames:
        g = gt[gt[:, 0] == f]; p = pr[pr[:, 0] == f]
        nn = nearest_other_gt(g)
        for gi, pj, _ in match_frame(xywh2xyxy(g[:, 2:6]), xywh2xyxy(p[:, 2:6]) if len(p) else np.zeros((0, 4)), thr):
            rows.append((int(f), int(g[gi, 1]), int(p[pj, 1]), float(nn[gi])))
    return pd.DataFrame(rows, columns=['f', 'gid', 'pid', 'nn_dist'])


def idsw_groups(gt, pr, close_px=12.7, long_gap=10, thr=0.5):
    """IDSW olaylarını üç gruba ayırır: yakın çift (nn <= close_px), uzun boşluk (>long_gap kare), diğer.
    Öncelik sırası: yakın çift, sonra uzun boşluk, sonra diğer.
    Not: IDSW tanımı TrackEval'deki tanıma yaklaşıktır; resmî sayılar eval_tracker çıktısından alınır."""
    A = assignments(gt, pr, thr)
    ev = []
    for gid, g in A.sort_values('f').groupby('gid'):
        last_pid = None; last_f = None
        for _, r in g.iterrows():
            if last_pid is not None and r.pid != last_pid:
                gap = int(r.f - last_f - 1)
                if r.nn_dist <= close_px:
                    grp = 'yakın çift'
                elif gap > long_gap:
                    grp = 'uzun boşluk'
                else:
                    grp = 'diğer'
                ev.append((int(r.f), gid, grp, gap, float(r.nn_dist)))
            last_pid, last_f = r.pid, r.f
    return pd.DataFrame(ev, columns=['f', 'gid', 'grup', 'gap', 'nn_dist'])


def long_gap_reid_rate(gt, pr, long_gap=10, thr=0.5):
    """Uzun boşluk (>long_gap kare) sonrası, aynı GT'nin önceki tahmin kimliğine dönme oranı."""
    A = assignments(gt, pr, thr)
    ok = tot = 0
    for gid, g in A.sort_values('f').groupby('gid'):
        fs = g.f.values; ps = g.pid.values
        for k in range(1, len(fs)):
            if fs[k] - fs[k - 1] - 1 > long_gap:
                tot += 1
                ok += int(ps[k] == ps[k - 1])
    return ok / tot if tot else float('nan'), tot


def idsw_enrichment(seqs, gt_get, pr_get, close_px=12.7, B=2000, seed=0):
    """Zenginleşme oranı = (IDSW olaylarında yakın çift payı) / (eşleşen tüm GT kutularında yakın çift payı).
    Sekans düzeyinde bootstrap CI. Oran ~1 ise yakın çift, IDSW'ye taban oranından fazla katkı yapmıyor."""
    per = []
    for s in seqs:
        gt = gt_get(s); pr = pr_get(s)
        A = assignments(gt, pr)
        ev = idsw_groups(gt, pr, close_px=close_px)
        per.append((len(ev), int((ev.grup == 'yakın çift').sum()) if len(ev) else 0,
                    len(A), int((A.nn_dist <= close_px).sum()) if len(A) else 0))
    P = np.array(per, float)

    def ratio(Q):
        return (Q[:, 1].sum() / max(Q[:, 0].sum(), 1)) / max(Q[:, 3].sum() / max(Q[:, 2].sum(), 1), 1e-9)
    rng = np.random.default_rng(seed)
    boots = np.array([ratio(P[rng.integers(0, len(P), len(P))]) for _ in range(B)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(ratio(P)), (float(lo), float(hi)), P


def box_deviation(dets_dir, seqs=None, score_min=0.09, thr=0.3):
    """detector kutularının GT'den sapması (IoU>=thr Hungarian eşleşmesi).
    Dönüş: eşleşmiş kutular DataFrame, boyut sınıfına göre kaçırılan GT payı.
    Not: yalnız eşleşenler ölçülür; kaçırılanlar ayrıca raporlanır (eşleşme yanlılığını göstermek için)."""
    rows, miss = [], []
    for s in (seqs or VAL_SEQS):
        dets = load_dets(Path(dets_dir) / f'{s}.txt')
        gt = load_mot(DV1 / 'gt' / f'{s}.txt'); gt = gt[gt[:, 6] == 1]
        dets = dets[dets[:, 5] >= score_min] if len(dets) else dets
        for f in np.unique(gt[:, 0]):
            g = gt[gt[:, 0] == f]; d = dets[dets[:, 0] == f] if len(dets) else np.zeros((0, 6))
            pairs = match_frame(xywh2xyxy(g[:, 2:6]), d[:, 1:5], thr=thr) if len(d) else []
            mset = {p[0] for p in pairs}
            for gi, di, iou in pairs:
                gx, gy, gw, gh = g[gi, 2:6]
                px1, py1, px2, py2 = d[di, 1:5]
                rows.append(dict(seq=s, f=f, side=np.sqrt(gw * gh),
                                 dx=(px1 + px2) / 2 - (gx + gw / 2), dy=(py1 + py2) / 2 - (gy + gh / 2),
                                 dw=(px2 - px1) - gw, dh=(py2 - py1) - gh, iou=iou))
            for gi in range(len(g)):
                if gi not in mset:
                    miss.append(dict(seq=s, side=float(np.sqrt(g[gi, 4] * g[gi, 5]))))
    dev = pd.DataFrame(rows)
    dev['bin'] = pd.cut(dev.side, [0, 8, 16, 1e9], labels=['<8', '8-16', '>=16'], right=False)
    dev['center_err'] = np.hypot(dev.dx, dev.dy)
    ms = pd.DataFrame(miss)
    if len(ms):
        ms['bin'] = pd.cut(ms.side, [0, 8, 16, 1e9], labels=['<8', '8-16', '>=16'], right=False)
    return dev, ms


# ---- Tracker çalıştırıcı: track_7a.py (SB BoT-SORT) + isteğe bağlı gate/conf yaması ----
import shutil, hashlib
from trackeval_runner import evaluate as _te_evaluate

TRACK_7A = SCRIPTS / 'track_7a.py'
TRACK_NWD_C = 15.0
MT = {'iou': 0.7, 'nwd': 0.8}     # match_thresh: önceki ölçümlerden

def mt_for(patches):
    return MT['nwd'] if 'nwd' in patches.split(',') else MT['iou']

PATCH_SRC = r'''
import os, sys, runpy, json
SB = os.environ["SB_BOTSORT"]; TRACK = os.environ["TRACK_7A"]
sys.path.insert(0, SB)
import numpy as np
# fast_reid ve cython_bbox stub'ları: tracker import edilmeden önce kurulmalı
import types
class _FR:
    def __init__(self, *a, **k): pass
    def inference(self, img, dets): raise RuntimeError('FastReID kapalı: with_reid=False olmalı')
_pkg = types.ModuleType("fast_reid"); _mod = types.ModuleType("fast_reid.fast_reid_interfece")
_mod.FastReIDInterface = _FR; _pkg.fast_reid_interfece = _mod
sys.modules["fast_reid"] = _pkg; sys.modules["fast_reid.fast_reid_interfece"] = _mod
try:
    import cython_bbox  # noqa: F401
except ImportError:
    def _bbox_overlaps(a, b):
        a = np.asarray(a, float); b = np.asarray(b, float)
        iw = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]) + 1, 0, None)
        ih = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]) + 1, 0, None)
        inter = iw * ih
        aa = ((a[:, 2] - a[:, 0] + 1) * (a[:, 3] - a[:, 1] + 1))[:, None]
        bb = ((b[:, 2] - b[:, 0] + 1) * (b[:, 3] - b[:, 1] + 1))[None, :]
        return inter / (aa + bb - inter)
    _cb = types.ModuleType("cython_bbox"); _cb.bbox_overlaps = _bbox_overlaps; sys.modules["cython_bbox"] = _cb
import tracker.matching as M
import tracker.mc_bot_sort as MB
STATE = {"frame": 0}
_upd_orig = MB.BoTSORT.update
def _upd_wrap(self, *a, **k):
    STATE["frame"] = self.frame_id + 1   # update içinde frame_id bir artacak
    out, lost = _upd_orig(self, *a, **k)
    # Coasting: kayıp iz, son eşleşmesinden sonra en fazla coast_k kare boyunca çıktıya eklenir
    K = int(getattr(self, "coast_k", 0) or 0)
    if K > 0:
        extra = [t for t in lost if 1 <= self.frame_id - int(t.frame_id) <= K]
        out = list(out) + extra
    return out, lost
MB.BoTSORT.update = _upd_wrap
MODE = set(x for x in os.environ.get("PATCHES", "").split(",") if x)
TAU = float(os.environ.get("GATE_TAU", "50"))
MIN_MISSED = int(os.environ.get("GATE_MIN_MISSED", "0"))
CHI2 = 9.4877  # chi2(0.95, df=4)
STATS = {"gate_calls": 0, "gated_pairs": 0, "rejected_pairs": 0, "conf_pairs": 0}
# NWD maliyeti: SB matching.py'de yok, burada uygulanır. track_7a COST["mode"] ve COST["C"] değerlerini ayarlar.
# KF_SCALE: track_7a --kf_q / --kf_r. Varsayılan 1.0 = orijinal Kalman davranışı.
# Q: predict/multi_predict'e (s_Q^2 - 1)*diag(std^2) eklenir; R: project'e aynı şekilde.
import tracker.kalman_filter as KFM
KFM.KF_SCALE = {"Q": 1.0, "R": 1.0}
_KF = KFM.KalmanFilter
_pred0, _proj0, _mpred0 = _KF.predict, _KF.project, _KF.multi_predict

def _q_std(self, m):
    wp, wv = self._std_weight_position, self._std_weight_velocity
    return np.r_[wp*m[2], wp*m[3], wp*m[2], wp*m[3], wv*m[2], wv*m[3], wv*m[2], wv*m[3]]

def _pred_w(self, mean, covariance):
    m, c = _pred0(self, mean, covariance)
    q = KFM.KF_SCALE["Q"]
    if q != 1.0:
        s = _q_std(self, mean)
        c = c + (q*q - 1.0) * np.diag(np.square(s))
    return m, c

def _mpred_w(self, mean, covariance):
    m, c = _mpred0(self, mean, covariance)
    q = KFM.KF_SCALE["Q"]
    if q != 1.0:
        wp, wv = self._std_weight_position, self._std_weight_velocity
        s = np.stack([wp*mean[:,2], wp*mean[:,3], wp*mean[:,2], wp*mean[:,3],
                      wv*mean[:,2], wv*mean[:,3], wv*mean[:,2], wv*mean[:,3]], axis=1)
        c = c + (q*q - 1.0) * (np.square(s)[:, :, None] * np.eye(8)[None])
    return m, c

def _proj_w(self, mean, covariance):
    m, c = _proj0(self, mean, covariance)
    r = KFM.KF_SCALE["R"]
    if r != 1.0:
        wp = self._std_weight_position
        s = np.r_[wp*mean[2], wp*mean[3], wp*mean[2], wp*mean[3]]
        c = c + (r*r - 1.0) * np.diag(np.square(s))
    return m, c

_KF.predict, _KF.project, _KF.multi_predict = _pred_w, _proj_w, _mpred_w

# NWD maliyeti: SB matching.py'de yok, burada uygulanır. track_7a COST["mode"] ve COST["C"] değerlerini ayarlar.
M.COST = {"mode": "iou", "C": 15.0}

def _to_cwh(x):
    if isinstance(x, np.ndarray):
        x1, y1, x2, y2 = x[:4]
        return [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]
    return [x.tlwh[0] + x.tlwh[2] / 2, x.tlwh[1] + x.tlwh[3] / 2, x.tlwh[2], x.tlwh[3]]

def _nwd_cost(a, b):
    A = np.array([_to_cwh(t) for t in a], float).reshape(-1, 4)
    B = np.array([_to_cwh(d) for d in b], float).reshape(-1, 4)
    if len(A) == 0 or len(B) == 0:
        return np.zeros((len(A), len(B)))
    d2 = ((A[:, None, :2] - B[None, :, :2]) ** 2).sum(-1) + (((A[:, None, 2:] - B[None, :, 2:]) / 2) ** 2).sum(-1)
    return 1.0 - np.exp(-np.sqrt(d2) / M.COST["C"])

_iou_orig = M.iou_distance
def _cost_dispatch(a, b):
    if M.COST["mode"] == "nwd":
        return _nwd_cost(a, b)
    return _iou_orig(a, b)
M.iou_distance = _cost_dispatch
_orig = M.iou_distance

def gate_cost(a, b):
    cost = _orig(a, b)
    if not (MODE & {"gate", "conf"}) or len(a) == 0 or len(b) == 0 or isinstance(a[0], np.ndarray):
        return cost
    STATS["gate_calls"] += 1
    dets = np.array([[d.tlwh[0] + d.tlwh[2] / 2, d.tlwh[1] + d.tlwh[3] / 2, d.tlwh[2], d.tlwh[3]] for d in b], float)
    for i, t in enumerate(a):
        missed = STATE["frame"] - int(t.frame_id) - 1   # kayıp update sayısı: tracker sayacı - track'in son update karesi - 1
        if missed <= 0 or missed < MIN_MISSED:
            continue
        STATS["gated_pairs"] += len(b)
        if "gate" in MODE:
            m, S = t.kalman_filter.project(t.mean, t.covariance)
            Si = np.linalg.inv(S)
            r = dets - np.asarray(m, float)[None, :]
            bad = np.einsum("ij,jk,ik->i", r, Si, r) > CHI2
            STATS["rejected_pairs"] += int(bad.sum())
            cost[i, bad] = 1.0
        if "conf" in MODE:
            ok = cost[i] < 1.0
            cost[i, ok] = 1 - np.exp(-missed / TAU) * (1 - cost[i, ok])
            STATS["conf_pairs"] += int(ok.sum())
    return cost

if MODE & {"gate", "conf"}:
    M.iou_distance = gate_cost
sys.argv = [TRACK] + sys.argv[1:]
try:
    runpy.run_path(TRACK, run_name="__main__")
finally:
    print("PATCH_STATS", json.dumps(STATS))
'''
PATCH_FILE = LOCAL / 'run_patched.py'   # D2: yerel (ana notebook'un WORK/run_patched.py dosyasına dokunmaz)
PATCH_FILE.write_text(PATCH_SRC)

def make_data_dir(dets_dir):
    """track_7a.py'nin beklediği yapı: dets/, gt/, val.txt. Her çıktı için ayrı klasör."""
    key = hashlib.md5(str(dets_dir).encode()).hexdigest()[:10]
    d = LOCAL / 'data' / key
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)
    (d / 'dets').symlink_to(Path(dets_dir).resolve())
    (d / 'gt').symlink_to((DV1 / 'gt').resolve())
    shutil.copy(VAL_TXT, d / 'val.txt')
    return d

def run_tracker_patched(dets_dir, out_dir, patches='', cmc='sparseOptFlow', gate_tau=50, gate_min_missed=0,
                        match_thresh=None, coast_k=0, extra=()):
    """patches: '' | 'nwd' | 'nwd,gate' | 'nwd,gate,conf'. 'nwd' = --cost_mode nwd (yama yok).
    Doğrulama: run.json içindeki bayraklar ve yama sayaçları kontrol edilir."""
    pset = set(x for x in patches.split(',') if x)
    cost = 'nwd' if 'nwd' in pset else 'iou'
    mt = match_thresh if match_thresh is not None else mt_for(patches)
    data = make_data_dir(dets_dir)
    cmd = [sys.executable, str(PATCH_FILE),
           '--bot_root', str(SB_BOTSORT), '--data_dir', str(data), '--images', str(FRAMES_ROOT),
           '--out', str(out_dir), '--cmc', cmc, '--cost_mode', cost, '--nwd_C', str(TRACK_NWD_C),
           '--match_thresh', str(mt), '--coast_k', str(coast_k),
           '--seqs', ','.join(VAL_SEQS), *extra]
    env = {**os.environ, 'SB_BOTSORT': str(SB_BOTSORT), 'TRACK_7A': str(TRACK_7A),
           'PATCHES': patches, 'GATE_TAU': str(gate_tau), 'GATE_MIN_MISSED': str(gate_min_missed)}
    r = subprocess.run(cmd, env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-3000:]); raise RuntimeError('tracker başarısız')
    line = [l for l in r.stdout.splitlines() if l.startswith('PATCH_STATS')]
    stats = json.loads(line[-1].split(' ', 1)[1]) if line else {}
    ay = json.load(open(Path(out_dir) / 'run.json'))['ayarlar']
    if ay['cost_mode'] != cost or int(ay['coast_k']) != int(coast_k):
        raise RuntimeError(f'Bayraklar track_7a\'ya geçmedi: {ay["cost_mode"]}, {ay["coast_k"]}')
    if pset & {'gate', 'conf'} and stats.get('gated_pairs', 0) <= 0:
        raise RuntimeError(f'Kapı yaması hiçbir çifte uygulanmadı: {stats}')
    print('PATCH_STATS', stats)
    return stats

# ---- Değerlendirme: trackeval_runner (tek klasör) ----
def make_eval_gt(gt_src_dir, seqs, frames_root, out_dir):
    """TrackEval için GT: frame,id,x,y,w,h,1,1,1 + <seq>.len."""
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    for s in seqs:
        g = np.loadtxt(Path(gt_src_dir) / f'{s}.txt', delimiter=',', ndmin=2)[:, :6]
        np.savetxt(out / f'{s}.txt', np.c_[g, np.ones((len(g), 3))], delimiter=',', fmt='%g')
        (out / f'{s}.len').write_text(str(len(glob.glob(f'{frames_root}/{s}/*.jpg'))))
    return out

# Val kareleri kontrolü (seqLength için şart; eksikse sessizce 0 olur)
_eksik = [s for s in VAL_SEQS if len(glob.glob(f'{FRAMES_ROOT}/{s}/*.jpg')) == 0]
if _eksik:
    raise RuntimeError(f'Val kareleri eksik ({len(_eksik)} sekans), örn: {_eksik[:3]}. FRAMES_ROOT: {FRAMES_ROOT}')

GT_EVAL = make_eval_gt(DV1 / 'gt', VAL_SEQS, FRAMES_ROOT, LOCAL / 'gt_eval')

def _tname(tag):
    return tag.replace('.', '_').replace('/', '_')

def eval_multi(gt_eval_dir, pred_dirs, seqs):
    """{ad: trackeval_runner çıktısı (overall + per_sequence)}"""
    return {n: _te_evaluate(gt_eval_dir, d, seqs, tracker=_tname(n)) for n, d in pred_dirs.items()}

def eval_dirs(pred_dirs, tag='all'):
    """{ad: overall metrikler}"""
    return {n: r['overall'] for n, r in eval_multi(GT_EVAL, pred_dirs, VAL_SEQS).items()}

PS_CACHE = {}
def per_seq_table(pred_dir, tag):
    key = str(pred_dir)
    if key not in PS_CACHE:
        r = _te_evaluate(GT_EVAL, pred_dir, VAL_SEQS, tracker=_tname(tag))
        PS_CACHE[key] = pd.DataFrame(r['per_sequence']).T
    return PS_CACHE[key]

def summary_table(dirs, tag):
    """Birleşik metrikler + sekans ortalaması (MOTA_seq, HOTA_seq). Resmî sütun: official_mota_col()."""
    res = eval_dirs(dirs, tag)
    df = pd.DataFrame(res).T
    seqm = {}
    for n, d in dirs.items():
        t = per_seq_table(d, f'{tag}_{n}')
        seqm[n] = {'MOTA_seq': float(t['MOTA'].astype(float).mean()), 'HOTA_seq': float(t['HOTA'].astype(float).mean())}
    return df.join(pd.DataFrame(seqm).T)

def compare(tbl_base, tbl_new, metric='MOTA', B=10000):
    b = tbl_base.loc[VAL_SEQS, metric].astype(float).values
    n = tbl_new.loc[VAL_SEQS, metric].astype(float).values
    m, ci, p = bootstrap_diff(b, n, B=B)
    return {'metrik': metric, 'fark': round(m, 4), 'CI95': (round(ci[0], 4), round(ci[1], 4)), 'p': round(p, 4)}

# ---- Görsel kayıt: GT (yeşil) ve tahmin (kırmızı, kimlik) ----
def render_video(seq, gt, pred, out_mp4, f0=1, f1=200, fps=10, frames_root=None):
    files = sorted(glob.glob(f'{frames_root or FRAMES_ROOT}/{seq}/*.jpg'))
    H, W = cv2.imread(files[0]).shape[:2]
    vw = cv2.VideoWriter(str(out_mp4), cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))
    for f in range(f0, min(f1, len(files)) + 1):
        img = cv2.imread(files[f - 1])
        for r in gt[gt[:, 0] == f]:
            x, y, w, h = map(int, r[2:6])
            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 1)
        for r in pred[pred[:, 0] == f]:
            x, y, w, h = map(int, r[2:6])
            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 0, 255), 1)
            cv2.putText(img, str(int(r[1])), (x, max(y - 2, 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 0, 255), 1)
        vw.write(img)
    vw.release()
    return out_mp4


print('Ortak kütüphane hazır.')

In [ ]:
# 1.6) E0-c kapısı ve ön kayıt: karar kuralı eğitimden ÖNCE kilitlenir
require_gate('E0c')

PREREG = D2_DIR / 'D2_prereg.json'
prereg = {
    'deney': 'D2: P2 katmanı (stride 4), detector-v1 üzerine',
    'kollar': {'v1_ref': str(DETECTOR_WEIGHTS), 'ctrl': str(D1_CTRL_WEIGHTS), 'p2': str(D2_DIR / 'p2' / 'weights' / 'last.pt')},
    'egitim': {'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'lr0': LR0, 'optimizer': OPT, 'freeze': FREEZE,
               'seed': SEED, 'data': str(DATA_YAML), 'agirlik': 'last.pt', 'fork_commit': SB_COMMIT},
    'p2_mimari': 'katman 21 Upsample(14), 22 Concat(-1, 2), 23 A2C2f(256, a2=False), Detect [23, 14, 17, 20]',
    'p2_baslatma': 'Katman 0-20 ve Detect P3/P4/P5 dalları detector-v1\'den birebir; Detect P2 dalı P3 dalının kopyası, '
                   'P2 sınıf biası bias_init (stride 4); katman 21-23 rastgele.',
    'tespit': {'imgsz': IMGSZ, 'conf': DET_CONF},
    'tracker_karar_protokolu': {'patches': '', 'cmc': 'sparseOptFlow', 'match_thresh': 0.7, 'coast_k': 0},
    'birincil': 'DetA farkı (p2 - ctrl), sekans bootstrap B=10000, %95 CI. CI alt sınırı > 0 ise detector-v1-P2 '
                'kabul edilir; değilse detector-v1 kalır.',
    'bootstrap_tanimi': 'Eşleştirilmiş: her örnekte aynı sekanslar; fark = sekans başına (p2 - ctrl) farklarının ortalaması '
                        '(D1/E serisindeki compare ile aynı). Birleşik (combined) DetA farkı ayrıca nokta tahmini olarak raporlanır.',
    'ikincil_holm': ['recall_<8px', 'HOTA', 'AssA', 'IDSW'],
    'kucuk_hedef_tanimi': 'sqrt(w*h) < 8 px; recall: IoU >= 0.5, skor >= 0.09, kare başına Hungarian; '
                          'küçük GT si olmayan sekanslar dışarıda',
    'tekrar_uretilebilirlik_toleransi': {'MOTA_HOTA_DetA_AssA_abs': REPRO_TOL, 'IDSW_goreli': 0.02},
    'bilgi': ['p2 - v1_ref ve ctrl - v1_ref DetA (fine-tune gürültüsü referansı)', 'birleşik DetA farkı (nokta)',
              'dondurulmuş tracker (E2) HOTA ve MOTA_seq', 'hız ms/kare (fp32, batch 1; çıkarım NMS hariç, uçtan uca NMS dahil)',
              'parametre, GFLOPs', 'boyut sınıfına göre recall', 'aday anchor kapsaması (P3 vs P2, yaklaşık)'],
}
_rule_sha = hashlib.sha1(json.dumps(prereg, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
if PREREG.exists():
    _old = json.load(open(PREREG))
    if _old.get('kural_sha1') != _rule_sha:
        raise RuntimeError('Ön kayıt değişti; karar kuralı sonradan değiştirilemez. Bilinçli bir değişiklikse eski '
                           f'dosyayı yeniden adlandırın ve nedenini rapora yazın: {PREREG}')
    print('Ön kayıt mevcut ve değişmedi (', _old['olusturma'], ')')
else:
    with open(PREREG, 'w') as fh:
        json.dump({**prereg, 'kural_sha1': _rule_sha, 'olusturma': time.strftime('%Y-%m-%d %H:%M:%S')},
                  fh, indent=1, ensure_ascii=False)
    print('Ön kayıt yazıldı:', PREREG)
PREREG_SHA256 = hashlib.sha256(PREREG.read_bytes()).hexdigest()
print('Kural sha1:', _rule_sha, '| ön kayıt dosyası sha256:', PREREG_SHA256, '| fork commit:', SB_COMMIT)

## Adım 2: P2 modelinin kurulması ve doğrulanması

Bu adım eğitimden önce modeli kurar ve dört şeyi `assert` ile doğrular. Herhangi biri başarısız olursa eğitim başlamaz:

1. **Mimari:** P2 yapılandırması detector-v1'in kendi yaml'ından türetilir (aynı scale ve backbone). Detect 4 seviyeli, stride'lar `[4, 8, 16, 32]`.
2. **Ağırlık aktarımı:** detector-v1'in her ağırlığı hedefte aynı şekille bulunur. Ultralytics ağırlıkları katman adıyla eşleştirdiği için Detect başlığı `model.21` → `model.24` olarak elle yeniden eşlenir; aksi halde başlık sessizce sıfırdan başlardı. Yalnızca katman 21–23 ve Detect'in P2 dalı yenidir.
3. **Eşdeğerlik:** Aynı giriş için P2 modelinin P3/P4/P5 çıktıları detector-v1'in çıktılarıyla birebir aynı olmalı (CPU, float32, |fark| < 1e-4). Bu, aktarımın doğru olduğunun en güçlü kanıtıdır.
4. **Eğitim yolu:** Trainer'ın ağırlık yükleme yolu simüle edilir; aktarılan anahtar sayısı toplam anahtar sayısına eşit olmalı. Eğitim argümanları (single_cls, imgsz, data) detector-v1 checkpoint'iyle aynı olmalı.

In [ ]:
# 2.1) P2 modeli: kur, ağırlıkları aktar, doğrula, başlangıç checkpoint'ini kaydet
import yaml
from ultralytics.nn.tasks import DetectionModel, attempt_load_one_weight
from ultralytics.nn.modules import Detect
from ultralytics.utils.torch_utils import intersect_dicts, get_num_params, get_flops

v1, v1_ckpt = attempt_load_one_weight(str(DETECTOR_WEIGHTS))
v1 = v1.cpu().float().eval()
cfg_v1 = copy.deepcopy(v1.yaml)
N_V1 = len(cfg_v1['backbone']) + len(cfg_v1['head'])
DET_V1 = cfg_v1['head'][-1]
assert N_V1 == 22 and list(DET_V1[0]) == [14, 17, 20] and DET_V1[2] == 'Detect', f'Beklenmeyen detector-v1 yapısı: {N_V1}, {DET_V1}'
assert cfg_v1.get('scale') == 'l', f"detector-v1 scale 'l' değil: {cfg_v1.get('scale')}"
assert v1.model[-1].nc == 1, f'detector-v1 sınıf sayısı 1 değil: {v1.model[-1].nc}'
assert v1.stride.tolist() == [8.0, 16.0, 32.0], v1.stride

P2_LAYERS = [
    [14, 1, 'nn.Upsample', [None, 2, 'nearest']],   # 21: P3 çıkışı (katman 14) -> stride 4
    [[-1, 2], 1, 'Concat', [1]],                    # 22: backbone P2/4 (katman 2) ile birleştir
    [-1, 2, 'A2C2f', [256, False, -1]],             # 23: P2 bloğu (katman 14 ile aynı yapı, 256 kanal)
]
cfg_p2 = copy.deepcopy(cfg_v1)
cfg_p2['head'] = cfg_p2['head'][:-1] + P2_LAYERS + [[[23, 14, 17, 20], DET_V1[1], DET_V1[2], copy.deepcopy(DET_V1[3])]]
cfg_p2['yaml_file'] = 'yolov12l-p2 (D2: detector-v1 yaml + P2 dalı)'
OLD_DET = N_V1 - 1                                               # 21
NEW_DET = len(cfg_p2['backbone']) + len(cfg_p2['head']) - 1      # 24
assert (OLD_DET, NEW_DET) == (21, 24)

torch.manual_seed(SEED)
p2 = DetectionModel(copy.deepcopy(cfg_p2), nc=1, verbose=True).float().eval()
det_p2 = p2.model[-1]
assert isinstance(det_p2, Detect) and det_p2.nl == 4 and list(det_p2.f) == [23, 14, 17, 20]
assert p2.stride.tolist() == [4.0, 8.0, 16.0, 32.0], p2.stride
_shapes = lambda m: [tuple(t.shape) for t in m.state_dict().values()]
assert _shapes(det_p2.cv2[0]) == _shapes(det_p2.cv2[1]) and _shapes(det_p2.cv3[0]) == _shapes(det_p2.cv3[1]), \
    'P2 dalı P3 dalıyla aynı şekilde değil (kanal varsayımı bozuk)'

# --- Ağırlık aktarımı (Detect model.21 -> model.24; cv2/cv3 seviye indeksi +1, dfl aynı) ---
src_sd, dst_sd = v1.state_dict(), p2.state_dict()

def remap_key(k):
    p = k.split('.')
    if p[0] == 'model' and p[1] == str(OLD_DET):
        p[1] = str(NEW_DET)
        if p[2] in ('cv2', 'cv3'):
            p[3] = str(int(p[3]) + 1)       # P3, P4, P5 -> Detect giriş indeksi 1, 2, 3 (0 = P2)
        elif p[2] != 'dfl':
            raise KeyError(f'Beklenmeyen Detect anahtarı: {k}')
    return '.'.join(p)

mapped = {}
for k, v in src_sd.items():
    nk = remap_key(k)
    assert nk in dst_sd, f'Hedefte yok: {k} -> {nk}'
    assert dst_sd[nk].shape == v.shape, f'Şekil farkı: {k} {tuple(v.shape)} -> {nk} {tuple(dst_sd[nk].shape)}'
    mapped[nk] = v.clone()
assert len(mapped) == len(src_sd)
new_only = sorted(set(dst_sd) - set(mapped))
_allowed = ('model.21.', 'model.22.', 'model.23.', f'model.{NEW_DET}.cv2.0.', f'model.{NEW_DET}.cv3.0.')
_bad = [k for k in new_only if not k.startswith(_allowed)]
assert not _bad, f'Beklenmeyen yeni anahtarlar: {_bad[:10]}'

# --- P2 başlığı: P3 dalının kopyası; sınıf biası stride 4 önseli (bias_init) ---
head_copy = {}
for k in new_only:
    if k.startswith((f'model.{NEW_DET}.cv2.0.', f'model.{NEW_DET}.cv3.0.')):
        sk = k.replace('.cv2.0.', '.cv2.1.', 1).replace('.cv3.0.', '.cv3.1.', 1)
        head_copy[k] = mapped[sk].clone()
CLS_BIAS_KEY = f'model.{NEW_DET}.cv3.0.2.bias'
assert CLS_BIAS_KEY in head_copy, CLS_BIAS_KEY
_prior = math.log(5 / 1 / (640 / 4) ** 2)
assert torch.allclose(dst_sd[CLS_BIAS_KEY], torch.full_like(dst_sd[CLS_BIAS_KEY], _prior)), 'bias_init beklenen değerde değil'
head_copy[CLS_BIAS_KEY] = dst_sd[CLS_BIAS_KEY].clone()
p2.load_state_dict({**dst_sd, **mapped, **head_copy}, strict=True)
p2 = p2.float().eval()
print(f'Aktarılan: {len(mapped)}/{len(src_sd)} detector-v1 anahtarı | P2 başlığı (P3 kopyası): {len(head_copy)} | '
      f'rastgele (katman 21-23): {len(new_only) - len(head_copy)}')

# --- Eşdeğerlik testi: P3/P4/P5 çıktıları detector-v1 ile birebir aynı olmalı ---
torch.manual_seed(0)
_x = torch.rand(1, 3, IMGSZ, IMGSZ)
with torch.no_grad():
    _y1, _f1 = v1(_x)
    _y2, _f2 = p2(_x)
assert len(_f1) == 3 and len(_f2) == 4
for _i in range(3):
    _d = (_f1[_i] - _f2[_i + 1]).abs().max().item()
    print(f'  P{_i + 3} çıktı farkı (detector-v1 vs P2 modeli): {_d:.2e}')
    assert _d < 1e-4, f'P{_i + 3} çıktıları farklı: aktarım hatalı'
assert _y2.shape[-1] == _y1.shape[-1] + (IMGSZ // 4) ** 2, (_y1.shape, _y2.shape)
print('Eşdeğerlik testi geçti: P3/P4/P5 dalları detector-v1 ile birebir aynı.')

# --- Freeze kontrolü: yeni katmanlar donmamalı (trainer ile aynı kural) ---
_freeze_names = [f'model.{x}.' for x in range(FREEZE)] + ['.dfl']
_new_params = [k for k, _ in p2.named_parameters() if k.startswith(_allowed)]
assert _new_params and not [k for k in _new_params if any(n in k for n in _freeze_names)], 'Yeni katmanlardan biri donuk'

# --- Model maliyeti ---
N_PARAMS = {'v1': get_num_params(v1), 'p2': get_num_params(p2)}
GFLOPS = {'v1': get_flops(v1, IMGSZ), 'p2': get_flops(p2, IMGSZ)}
print('Parametre:', N_PARAMS, '| GFLOPs @640:', GFLOPS, '(0 ise thop yok)')

# --- Başlangıç checkpoint'i (train_args detector-v1'den: D1'de single_cls/imgsz/data bu yoldan geldi) ---
P2_INIT_PT = D2_DIR / 'p2_init_from_detector-v1.pt'
_p2_save = copy.deepcopy(p2).float()
_p2_save.names = getattr(v1, 'names', {0: '0'})
torch.save({'date': time.strftime('%Y-%m-%dT%H:%M:%S'), 'version': ultralytics.__version__, 'model': _p2_save,
            'ema': None, 'updates': None, 'optimizer': None, 'train_args': dict(v1_ckpt.get('train_args', {})),
            'epoch': -1, 'train_metrics': None, 'train_results': None,
            'd2_note': 'detector-v1 + P2 dalı (D2); bkz. D2_prereg.json'}, P2_INIT_PT)
with open(D2_DIR / 'yolov12l-p2.yaml', 'w') as fh:
    yaml.safe_dump(cfg_p2, fh, sort_keys=False, allow_unicode=True)

# --- Eğitim yolu simülasyonu: trainer, checkpoint'teki modeli yaml'dan yeniden kurup ada göre yükler ---
_m_init = YOLO(str(P2_INIT_PT))
_probe = DetectionModel(copy.deepcopy(_m_init.model.yaml), nc=1, verbose=False)
_csd = intersect_dicts(_m_init.model.float().state_dict(), _probe.state_dict())
assert len(_csd) == len(_probe.state_dict()), f'Eğitimde aktarılacak: {len(_csd)}/{len(_probe.state_dict())}'
assert _probe.stride.tolist() == [4.0, 8.0, 16.0, 32.0]
_ov_v1 = YOLO(str(DETECTOR_WEIGHTS)).overrides
_ov_p2 = _m_init.overrides
_ov_diff = {k: (_ov_v1.get(k), _ov_p2.get(k)) for k in set(_ov_v1) | set(_ov_p2) if k != 'model' and _ov_v1.get(k) != _ov_p2.get(k)}
assert not _ov_diff, f'Eğitim argümanları detector-v1 ile farklı: {_ov_diff}'
print(f'Eğitim yolu doğrulandı: {len(_csd)}/{len(_probe.state_dict())} anahtar aktarılacak | overrides: {_ov_p2}')
print('Başlangıç checkpoint:', P2_INIT_PT)
del _probe, _m_init

## Adım 3: Eğitim (D1 ile birebir aynı ayarlar)

- MLflow: deney adı `TIR-JDE-D2`, koşu `D2_p2_train`. Veritabanı yerel diskte tutulur (`/content/local/mlflow_d2.db`); Drive'daki sqlite dosyasında paralel yazım kilitlenmesini önlemek için. Sonda Drive'a kopyalanır (`WORK/D2/mlflow_d2.db`). Artifact'lar `WORK/D2/mlartifacts` altında.
- Eğitim tamamlanmışsa (4 epoch ve `last.pt` var) tekrar edilmez. Yarım kalmışsa baştan başlar (resume değil).
- Eğitimden sonra `args.yaml`, D1 ctrl'ün `args.yaml`'ı ile karşılaştırılır; yalnız `model`, `project`, `name`, `save_dir` farklı olabilir.

In [ ]:
# 3.0) MLflow (yerel sqlite; paralel çalışmada güvenli)
_db_local = LOCAL / 'mlflow_d2.db'
_db_drive = D2_DIR / 'mlflow_d2.db'
if _db_drive.exists() and not _db_local.exists():
    shutil.copy(_db_drive, _db_local)             # önceki oturumun kaydına devam
os.environ['MLFLOW_TRACKING_URI'] = f'sqlite:///{_db_local}'
os.environ['MLFLOW_EXPERIMENT_NAME'] = 'TIR-JDE-D2'
os.environ['MLFLOW_RUN'] = 'D2_p2_train'
try:
    import mlflow
    mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
    if mlflow.get_experiment_by_name('TIR-JDE-D2') is None:
        mlflow.create_experiment('TIR-JDE-D2', artifact_location=(D2_DIR / 'mlartifacts').as_uri())
    MLFLOW_OK = True
    print('MLflow hazır:', os.environ['MLFLOW_TRACKING_URI'])
except Exception as e:
    MLFLOW_OK = False
    print('MLflow kullanılamıyor (eğitim etkilenmez):', repr(e))

In [ ]:
# 3.1) p2 eğitimi
import yaml
P2_RUN = D2_DIR / 'p2'
P2_LAST = P2_RUN / 'weights' / 'last.pt'

def epochs_done(run):
    f = Path(run) / 'results.csv'
    return len(pd.read_csv(f)) if f.exists() else 0

INIT_CHECK = {}

def make_init_check(init_pt):
    """Trainer modeli kurduktan sonra, ilk adımdan önce: ağırlıklar başlangıç checkpoint'iyle birebir aynı mı?
    (Trainer modeli yaml'dan yeniden kurup ada göre yükler; aktarım kaçarsa model sessizce rastgele başlardı.)"""
    _ref_model, _ = attempt_load_one_weight(str(init_pt))
    ref = {k: v.detach().float().cpu() for k, v in _ref_model.state_dict().items()}
    del _ref_model

    def _cb(trainer):
        sd = trainer.model.state_dict()
        assert set(sd) == set(ref), f'Anahtar kümesi farklı: eksik {len(set(ref) - set(sd))}, fazla {len(set(sd) - set(ref))}'
        bad = [k for k in ref if not torch.equal(sd[k].detach().float().cpu(), ref[k])]
        old = [k for k in ref if not k.startswith(('model.21.', 'model.22.', 'model.23.')) and '.cv2.0.' not in k and '.cv3.0.' not in k]
        INIT_CHECK.update(n_keys=len(ref), n_detector_v1_keys=len(old), n_mismatch=len(bad))
        if bad:
            raise RuntimeError(f'Trainer modeli başlangıç checkpoint\'inden farklı ({len(bad)} anahtar, örn. {bad[:3]}). '
                               'Ağırlık aktarımı eğitimde kaybolmuş; eğitim durduruldu.')
        print(f'[init kontrolü] trainer modeli başlangıç checkpoint\'iyle birebir aynı: {len(ref)}/{len(ref)} anahtar '
              f'({len(old)} tanesi detector-v1 kaynaklı)')
    return _cb


if epochs_done(P2_RUN) >= EPOCHS and P2_LAST.exists() and not FORCE_RETRAIN:
    print('p2 eğitimi zaten tamamlanmış; atlanıyor:', P2_LAST)
else:
    os.chdir(YOLO_FORK)                            # D1 ile aynı çalışma dizini
    _t0 = time.time()
    _m = YOLO(str(P2_INIT_PT))
    _m.add_callback('on_pretrain_routine_end', make_init_check(P2_INIT_PT))
    _m.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, lr0=LR0, optimizer=OPT, freeze=FREEZE,
             seed=SEED, project=str(D2_DIR), name='p2', exist_ok=True, val=True, verbose=False)
    print(f'Eğitim süresi: {(time.time() - _t0) / 60:.1f} dk')
    assert INIT_CHECK.get('n_mismatch') == 0 and INIT_CHECK.get('n_keys', 0) > 0, f'init kontrolü çalışmadı: {INIT_CHECK}'
    with open(P2_RUN / 'init_check.json', 'w') as fh:
        json.dump(INIT_CHECK, fh, indent=1)
    del _m
    torch.cuda.empty_cache()

# --- Eğitim sonrası kontroller ---
assert epochs_done(P2_RUN) == EPOCHS and P2_LAST.exists(), 'Eğitim tamamlanmadı'
_a_ctrl = yaml.safe_load(open(D1_CTRL_ARGS))
_a_p2 = yaml.safe_load(open(P2_RUN / 'args.yaml'))
_allowed_diff = {'model', 'project', 'name', 'save_dir'}
_args_diff = {k: (_a_ctrl.get(k), _a_p2.get(k)) for k in set(_a_ctrl) | set(_a_p2)
              if k not in _allowed_diff and _a_ctrl.get(k) != _a_p2.get(k)}
assert not _args_diff, f'D1 ctrl ile eğitim argümanları farklı: {_args_diff}'
_mp2 = YOLO(str(P2_LAST))
assert _mp2.model.stride.tolist() == [4.0, 8.0, 16.0, 32.0], _mp2.model.stride
print('Eğitim argümanları D1 ctrl ile aynı (model/project/name/save_dir hariç). p2 last.pt:', P2_LAST)
del _mp2

## Adım 4: Değerlendirme

- **4.1** Üç kol için tespit (D1'deki `detect_full_model` ile aynı: imgsz 640, conf 0,05) ve tracker: D1 karar protokolü (IoU, K = 0) ve dondurulmuş E2 konfigürasyonu (bilgi). Tespitler ağırlık md5'i ile işaretlenir; ağırlık değişmediyse yeniden üretilmez.
- **4.2** Metrikler, birincil karar, ikincil karşılaştırmalar (Holm), boyuta göre recall, D1 ile tekrar üretilebilirlik, hız ve model maliyeti.
- **4.3** Görseller ve video.
- **4.4** MLflow kaydı ve sonuç dosyaları.

In [ ]:
# 4.1) Tespit ve tracker (üç kol, aynı oturum, aynı kod)
ARMS = {'v1_ref': DETECTOR_WEIGHTS, 'ctrl': D1_CTRL_WEIGHTS, 'p2': P2_LAST}

import gc
gc.collect(); torch.cuda.empty_cache()          # eğitimden kalan GPU belleğini boşalt

def detect_seq(model, seq, conf=DET_CONF, chunk=32):
    """D1 detect_full_model ile aynı çıkarım (imgsz 640, conf 0,05); kareler 32'lik parçalarla işlenir.
    Not: predict'e liste verilince Ultralytics listenin tamamını tek batch yapar; eval modunda sonuç batch'ten bağımsızdır."""
    files = sorted(glob.glob(f'{FRAMES_ROOT}/{seq}/*.jpg'))
    rows = []
    for c0 in range(0, len(files), chunk):
        for k, r in enumerate(model.predict(source=files[c0:c0 + chunk], imgsz=IMGSZ, conf=conf, verbose=False)):
            if r.boxes is None or len(r.boxes) == 0:
                continue
            rows.append(np.c_[np.full(len(r.boxes), c0 + k + 1), r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()])
    return np.vstack(rows) if rows else np.zeros((0, 6))

DET_DIRS = {}
for name, w in ARMS.items():
    dd = D2_DIR / name / 'dets'
    dd.mkdir(parents=True, exist_ok=True)
    stamp, w_md5 = dd / '_weights_md5.txt', _md5(w)
    done = stamp.exists() and stamp.read_text().strip() == w_md5 and all((dd / f'{s}.txt').exists() for s in VAL_SEQS)
    if not done:
        if stamp.exists():
            stamp.unlink()
        _t0 = time.time()
        _model = YOLO(str(w))
        for s in VAL_SEQS:
            np.savetxt(dd / f'{s}.txt', detect_seq(_model, s), delimiter=',', fmt='%.4f')
        stamp.write_text(w_md5)
        del _model
        torch.cuda.empty_cache()
        print(f'{name}: tespit tamam ({time.time() - _t0:.0f} s)')
    else:
        print(f'{name}: tespitler güncel (ağırlık md5 {w_md5[:8]})')
    DET_DIRS[name] = dd

# Dondurulmuş tracker (E2)
_fz_default = {'cmc': 'sparseOptFlow', 'patches': 'nwd', 'match_thresh': 0.8, 'coast_k': 10}
if FROZEN_CONFIG.exists():
    _fz = json.load(open(FROZEN_CONFIG))['kosuya_verilen']
    FROZEN_ARGS = {k: _fz[k] for k in ('cmc', 'patches', 'match_thresh', 'coast_k')}
else:
    FROZEN_ARGS = dict(_fz_default)
if FROZEN_ARGS != _fz_default:
    print('UYARI: frozen_config beklenen E2 seçiminden farklı:', FROZEN_ARGS)
print('Dondurulmuş tracker:', FROZEN_ARGS)

TRK_IOU, TRK_FROZEN = {}, {}
for name, dd in DET_DIRS.items():
    TRK_IOU[name] = str(D2_DIR / name / 'track_iou')
    run_tracker_patched(dd, TRK_IOU[name], patches='', match_thresh=MT['iou'])           # D1 karar protokolü
    TRK_FROZEN[name] = str(D2_DIR / name / 'track_frozen')
    run_tracker_patched(dd, TRK_FROZEN[name], **FROZEN_ARGS)                              # bilgi
print('Tracker koşuları tamam.')

In [ ]:
# 4.2) Metrikler, karar, ikincil karşılaştırmalar, tekrar üretilebilirlik, hız
COLS = ['MOTA', 'MOTA_seq', 'HOTA', 'DetA', 'AssA', 'IDF1', 'IDSW', 'Frag', 'FP', 'FN']
tab_iou = summary_table(TRK_IOU, 'D2_iou')[COLS].astype(float)
tab_fz = summary_table(TRK_FROZEN, 'D2_frozen')[COLS].astype(float)
print('D1 karar protokolü (IoU, K=0):'); print(tab_iou.round(4))
print('\nDondurulmuş tracker (bilgi):'); print(tab_fz.round(4))

PS = {a: per_seq_table(d, f'D2_iou_{a}') for a, d in TRK_IOU.items()}
PS_FZ = {a: per_seq_table(d, f'D2_frozen_{a}') for a, d in TRK_FROZEN.items()}

# --- Tekrar üretilebilirlik (D1 sayfasındaki değerler) ---
REPRO = {}
for arm, ref in D1_REF.items():
    for k in ('MOTA', 'HOTA', 'DetA', 'AssA'):
        REPRO[f'{arm}.{k}'] = round(float(tab_iou.loc[arm, k]) - ref[k], 4)
    REPRO[f'{arm}.IDSW'] = int(tab_iou.loc[arm, 'IDSW'] - ref['IDSW'])
REPRO_OK = all(abs(v) <= REPRO_TOL for k, v in REPRO.items() if not k.endswith('IDSW')) and \
    all(abs(REPRO[f'{a}.IDSW']) <= 0.02 * D1_REF[a]['IDSW'] for a in D1_REF)
print('\nD1 ile fark:', REPRO, '| TUTARLI' if REPRO_OK else
      '| UYARI: D1 değerleri birebir üretilemedi. Karar yine geçerlidir (ctrl ve p2 bu oturumda aynı kodla koşuldu); raporda belirtin.')

# --- Birincil karar (önceden kayıtlı) ---
PRIMARY = compare(PS['ctrl'], PS['p2'], 'DetA')
ACCEPT = PRIMARY['CI95'][0] > 0
DECISION = 'detector-v1-P2 kabul edilir' if ACCEPT else 'detector-v1 kalır'
COMBINED_DETA_DIFF = round(float(tab_iou.loc['p2', 'DetA'] - tab_iou.loc['ctrl', 'DetA']), 4)
print('\nBİRİNCİL  DetA (p2 - ctrl; sekans başına farkların ortalaması, eşleştirilmiş bootstrap):', PRIMARY)
print('Birleşik (combined) DetA farkı, nokta tahmini:', COMBINED_DETA_DIFF)
print('D2 KARAR:', DECISION)

# --- Boyut sınıfına göre recall (IoU >= 0,5, skor >= 0,09; D1 ap_and_recall ile aynı eşleşme) ---
BINS = [(0, 8, '<8'), (8, 16, '8-16'), (16, 1e9, '>=16')]

def recall_by_bin(dets_dir, score_min=0.09, thr=0.5):
    rows = []
    for s in VAL_SEQS:
        gt = load_mot(DV1 / 'gt' / f'{s}.txt')
        gt = gt[gt[:, 6] == 1]
        dets = load_dets(Path(dets_dir) / f'{s}.txt')
        dets = dets[dets[:, 5] >= score_min] if len(dets) else dets
        side = np.sqrt(gt[:, 4] * gt[:, 5])
        hit = np.zeros(len(gt), bool)
        for f in np.unique(gt[:, 0]):
            gi = np.where(gt[:, 0] == f)[0]
            d = dets[dets[:, 0] == f] if len(dets) else np.zeros((0, 6))
            for a, _, _ in match_frame(xywh2xyxy(gt[gi, 2:6]), d[:, 1:5], thr):
                hit[gi[a]] = True
        for lo, hi, lab in BINS:
            m = (side >= lo) & (side < hi)
            rows.append({'seq': s, 'bin': lab, 'n': int(m.sum()), 'matched': int(hit[m].sum())})
    return pd.DataFrame(rows)

RB = {a: recall_by_bin(DET_DIRS[a]) for a in ARMS}
REC_POOLED = pd.DataFrame({a: df.groupby('bin').matched.sum() / df.groupby('bin').n.sum() for a, df in RB.items()}).loc[[b[2] for b in BINS]]
REC_N = RB['ctrl'].groupby('bin').n.sum().loc[[b[2] for b in BINS]]
print('\nBoyut sınıfına göre recall (havuzlanmış):'); print(REC_POOLED.round(4).assign(n_GT=REC_N))

def small_per_seq(df):
    x = df[(df.bin == '<8') & (df.n > 0)].set_index('seq')
    return x.matched / x.n

_sc, _sp = small_per_seq(RB['ctrl']), small_per_seq(RB['p2'])
assert list(_sc.index) == list(_sp.index)
_m, _ci, _p = bootstrap_diff(_sc.values, _sp.values)
SECONDARY = [{'metrik': 'recall_<8px', 'fark': round(_m, 4), 'CI95': (round(_ci[0], 4), round(_ci[1], 4)), 'p': round(_p, 4)}]
for met in ('HOTA', 'AssA', 'IDSW'):
    SECONDARY.append(compare(PS['ctrl'], PS['p2'], met))
SEC_DF = pd.DataFrame(SECONDARY)
SEC_DF['p_holm'] = holm(SEC_DF.p.values).round(4)
print(f'\nİKİNCİL (p2 - ctrl, Holm; <8 px recall için {len(_sc)} sekans):'); print(SEC_DF.to_string(index=False))

# --- Aday anchor kapsaması (mekanizma): merkezi GT kutusunun içine düşen anchor sayısı, P3 (stride 8) ve P2 (stride 4) ---
def n_centers(lo, hi, s):
    """(lo, hi) aralığına kesin içeride düşen (i + 0.5)·s anchor merkezi sayısı (TAL aday kuralı)."""
    i_min = np.floor(lo / s - 0.5) + 1
    i_max = np.ceil(hi / s - 0.5) - 1
    return np.maximum(i_max - i_min + 1, 0)

_h0, _w0 = cv2.imread(sorted(glob.glob(f'{FRAMES_ROOT}/{VAL_SEQS[0]}/*.jpg'))[0]).shape[:2]
_r = IMGSZ / max(_h0, _w0)                         # letterbox ölçeği (dolgu kayması yok sayılır)
_ac = []
for s in VAL_SEQS:
    gt = load_mot(DV1 / 'gt' / f'{s}.txt')
    gt = gt[gt[:, 6] == 1]
    x, y, w, h = (gt[:, 2] * _r, gt[:, 3] * _r, gt[:, 4] * _r, gt[:, 5] * _r)
    side = np.sqrt(gt[:, 4] * gt[:, 5])            # boyut sınıfı orijinal piksellerde (recall ile aynı)
    _ac.append(pd.DataFrame({'side': side,
                             'n_P3': n_centers(x, x + w, 8) * n_centers(y, y + h, 8),
                             'n_P2': n_centers(x, x + w, 4) * n_centers(y, y + h, 4)}))
_ac = pd.concat(_ac, ignore_index=True)
_ac['bin'] = pd.cut(_ac.side, [b[0] for b in BINS] + [BINS[-1][1]], labels=[b[2] for b in BINS], right=False)
ANCHOR_COV = _ac.groupby('bin', observed=True).agg(
    n_GT=('side', 'size'),
    P3_aday_yok=('n_P3', lambda v: float((v == 0).mean())),
    P2_aday_yok=('n_P2', lambda v: float((v == 0).mean())),
    P3_medyan_aday=('n_P3', 'median'), P2_medyan_aday=('n_P2', 'median'))
print(f'\nAday anchor kapsaması (yaklaşık; kare {_w0}x{_h0}, ölçek {_r:.3f}, augment yok):'); print(ANCHOR_COV.round(3))

# --- Bilgi ---
INFO = {'DetA p2 - v1_ref': compare(PS['v1_ref'], PS['p2'], 'DetA'),
        'DetA ctrl - v1_ref': compare(PS['v1_ref'], PS['ctrl'], 'DetA'),
        'frozen HOTA p2 - ctrl': compare(PS_FZ['ctrl'], PS_FZ['p2'], 'HOTA'),
        'frozen MOTA_seq p2 - ctrl': compare(PS_FZ['ctrl'], PS_FZ['p2'], 'MOTA'),
        'frozen IDSW p2 - ctrl': compare(PS_FZ['ctrl'], PS_FZ['p2'], 'IDSW')}
print('\nBİLGİ:')
for k, v in INFO.items():
    print(f'  {k}: {v}')

# --- Eğitim sonu doğrulama metrikleri (Ultralytics val, son epoch) ---
def last_val(results_csv):
    r = pd.read_csv(results_csv)
    r.columns = r.columns.str.strip()
    keys = ['metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']
    return {k.split('/')[1].replace('(B)', ''): float(r[k].iloc[-1]) for k in keys}

VAL_DET = {'ctrl': last_val(D1_CTRL_RESULTS), 'p2': last_val(P2_RUN / 'results.csv')}
print('\nUltralytics val (son epoch):'); print(pd.DataFrame(VAL_DET).T.round(4))

# --- Hız (aynı GPU, batch 1, ilk 30 kare ısınma) ---
def speed(weights, seq, n=300, warm=30):
    """Gerçek batch 1: her kare ayrı predict çağrısıyla."""
    model = YOLO(str(weights))
    files = sorted(glob.glob(f'{FRAMES_ROOT}/{seq}/*.jpg'))[:warm + n]
    sp = [model.predict(source=f, imgsz=IMGSZ, conf=DET_CONF, verbose=False)[0].speed for f in files][warm:]
    med = {k: float(np.median([x[k] for x in sp])) for k in ('preprocess', 'inference', 'postprocess')}
    med['fps_uctan_uca'] = 1000.0 / sum(med.values())
    del model
    torch.cuda.empty_cache()
    return med

_speed_seq = max(VAL_SEQS, key=lambda s: len(glob.glob(f'{FRAMES_ROOT}/{s}/*.jpg')))
SPEED = {a: speed(ARMS[a], _speed_seq) for a in ARMS}
print(f'\nHız (ms/kare, medyan, fp32, batch 1; sekans {_speed_seq}). "inference" NMS hariç; '
      '"postprocess" NMS dahil; fps_uctan_uca = 1000 / (pre + inference + post):')
print(pd.DataFrame(SPEED).T.round(2))

In [ ]:
# 4.3) Görseller ve video
import matplotlib.pyplot as plt
FIGS = []

# (a) Boyut sınıfına göre recall
fig, ax = plt.subplots(figsize=(7, 3.6))
_x = np.arange(len(BINS)); _w = 0.26
for i, a in enumerate(ARMS):
    ax.bar(_x + (i - 1) * _w, REC_POOLED[a].values, _w, label=a)
ax.set_xticks(_x, [f'{b[2]} px\n(n={int(REC_N[b[2]])})' for b in BINS])
ax.set_ylabel('recall (IoU ≥ 0,5, skor ≥ 0,09)'); ax.set_ylim(0, 1); ax.legend(); ax.set_title('Boyut sınıfına göre recall')
fig.tight_layout(); _f = D2_DIR / 'D2_recall_by_size.png'; fig.savefig(_f, dpi=130); FIGS.append(_f); plt.show()

# (b) Eğitim eğrileri: ctrl vs p2
def _res(p):
    r = pd.read_csv(p); r.columns = r.columns.str.strip(); return r
_rc, _rp = _res(D1_CTRL_RESULTS), _res(P2_RUN / 'results.csv')
fig, axs = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, k in zip(axs, ['train/box_loss', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']):
    ax.plot(_rc['epoch'], _rc[k], 'o-', label='ctrl'); ax.plot(_rp['epoch'], _rp[k], 's-', label='p2')
    ax.set_title(k); ax.set_xlabel('epoch'); ax.legend()
fig.tight_layout(); _f = D2_DIR / 'D2_training_curves.png'; fig.savefig(_f, dpi=130); FIGS.append(_f); plt.show()

# (c) Sekans başına DetA farkı (p2 - ctrl)
_dd = (PS['p2'].loc[VAL_SEQS, 'DetA'].astype(float) - PS['ctrl'].loc[VAL_SEQS, 'DetA'].astype(float)).sort_values()
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(range(len(_dd)), _dd.values, color=['#c0504d' if v < 0 else '#4f81bd' for v in _dd.values])
ax.set_xticks(range(len(_dd)), _dd.index, rotation=60, ha='right'); ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('DetA farkı'); ax.set_title('Sekans başına DetA (p2 − ctrl)')
fig.tight_layout(); _f = D2_DIR / 'D2_detA_per_seq.png'; fig.savefig(_f, dpi=130); FIGS.append(_f); plt.show()

# (d) Küçük hedef örnekleri (<8 px): v1_ref / ctrl / p2 tespitleri yan yana, yakınlaştırılmış kırpıntılar
#     Yanlılık olmasın diye dört durumdan da örnek: p2 bulur-ctrl kaçırır, ctrl bulur-p2 kaçırır, ikisi bulur, ikisi kaçırır
def _hit(dets, f, gbox, thr=0.5, score_min=0.09):
    d = dets[(dets[:, 0] == f) & (dets[:, 5] >= score_min)] if len(dets) else np.zeros((0, 6))
    return bool(len(d)) and bool((iou_matrix(gbox, d[:, 1:5]) >= thr).any())

_rng = np.random.default_rng(0)
_dets = {a: {s: load_dets(DET_DIRS[a] / f'{s}.txt') for s in VAL_SEQS} for a in ARMS}
_cases = {'p2 bulur, ctrl kaçırır': [], 'ctrl bulur, p2 kaçırır': [], 'ikisi de bulur': [], 'ikisi de kaçırır': []}
_gts = {}
for s in VAL_SEQS:
    gt = load_mot(DV1 / 'gt' / f'{s}.txt')
    gt = gt[gt[:, 6] == 1]
    _gts[s] = gt
    idx = np.where(np.sqrt(gt[:, 4] * gt[:, 5]) < 8)[0]
    for j in (_rng.choice(idx, size=min(len(idx), 60), replace=False) if len(idx) else []):
        f, gb = int(gt[j, 0]), xywh2xyxy(gt[j:j + 1, 2:6])
        hc, hp = _hit(_dets['ctrl'][s], f, gb), _hit(_dets['p2'][s], f, gb)
        key = ('ikisi de bulur' if hc else 'p2 bulur, ctrl kaçırır') if hp else ('ctrl bulur, p2 kaçırır' if hc else 'ikisi de kaçırır')
        _cases[key].append((s, f, int(j)))
EXAMPLES = []
for key, k in (('p2 bulur, ctrl kaçırır', 2), ('ctrl bulur, p2 kaçırır', 2), ('ikisi de bulur', 1), ('ikisi de kaçırır', 1)):
    lst = _cases[key]
    EXAMPLES += [(key,) + lst[i] for i in _rng.permutation(len(lst))[:k]]
print('Örnek havuzu (rastgele <8 px GT, sekans başına en çok 60):', {k: len(v) for k, v in _cases.items()})

CROP, ZOOM = 40, 6

def crop_with_boxes(seq, f, g, dets):
    files = sorted(glob.glob(f'{FRAMES_ROOT}/{seq}/*.jpg'))
    img = cv2.imread(files[f - 1])
    H, W = img.shape[:2]
    cx, cy = g[2] + g[4] / 2, g[3] + g[5] / 2
    x0 = int(np.clip(cx - CROP / 2, 0, W - CROP)); y0 = int(np.clip(cy - CROP / 2, 0, H - CROP))
    patch = cv2.resize(img[y0:y0 + CROP, x0:x0 + CROP], None, fx=ZOOM, fy=ZOOM, interpolation=cv2.INTER_NEAREST)

    def _rect(b, color):
        cv2.rectangle(patch, (int((b[0] - x0) * ZOOM), int((b[1] - y0) * ZOOM)),
                      (int((b[2] - x0) * ZOOM), int((b[3] - y0) * ZOOM)), color, 1)

    _rect(xywh2xyxy(g[None, 2:6])[0], (0, 255, 0))                       # GT: yeşil
    d = dets[(dets[:, 0] == f) & (dets[:, 5] >= 0.09)] if len(dets) else np.zeros((0, 6))
    for r in d:                                                           # kare, x1, y1, x2, y2, skor
        if r[3] >= x0 and r[1] <= x0 + CROP and r[4] >= y0 and r[2] <= y0 + CROP:
            _rect(r[1:5], (0, 0, 255))                                    # tespit: kırmızı
            cv2.putText(patch, f'{r[5]:.2f}', (max(int((r[1] - x0) * ZOOM), 0), max(int((r[2] - y0) * ZOOM) - 3, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
    return patch[..., ::-1]

if EXAMPLES:
    fig, axs = plt.subplots(len(EXAMPLES), len(ARMS), figsize=(3.0 * len(ARMS), 3.0 * len(EXAMPLES)), squeeze=False)
    for r_i, (key, s, f, j) in enumerate(EXAMPLES):
        for c_i, a in enumerate(ARMS):
            ax = axs[r_i][c_i]
            ax.imshow(crop_with_boxes(s, f, _gts[s][j], _dets[a][s])); ax.set_xticks([]); ax.set_yticks([])
            if r_i == 0:
                ax.set_title(a)
            if c_i == 0:
                ax.set_ylabel(f'{key}\n{s} kare {f}', fontsize=8)
    fig.suptitle('<8 px hedefler: yeşil GT, kırmızı tespit (skor ≥ 0,09); 40×40 px kırpıntı, 6× büyütme', fontsize=9)
    fig.tight_layout(); _f = D2_DIR / 'D2_small_examples.png'; fig.savefig(_f, dpi=120); FIGS.append(_f); plt.show()

# (e) Video: küçük hedef payı en yüksek sekans, ctrl ve p2 yan yana (D1 protokolü izleri)
_share = {s: RB['ctrl'][(RB['ctrl'].seq == s) & (RB['ctrl'].bin == '<8')].n.sum() / max(RB['ctrl'][RB['ctrl'].seq == s].n.sum(), 1)
          for s in VAL_SEQS}
VIDEO_SEQ = max(_share, key=_share.get)
_gt = load_mot(DV1 / 'gt' / f'{VIDEO_SEQ}.txt'); _gt = _gt[_gt[:, 6] == 1]
_vids = {}
for a in ('ctrl', 'p2'):
    _vids[a] = LOCAL / f'D2_{VIDEO_SEQ}_{a}.mp4'
    render_video(VIDEO_SEQ, _gt, load_mot(Path(TRK_IOU[a]) / f'{VIDEO_SEQ}.txt'), _vids[a], f0=1, f1=300)

def side_by_side(a, b, out, labels=('ctrl', 'p2')):
    ca, cb = cv2.VideoCapture(str(a)), cv2.VideoCapture(str(b))
    fps = ca.get(cv2.CAP_PROP_FPS) or 10
    vw = None
    while True:
        ok1, f1 = ca.read(); ok2, f2 = cb.read()
        if not (ok1 and ok2):
            break
        for fr, t in ((f1, labels[0]), (f2, labels[1])):
            cv2.putText(fr, t, (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
        frame = np.hstack([f1, f2])
        if vw is None:
            vw = cv2.VideoWriter(str(out), cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame.shape[1], frame.shape[0]))
        vw.write(frame)
    ca.release(); cb.release()
    if vw is not None:
        vw.release()
    return out

VIDEO = D2_DIR / f'D2_{VIDEO_SEQ}_ctrl_vs_p2.mp4'
_tmp_vid = side_by_side(_vids['ctrl'], _vids['p2'], LOCAL / VIDEO.name)
shutil.copy(_tmp_vid, VIDEO)
print(f'Video sekansı: {VIDEO_SEQ} (<8 px GT payı {_share[VIDEO_SEQ]:.2f}) | yeşil = GT, kırmızı = tahmin')
print('Görseller:', [str(f) for f in FIGS] + [str(VIDEO)])

In [ ]:
# 4.4) Sonuç dosyaları ve MLflow
def _py(o):
    if hasattr(o, 'item'):
        return o.item()
    if isinstance(o, (Path,)):
        return str(o)
    return str(o)

INIT_CHECK_SAVED = json.load(open(P2_RUN / 'init_check.json')) if (P2_RUN / 'init_check.json').exists() else {}
RESULTS = {
    'karar': DECISION, 'kabul': bool(ACCEPT), 'birincil_DetA_p2_eksi_ctrl': PRIMARY,
    'birlesik_DetA_farki_nokta': COMBINED_DETA_DIFF, 'on_kayit_sha256': PREREG_SHA256,
    'init_kontrolu': INIT_CHECK_SAVED, 'aday_anchor_kapsamasi': ANCHOR_COV.round(4).to_dict(orient='index'),
    'kucuk_hedef_ornekleri': [list(e) for e in EXAMPLES],
    'ikincil_holm': SEC_DF.to_dict(orient='records'), 'bilgi': INFO,
    'tablo_D1_protokolu': tab_iou.round(4).to_dict(orient='index'),
    'tablo_dondurulmus': tab_fz.round(4).to_dict(orient='index'),
    'recall_boyut': REC_POOLED.round(4).to_dict(), 'recall_boyut_n': REC_N.to_dict(),
    'ultralytics_val': VAL_DET, 'hiz_ms': SPEED, 'parametre': N_PARAMS, 'gflops': GFLOPS,
    'tekrar_uretilebilirlik_D1': {'fark': REPRO, 'tutarli': bool(REPRO_OK)},
    'betik_kaynagi': SCRIPTS_SRC, 'fork_commit': SB_COMMIT, 'tracker_dondurulmus': FROZEN_ARGS,
    'video': str(VIDEO), 'gorseller': [str(f) for f in FIGS],
    'tarih': time.strftime('%Y-%m-%d %H:%M:%S'),
}
with open(D2_DIR / 'D2_results.json', 'w') as fh:
    json.dump(RESULTS, fh, indent=1, ensure_ascii=False, default=_py)
with open(D2_DIR / 'D2_decision.json', 'w') as fh:
    json.dump({'karar': DECISION, 'kabul': bool(ACCEPT), 'DetA': PRIMARY}, fh, indent=1, ensure_ascii=False, default=_py)
print('Kaydedildi:', D2_DIR / 'D2_results.json')

if MLFLOW_OK:
    try:
        mlflow.set_experiment('TIR-JDE-D2')
        with mlflow.start_run(run_name='D2_eval'):
            mlflow.log_params({'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'lr0': LR0, 'optimizer': OPT,
                               'freeze': FREEZE, 'seed': SEED, 'det_conf': DET_CONF, 'fork_commit': SB_COMMIT,
                               'p2_head_init': 'p3_copy', 'karar': DECISION})
            _met = {'DetA_diff_p2_ctrl': PRIMARY['fark'], 'DetA_ci_lo': PRIMARY['CI95'][0],
                    'DetA_ci_hi': PRIMARY['CI95'][1], 'DetA_p': PRIMARY['p'], 'kabul': float(ACCEPT),
                    'repro_ok': float(REPRO_OK)}
            for r in SECONDARY:
                _key = r['metrik'].replace('<', 'lt').replace('_', '')
                _met[f'{_key}_diff_p2_ctrl'] = r['fark']
            for a in ARMS:
                for k in ('HOTA', 'DetA', 'AssA', 'MOTA_seq', 'IDSW'):
                    _met[f'{a}_{k}'] = float(tab_iou.loc[a, k])
                for b in REC_POOLED.index:
                    _met[f'{a}_recall_{b.replace("<", "lt").replace(">=", "ge").replace("-", "to")}'] = float(REC_POOLED.loc[b, a])
            for a, s in SPEED.items():
                _met[f'{a}_inference_ms'] = s['inference']
            mlflow.log_metrics(_met)
            for f in FIGS + [D2_DIR / 'D2_results.json', D2_DIR / 'D2_prereg.json', D2_DIR / 'yolov12l-p2.yaml']:
                mlflow.log_artifact(str(f))
        print('MLflow: D2_eval kaydedildi.')
    except Exception as e:
        print('MLflow kaydı başarısız (sonuçlar dosyalarda):', repr(e))
if (LOCAL / 'mlflow_d2.db').exists():
    shutil.copy(LOCAL / 'mlflow_d2.db', D2_DIR / 'mlflow_d2.db')
    print('MLflow veritabanı Drive\'a kopyalandı:', D2_DIR / 'mlflow_d2.db')

## Adım 5: Confluence raporu

Aşağıdaki hücre raporu sonuçlardan **otomatik** üretir (elle sayı girilmez). Çıktıyı Confluence'a yapıştırabilir veya bana gönderebilirsiniz. Dosya: `WORK/D2/D2_confluence_raporu.md`. Görselleri ve videoyu sayfaya ayrıca ekleyin.

In [ ]:
# 5.1) Confluence raporu (sonuçlardan otomatik)
def f4(x):
    return f'{float(x):.4f}'.replace('.', ',')

def f3(x):
    return f'{float(x):.3f}'.replace('.', ',')

def ci(c):
    return f'({f4(c["CI95"][0])}; {f4(c["CI95"][1])}), p {f4(c["p"])}'

_rows = '\n'.join(
    f'| {a} | {f4(tab_iou.loc[a, "MOTA_seq"])} | {f4(tab_iou.loc[a, "HOTA"])} | {f4(tab_iou.loc[a, "DetA"])} | '
    f'{f4(tab_iou.loc[a, "AssA"])} | {int(tab_iou.loc[a, "IDSW"])} | {int(tab_iou.loc[a, "FP"])} | {int(tab_iou.loc[a, "FN"])} |'
    for a in ARMS)
_rows_fz = '\n'.join(
    f'| {a} | {f4(tab_fz.loc[a, "MOTA_seq"])} | {f4(tab_fz.loc[a, "HOTA"])} | {int(tab_fz.loc[a, "IDSW"])} |' for a in ARMS)
_rows_rec = '\n'.join(
    f'| {b} px | {int(REC_N[b])} | ' + ' | '.join(f4(REC_POOLED.loc[b, a]) for a in ARMS) + ' |' for b in REC_POOLED.index)
_rows_sec = '\n'.join(
    f'| {r.metrik} | {f4(r.fark)} | ({f4(r.CI95[0])}; {f4(r.CI95[1])}) | {f4(r.p)} | {f4(r.p_holm)} |' for r in SEC_DF.itertuples())
_rows_anc = '\n'.join(
    f'| {b} px | {int(r.n_GT)} | %{100 * r.P3_aday_yok:.1f} | %{100 * r.P2_aday_yok:.1f} | {r.P3_medyan_aday:g} | {r.P2_medyan_aday:g} |'
    for b, r in zip(ANCHOR_COV.index, ANCHOR_COV.itertuples()))
_rows_speed = '\n'.join(
    f'| {a} | {SPEED[a]["inference"]:.1f} | {SPEED[a]["postprocess"]:.1f} | {SPEED[a]["fps_uctan_uca"]:.0f} |' for a in SPEED)
_init_txt = (f"Trainer'ın kurduğu model, başlangıç checkpoint'iyle birebir aynı ({INIT_CHECK_SAVED.get('n_keys')} anahtar, "
             f"{INIT_CHECK_SAVED.get('n_mismatch')} fark; {INIT_CHECK_SAVED.get('n_detector_v1_keys')} anahtar detector-v1 kaynaklı)."
             if INIT_CHECK_SAVED else 'init_check.json bulunamadı.')

REPORT = f"""## D2: P2 katmanı (stride 4), sonuç raporu

**Durum:** Geliştirme seti (val, 16 sekans), tek tohum (seed 0). Deney tamamlandı. **Karar: {DECISION}.**

**Ne:** detector-v1'e (SOT → MOT fine-tune edilmiş YOLOv12l) stride 4'te çalışan bir P2 çıkış dalı eklendi. Katman 0–20 ve Detect'in P3/P4/P5 dalları detector-v1'den birebir aktarıldı (eşdeğerlik testiyle doğrulandı). Yeni katmanlar: 21 Upsample, 22 Concat (backbone P2), 23 A2C2f (256 kanal), Detect'in P2 dalı (P3 dalından başlatıldı).

**Neden:** Küçük hedef recall'u düşük (E1a: <8 px GT'lerin %18'i IoU ≥ 0,3 ile bile eşleşmiyor). Atayıcı (TaskAlignedAssigner) yalnız merkezi GT kutusunun içine düşen anchor'ları aday alır; P3'te anchor merkezleri 8 px aralıklı olduğu için stride'dan küçük hedeflerde aday anchor oluşmayabilir.

**Protokol:** D1 ile birebir: 4 epoch, batch 16, imgsz 640, AdamW lr0 0,001, freeze 10, seed 0, aynı veri (manifest ile doğrulandı), last.pt. Eğitim argümanları D1 ctrl'ün args.yaml'ı ile karşılaştırıldı (yalnız model/project/name/save_dir farklı). {_init_txt} Kontrol: D1'in CIoU fine-tune'u (ctrl). Tracker: SB BoT-SORT, IoU, K = 0. Karar kuralı eğitimden önce kilitlendi (D2_prereg.json, sha256 {PREREG_SHA256[:16]}…; fork {SB_COMMIT[:7]}).

**İstatistik:** Eşleştirilmiş sekans bootstrap (B = 10.000); fark = sekans başına farkların ortalaması (D1/E serisi ile aynı). Küçük hedef: √(w·h) < 8 px; recall IoU ≥ 0,5, skor ≥ 0,09.

### Sonuç (D1 karar protokolü: IoU, K = 0)

| Kol | MOTA_seq | HOTA | DetA | AssA | IDSW | FP | FN |
| --- | --- | --- | --- | --- | --- | --- | --- |
{_rows}

**Birincil:** DetA farkı (p2 − ctrl) = {f4(PRIMARY['fark'])}, CI95 {ci(PRIMARY)}. **{DECISION}.** Birleşik (combined) DetA farkı, nokta tahmini: {f4(COMBINED_DETA_DIFF)}.

**İkincil (Holm, kararı değiştirmez):**

| Metrik | Fark (p2 − ctrl) | CI95 | p | p (Holm) |
| --- | --- | --- | --- | --- |
{_rows_sec}

### Boyut sınıfına göre recall (IoU ≥ 0,5, skor ≥ 0,09)

| Boyut | GT sayısı | {' | '.join(ARMS)} |
| --- | --- | {' | '.join('---' for _ in ARMS)} |
{_rows_rec}

### Mekanizma: aday anchor kapsaması (yaklaşık; augment yok)

Merkezi GT kutusunun içine düşen anchor sayısı (TaskAlignedAssigner aday kuralı). "Aday yok" = o seviyede hiç pozitif örnek üretemeyen GT payı.

| Boyut | GT | P3 aday yok | P2 aday yok | P3 medyan aday | P2 medyan aday |
| --- | --- | --- | --- | --- | --- |
{_rows_anc}

### Bilgi

- DetA p2 − v1_ref: {f4(INFO['DetA p2 - v1_ref']['fark'])}, CI95 {ci(INFO['DetA p2 - v1_ref'])}. DetA ctrl − v1_ref: {f4(INFO['DetA ctrl - v1_ref']['fark'])}, CI95 {ci(INFO['DetA ctrl - v1_ref'])} (aynı mimaride fine-tune gürültüsü için kaba referans).
- Dondurulmuş tracker ({FROZEN_ARGS['cmc']} + {FROZEN_ARGS['patches'] or 'IoU'} + K{FROZEN_ARGS['coast_k']}): HOTA p2 − ctrl {f4(INFO['frozen HOTA p2 - ctrl']['fark'])}, CI95 {ci(INFO['frozen HOTA p2 - ctrl'])}; MOTA_seq p2 − ctrl {f4(INFO['frozen MOTA_seq p2 - ctrl']['fark'])}, CI95 {ci(INFO['frozen MOTA_seq p2 - ctrl'])}.

| Kol (dondurulmuş tracker) | MOTA_seq | HOTA | IDSW |
| --- | --- | --- | --- |
{_rows_fz}

- Ultralytics val (son epoch): ctrl mAP50 {f3(VAL_DET['ctrl']['mAP50'])}, mAP50-95 {f3(VAL_DET['ctrl']['mAP50-95'])}; p2 mAP50 {f3(VAL_DET['p2']['mAP50'])}, mAP50-95 {f3(VAL_DET['p2']['mAP50-95'])}.
- Model: parametre {N_PARAMS['v1']:,} → {N_PARAMS['p2']:,}; GFLOPs @640 {GFLOPS['v1']:.1f} → {GFLOPS['p2']:.1f}.

Hız (aynı GPU, fp32, batch 1, ısınma sonrası medyan; çıkarım NMS hariç, son işlem NMS dahil):

| Kol | Çıkarım (ms) | Son işlem + NMS (ms) | Uçtan uca FPS |
| --- | --- | --- | --- |
{_rows_speed}

**Tekrar üretilebilirlik:** v1_ref ve ctrl, D1 sayfasındaki değerlerle {'tutarlı' if REPRO_OK else 'birebir tutmadı'} (farklar: {REPRO}). {'' if REPRO_OK else 'Karar yine geçerlidir, çünkü ctrl ve p2 bu oturumda aynı kodla koşuldu.'}

**Sınırlar:**
- Tek seed; val 16 sekans. Seed varyansı CI'ya girmez; ctrl − v1_ref farkı fine-tune gürültüsü için kaba referanstır.
- 4 epoch yeni katmanlar için kısa olabilir. P2 ile anchor sayısı ~8.400'den ~34.000'e çıkar; çoğu negatif olduğundan sınıflandırma kaybının dengesi ve atamalar değişir. Olumsuz sonuç, "bu eğitim bütçesinde fayda göstermedi" anlamına gelir.
- Freeze yalnız gradyanı durdurur; donuk backbone'daki BN istatistikleri train modunda güncellenir (ctrl'de de aynı).
- <8 px recall karşılaştırması {len(_sc)} sekans üzerinden (küçük GT'si olmayan sekanslar dışarıda). Aday anchor kapsaması yaklaşıktır (augment ve dolgu kayması yok sayıldı).

**Görseller:** D2_recall_by_size.png, D2_small_examples.png (<8 px örnekler; dört durumdan da örnek), D2_training_curves.png, D2_detA_per_seq.png, {VIDEO.name} (sekans {VIDEO_SEQ}; yeşil GT, kırmızı tahmin).

**Dosyalar:** WORK/D2/D2_results.json, D2_prereg.json, yolov12l-p2.yaml, p2/weights/last.pt. MLflow: deney TIR-JDE-D2 (D2_p2_train, D2_eval).
"""
(D2_DIR / 'D2_confluence_raporu.md').write_text(REPORT, encoding='utf-8')
print(REPORT)

## D2b: Merkez uzaklığına dayalı NMS (yinelenen kutular)

**Sorun:** D2'de p2'nin yinelenen tespit oranı üç kat (aynı karede merkezleri 3 px'ten yakın tespit çiftleri: v1_ref ‰49, ctrl ‰51, p2 ‰151). Kutular 4–8 px olduğu için 1–2 px kayan iki kutunun IoU'su NMS eşiğinin (0,7) altında kalıyor; standart NMS bunları elemiyor.

**Ne yapacağız:** Kaydedilmiş tespitlere ek bir NMS adımı: her karede skora göre sırala; tutulan bir kutuya merkezi **τ = 3 px**'ten yakın olan, daha düşük skorlu kutuları sil. Eğitim ve yeniden tespit yok; yalnızca tracker koşuları tekrarlanır.

**Neden 3 px:** Yinelenen kutu ölçümünde kullanılan eşik. Izgara taranmaz (tek değer, sonuçlara göre ayarlama yapılmaz). Gerçek iki hedefin bu kadar yakın olma payı GT üzerinden ayrıca raporlanır (yanlışlıkla silinebilecek gerçek hedeflerin üst sınırı).

**Kollar:** ctrl ve p2, her biri NMS'siz ve NMS'li. Karşılaştırma aynı detektörün kendi içinde: p2+cNMS − p2.

**Önceden yazılmış karar kuralı** (`D2_DIR/D2b_prereg.json`, çalıştırmadan önce kilitlenir):
- **Birincil:** Dondurulmuş tracker (E2: sparseOptFlow + NWD + K10) ile HOTA farkı (p2+cNMS − p2), eşleştirilmiş sekans bootstrap, %95 CI.
- **Koruma:** MOTA_seq farkının CI üst sınırı ≥ 0 olmalı (resmî metrikte anlamlı düşüş olmamalı).
- **Karar:** HOTA CI alt sınırı > 0 **ve** koruma sağlanırsa cNMS (τ = 3 px) P2 pipeline'ına eklenir; aksi halde eklenmez.
- **İkincil (Holm, kararı değiştirmez):** DetA, AssA, IDSW, <8 px recall (p2+cNMS − p2).
- **Bilgi:** ctrl+cNMS − ctrl; D1 protokolünde (IoU, K = 0) p2+cNMS − p2; yinelenen oran öncesi/sonrası; boyuta göre recall; GT'de 3 px'ten yakın gerçek çift payı.

**Sınırlar:** Motivasyon D2 sonuçlarından geldi (sonradan fark edilen bir sorun); bu yüzden karar kuralı burada, sonuçlardan önce sabitleniyor. Tek τ değeri; tek seed; val 16 sekans.

**Süre:** NMS < 1 dk; 4 tracker koşusu (ctrl ve p2 × iki tracker ayarı) ~10–15 dk; değerlendirme ~3–5 dk.

**Ön koşul:** Bu hücreler D2 notebook'unun açık oturumunda, 4.4'ten sonra çalışır (`DET_DIRS`, `TRK_FROZEN`, `PS_FZ`, `recall_by_bin` vb. gerekli). Oturum kapandıysa önce 1.1–3.0, 3.1 (veya `P2_RUN`/`P2_LAST` hücresi), 4.1 ve 4.2'yi çalıştırın. 4.1 kayıtlı tespitleri yeniden üretmez, yalnızca tracker koşularını tekrarlar (~15 dk).

In [ ]:
# D2b.1) Ön kayıt: karar kuralı sonuçlardan ÖNCE kilitlenir
CNMS_TAU = 3.0
D2B_ARMS = ('ctrl', 'p2')
D2B_PREREG = D2_DIR / 'D2b_prereg.json'
d2b_prereg = {
    'deney': 'D2b: merkez uzaklığına dayalı NMS (cNMS), kaydedilmiş tespitler üzerinde',
    'yontem': 'Her karede skora göre (kararlı sıralama) azalan; tutulan kutuya merkez uzaklığı < tau olan düşük skorlu kutular silinir. '
              'Kalan kutuların kare içi sırası ve değerleri değişmez.',
    'tau_px': CNMS_TAU, 'tau_gerekce': 'Yinelenen kutu ölçümündeki eşik; ızgara taranmaz.',
    'kollar': list(D2B_ARMS), 'tespit_kaynagi': {a: str(DET_DIRS[a]) for a in D2B_ARMS},
    'tracker_birincil': dict(FROZEN_ARGS), 'tracker_bilgi': {'patches': '', 'cmc': 'sparseOptFlow', 'match_thresh': 0.7, 'coast_k': 0},
    'birincil': 'HOTA farkı (p2+cNMS - p2), dondurulmuş tracker, eşleştirilmiş sekans bootstrap B=10000, %95 CI.',
    'koruma': 'MOTA_seq farkı (p2+cNMS - p2) CI üst sınırı >= 0.',
    'karar': 'HOTA CI alt sınırı > 0 VE koruma sağlanırsa cNMS (tau=3 px) P2 pipeline ına eklenir; aksi halde eklenmez.',
    'ikincil_holm': ['DetA', 'AssA', 'IDSW', 'recall_<8px'],
    'bilgi': ['ctrl+cNMS - ctrl (HOTA, dondurulmuş)', 'D1 protokolü p2+cNMS - p2 (HOTA, DetA)', 'yinelenen oran öncesi/sonrası',
              'boyuta göre recall', 'GT de merkezleri tau dan yakın gerçek çift payı'],
    'not': 'Motivasyon D2 sonuçlarından (yinelenen oran) geldi; kural D2b sonuçlarından önce sabitlendi.',
}
_d2b_sha = hashlib.sha1(json.dumps(d2b_prereg, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
if D2B_PREREG.exists():
    if json.load(open(D2B_PREREG)).get('kural_sha1') != _d2b_sha:
        raise RuntimeError(f'D2b ön kaydı değişti; kural sonradan değiştirilemez: {D2B_PREREG}')
    print('D2b ön kaydı mevcut ve değişmedi.')
else:
    with open(D2B_PREREG, 'w') as fh:
        json.dump({**d2b_prereg, 'kural_sha1': _d2b_sha, 'olusturma': time.strftime('%Y-%m-%d %H:%M:%S')}, fh, indent=1, ensure_ascii=False)
    print('D2b ön kaydı yazıldı:', D2B_PREREG)
D2B_PREREG_SHA256 = hashlib.sha256(D2B_PREREG.read_bytes()).hexdigest()
print('sha256:', D2B_PREREG_SHA256)

In [ ]:
# D2b.2) cNMS'yi uygula (ctrl ve p2) ve tracker koşuları
def center_nms(d, tau=CNMS_TAU):
    """Kare başına: skor sırasıyla (kararlı) tutulan kutuya merkezi tau'dan yakın düşük skorlu kutuları siler.
    Kalan satırların sırası ve değerleri değişmez."""
    if len(d) == 0:
        return d
    keep_all = np.zeros(len(d), bool)
    for f in np.unique(d[:, 0]):
        idx = np.where(d[:, 0] == f)[0]
        b = d[idx]
        order = np.argsort(-b[:, 5], kind='stable')
        c = np.c_[(b[order, 1] + b[order, 3]) / 2, (b[order, 2] + b[order, 4]) / 2]
        keep = np.ones(len(order), bool)
        for i in range(len(order)):
            if not keep[i]:
                continue
            dist = np.hypot(c[i + 1:, 0] - c[i, 0], c[i + 1:, 1] - c[i, 1])
            keep[i + 1:] &= ~(dist < tau)
        keep_all[idx[order[keep]]] = True
    return d[keep_all]


def duplicate_rate(dets_dir, dist_px=CNMS_TAU, score_min=0.09):
    n_det = n_dup = 0
    for s in VAL_SEQS:
        d = load_dets(Path(dets_dir) / f'{s}.txt')
        d = d[d[:, 5] >= score_min] if len(d) else d
        n_det += len(d)
        for f in (np.unique(d[:, 0]) if len(d) else []):
            b = d[d[:, 0] == f]
            if len(b) < 2:
                continue
            cc = np.c_[(b[:, 1] + b[:, 3]) / 2, (b[:, 2] + b[:, 4]) / 2]
            D = np.linalg.norm(cc[:, None] - cc[None], axis=-1)
            n_dup += int(np.triu(D < dist_px, k=1).sum())
    return {'tespit': n_det, 'yinelenen_cift': n_dup, 'binde': round(1000 * n_dup / max(n_det, 1), 2)}


# Kendi kendini test: sentetik kare
_t = np.array([[1, 10, 10, 14, 14, 0.9], [1, 11, 10, 15, 14, 0.5], [1, 30, 30, 34, 34, 0.4], [2, 10, 10, 14, 14, 0.3]])
_r = center_nms(_t, 3.0)
assert _r.shape[0] == 3 and not any((_r == _t[1]).all(1)), 'center_nms testi başarısız'

DET_DIRS_CNMS, N_REMOVED = {}, {}
for a in D2B_ARMS:
    dd = D2_DIR / f'{a}_cnms3' / 'dets'
    dd.mkdir(parents=True, exist_ok=True)
    n0 = n1 = 0
    for s in VAL_SEQS:
        d = load_dets(DET_DIRS[a] / f'{s}.txt')
        k = center_nms(d)
        n0 += len(d); n1 += len(k)
        np.savetxt(dd / f'{s}.txt', k, delimiter=',', fmt='%.4f')
    DET_DIRS_CNMS[a] = dd
    N_REMOVED[a] = {'once': n0, 'sonra': n1, 'silinen': n0 - n1, 'silinen_pay': round((n0 - n1) / max(n0, 1), 4)}
    print(f'{a}: {n0} -> {n1} tespit (silinen {n0 - n1}, %{100 * (n0 - n1) / max(n0, 1):.1f})')

TRK_B_FZ, TRK_B_IOU = {}, {}
for a in D2B_ARMS:
    TRK_B_FZ[a] = str(D2_DIR / f'{a}_cnms3' / 'track_frozen')
    run_tracker_patched(DET_DIRS_CNMS[a], TRK_B_FZ[a], **FROZEN_ARGS)                       # birincil
    TRK_B_IOU[a] = str(D2_DIR / f'{a}_cnms3' / 'track_iou')
    run_tracker_patched(DET_DIRS_CNMS[a], TRK_B_IOU[a], patches='', match_thresh=MT['iou'])  # bilgi
print('D2b tracker koşuları tamam.')

In [ ]:
# D2b.3) Değerlendirme, karar, görsel, kayıt
import matplotlib.pyplot as plt
_cols = ['MOTA', 'MOTA_seq', 'HOTA', 'DetA', 'AssA', 'IDF1', 'IDSW', 'Frag', 'FP', 'FN']
tab_b_fz = summary_table({**{a: TRK_FROZEN[a] for a in D2B_ARMS}, **{f'{a}_cnms': TRK_B_FZ[a] for a in D2B_ARMS}},
                         'D2b_frozen')[_cols].astype(float)
tab_b_iou = summary_table({**{a: TRK_IOU[a] for a in D2B_ARMS}, **{f'{a}_cnms': TRK_B_IOU[a] for a in D2B_ARMS}},
                          'D2b_iou')[_cols].astype(float)
print('Dondurulmuş tracker (birincil):'); print(tab_b_fz.round(4))
print('\nD1 protokolü (bilgi):'); print(tab_b_iou.round(4))

PSB_FZ = {a: per_seq_table(TRK_B_FZ[a], f'D2b_frozen_{a}_cnms') for a in D2B_ARMS}
PSB_IOU = {a: per_seq_table(TRK_B_IOU[a], f'D2b_iou_{a}_cnms') for a in D2B_ARMS}

# --- Birincil ve koruma ---
B_PRIMARY = compare(PS_FZ['p2'], PSB_FZ['p2'], 'HOTA')
B_GUARD = compare(PS_FZ['p2'], PSB_FZ['p2'], 'MOTA')      # sekans başına MOTA -> MOTA_seq farkı
B_ACCEPT = (B_PRIMARY['CI95'][0] > 0) and (B_GUARD['CI95'][1] >= 0)
B_DECISION = 'cNMS (tau=3 px) P2 pipeline\'ına eklenir' if B_ACCEPT else 'cNMS eklenmez'
print('\nBİRİNCİL HOTA (p2+cNMS - p2, dondurulmuş):', B_PRIMARY)
print('KORUMA MOTA_seq:', B_GUARD)
print('D2b KARAR:', B_DECISION)

# --- İkincil (Holm) ---
RB_B = {a: recall_by_bin(DET_DIRS_CNMS[a]) for a in D2B_ARMS}
_s0, _s1 = small_per_seq(RB['p2']), small_per_seq(RB_B['p2'])
assert list(_s0.index) == list(_s1.index)
_m, _ci, _p = bootstrap_diff(_s0.values, _s1.values)
B_SECONDARY = [compare(PS_FZ['p2'], PSB_FZ['p2'], m) for m in ('DetA', 'AssA', 'IDSW')]
B_SECONDARY.append({'metrik': 'recall_<8px', 'fark': round(_m, 4), 'CI95': (round(_ci[0], 4), round(_ci[1], 4)), 'p': round(_p, 4)})
B_SEC_DF = pd.DataFrame(B_SECONDARY)
B_SEC_DF['p_holm'] = holm(B_SEC_DF.p.values).round(4)
print('\nİKİNCİL (p2+cNMS - p2, Holm):'); print(B_SEC_DF.to_string(index=False))

# --- Bilgi ---
B_INFO = {'ctrl+cNMS - ctrl HOTA (dondurulmuş)': compare(PS_FZ['ctrl'], PSB_FZ['ctrl'], 'HOTA'),
          'D1 protokolü p2+cNMS - p2 HOTA': compare(PS['p2'], PSB_IOU['p2'], 'HOTA'),
          'D1 protokolü p2+cNMS - p2 DetA': compare(PS['p2'], PSB_IOU['p2'], 'DetA')}
for k, v in B_INFO.items():
    print(f'  {k}: {v}')
B_DUP = {f'{a}': duplicate_rate(DET_DIRS[a]) for a in D2B_ARMS}
B_DUP.update({f'{a}_cnms': duplicate_rate(DET_DIRS_CNMS[a]) for a in D2B_ARMS})
print('\nYinelenen oran:'); print(pd.DataFrame(B_DUP).T)
B_REC = pd.DataFrame({**{a: REC_POOLED[a] for a in D2B_ARMS},
                      **{f'{a}_cnms': (RB_B[a].groupby('bin').matched.sum() / RB_B[a].groupby('bin').n.sum()).loc[REC_POOLED.index]
                         for a in D2B_ARMS}})
print('\nBoyuta göre recall:'); print(B_REC.round(4))

# GT'de merkezleri tau'dan yakın gerçek çift payı (yanlışlıkla silinebilecek gerçek hedeflerin üst sınırı)
_close = _tot = 0
for s in VAL_SEQS:
    g = load_mot(DV1 / 'gt' / f'{s}.txt'); g = g[g[:, 6] == 1]
    for f in np.unique(g[:, 0]):
        nn = nearest_other_gt(g[g[:, 0] == f])
        _close += int((nn < CNMS_TAU).sum()); _tot += len(nn)
B_GT_CLOSE = {'gt_kutu': _tot, 'yakin_gt': _close, 'pay': round(_close / max(_tot, 1), 5)}
print('\nGT yakın çift payı (< 3 px):', B_GT_CLOSE)

# --- Görsel: yinelenen oran ve recall, öncesi/sonrası ---
fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
_lab = ['ctrl', 'ctrl_cnms', 'p2', 'p2_cnms']
axs[0].bar(_lab, [B_DUP[k]['binde'] for k in _lab], color=['#7f7f7f', '#bfbfbf', '#1f77b4', '#aec7e8'])
axs[0].set_ylabel('yinelenen çift (‰)'); axs[0].set_title('Yinelenen tespit oranı (merkez < 3 px)')
_x = np.arange(len(B_REC.index)); _w = 0.2
for i, k in enumerate(_lab):
    axs[1].bar(_x + (i - 1.5) * _w, B_REC[k].values, _w, label=k)
axs[1].set_xticks(_x, [f'{b} px' for b in B_REC.index]); axs[1].set_ylim(0, 1); axs[1].legend(fontsize=8)
axs[1].set_title('Boyuta göre recall (IoU ≥ 0,5, skor ≥ 0,09)')
fig.tight_layout(); D2B_FIG = D2_DIR / 'D2b_cnms.png'; fig.savefig(D2B_FIG, dpi=130); plt.show()

# --- Kayıt ---
D2B_RESULTS = {'karar': B_DECISION, 'kabul': bool(B_ACCEPT), 'birincil_HOTA': B_PRIMARY, 'koruma_MOTA_seq': B_GUARD,
               'ikincil_holm': B_SEC_DF.to_dict(orient='records'), 'bilgi': B_INFO, 'silinen_tespit': N_REMOVED,
               'yinelenen_oran': B_DUP, 'recall_boyut': B_REC.round(4).to_dict(), 'gt_yakin_cift': B_GT_CLOSE,
               'tablo_dondurulmus': tab_b_fz.round(4).to_dict(orient='index'),
               'tablo_D1_protokolu': tab_b_iou.round(4).to_dict(orient='index'),
               'on_kayit_sha256': D2B_PREREG_SHA256, 'tau_px': CNMS_TAU, 'tarih': time.strftime('%Y-%m-%d %H:%M:%S')}
with open(D2_DIR / 'D2b_results.json', 'w') as fh:
    json.dump(D2B_RESULTS, fh, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
print('\nKaydedildi:', D2_DIR / 'D2b_results.json')

if MLFLOW_OK:
    try:
        mlflow.set_experiment('TIR-JDE-D2')
        with mlflow.start_run(run_name='D2b_cnms'):
            mlflow.log_params({'tau_px': CNMS_TAU, 'karar': B_DECISION, 'tracker': json.dumps(FROZEN_ARGS)})
            _met = {'HOTA_diff': B_PRIMARY['fark'], 'HOTA_ci_lo': B_PRIMARY['CI95'][0], 'HOTA_ci_hi': B_PRIMARY['CI95'][1],
                    'MOTAseq_diff': B_GUARD['fark'], 'MOTAseq_ci_hi': B_GUARD['CI95'][1], 'kabul': float(B_ACCEPT)}
            for k in _lab:
                _met[f'{k}_dup_permille'] = B_DUP[k]['binde']
                _met[f'{k}_HOTA_frozen'] = float(tab_b_fz.loc[k, 'HOTA'])
                _met[f'{k}_MOTAseq_frozen'] = float(tab_b_fz.loc[k, 'MOTA_seq'])
            mlflow.log_metrics(_met)
            for f in (D2B_FIG, D2_DIR / 'D2b_results.json', D2B_PREREG):
                mlflow.log_artifact(str(f))
        shutil.copy(LOCAL / 'mlflow_d2.db', D2_DIR / 'mlflow_d2.db')
        print('MLflow: D2b_cnms kaydedildi ve veritabanı Drive\'a kopyalandı.')
    except Exception as e:
        print('MLflow kaydı başarısız (sonuçlar dosyada):', repr(e))

In [ ]:
# D2b.3b) Görsel kayıt: p2 ve p2+cNMS yan yana video (dondurulmuş tracker) + yakınlaştırılmış örnekler
import subprocess
VID_SCALE, VID_LEN, VID_FPS = 2, 300, 10

def removed_mask(d, k):
    """d içindeki satırlardan cNMS'nin sildiklerini işaretler (k = center_nms(d))."""
    kept = set(map(tuple, np.round(k, 4)))
    return np.array([tuple(r) not in kept for r in np.round(d, 4)], bool) if len(d) else np.zeros(0, bool)

# Sekans ve pencere seçimi SONUÇLARA göre değil mekanizmaya göre: p2'de cNMS'nin en çok kutu sildiği 300 karelik pencere
_rm = {}
for s in VAL_SEQS:
    d = load_dets(DET_DIRS['p2'] / f'{s}.txt')
    d = d[d[:, 5] >= 0.09] if len(d) else d
    _rm[s] = d[removed_mask(d, center_nms(d))]
_best = (None, 1, -1)
for s in VAL_SEQS:
    n_fr = len(glob.glob(f'{FRAMES_ROOT}/{s}/*.jpg'))
    cnt = np.bincount(_rm[s][:, 0].astype(int), minlength=n_fr + 2)[1:n_fr + 1] if len(_rm[s]) else np.zeros(n_fr)
    win = np.convolve(cnt, np.ones(min(VID_LEN, n_fr)), 'valid')
    if win.max() > _best[2]:
        _best = (s, int(win.argmax()) + 1, int(win.max()))
D2B_VSEQ, D2B_F0, _nrm = _best
D2B_F1 = D2B_F0 + VID_LEN - 1
print(f'Video: {D2B_VSEQ}, kare {D2B_F0}-{D2B_F1} (bu pencerede cNMS {_nrm} kutu siliyor)')

_vgt = load_mot(DV1 / 'gt' / f'{D2B_VSEQ}.txt'); _vgt = _vgt[_vgt[:, 6] == 1]
_vtr = {'p2': load_mot(Path(TRK_FROZEN['p2']) / f'{D2B_VSEQ}.txt'),
        'p2_cnms': load_mot(Path(TRK_B_FZ['p2']) / f'{D2B_VSEQ}.txt')}
_vfiles = sorted(glob.glob(f'{FRAMES_ROOT}/{D2B_VSEQ}/*.jpg'))
_S = VID_SCALE

def _draw(img, f, tracks, removed=None, label=''):
    img = cv2.resize(img, None, fx=_S, fy=_S, interpolation=cv2.INTER_NEAREST)
    for r in tracks[tracks[:, 0] == f]:                                        # iz: kırmızı + kimlik
        x, y, w, h = r[2:6] * _S
        cv2.rectangle(img, (int(x), int(y)), (int(x + w), int(y + h)), (0, 0, 255), 1)
        cv2.putText(img, str(int(r[1])), (int(x), max(int(y) - 3, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
    if removed is not None:                                                    # cNMS'nin sileceği tespitler: sarı
        for r in removed[removed[:, 0] == f]:
            x1, y1, x2, y2 = r[1:5] * _S
            cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 255), 1)
    for r in _vgt[_vgt[:, 0] == f]:                                            # GT: yeşil, en son (görünsün)
        x, y, w, h = r[2:6] * _S
        cv2.rectangle(img, (int(x) - 1, int(y) - 1), (int(x + w) + 1, int(y + h) + 1), (0, 255, 0), 1)
    n_tr = int((tracks[:, 0] == f).sum()); n_gt = int((_vgt[:, 0] == f).sum())
    cv2.putText(img, f'{label} | kare {f} | GT {n_gt} | iz {n_tr}', (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    return img

_raw_vid = LOCAL / f'D2b_{D2B_VSEQ}_p2_vs_cnms_mp4v.mp4'
_vw = None
for f in range(D2B_F0, min(D2B_F1, len(_vfiles)) + 1):
    im = cv2.imread(_vfiles[f - 1])
    fr = np.hstack([_draw(im.copy(), f, _vtr['p2'], _rm[D2B_VSEQ], 'p2 (sari: cNMS silecek)'),
                    _draw(im.copy(), f, _vtr['p2_cnms'], None, 'p2 + cNMS 3 px')])
    if _vw is None:
        _vw = cv2.VideoWriter(str(_raw_vid), cv2.VideoWriter_fourcc(*'mp4v'), VID_FPS, (fr.shape[1], fr.shape[0]))
    _vw.write(fr)
_vw.release()

# Tarayıcıda/Confluence'ta oynasın diye H.264'e çevir (ffmpeg yoksa mp4v kalır)
D2B_VIDEO = D2_DIR / f'D2b_{D2B_VSEQ}_p2_vs_cnms.mp4'
try:
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(_raw_vid), '-c:v', 'libx264', '-preset', 'veryfast',
                    '-crf', '23', '-pix_fmt', 'yuv420p', str(LOCAL / D2B_VIDEO.name)], check=True)
    if (LOCAL / D2B_VIDEO.name).resolve() != D2B_VIDEO.resolve():
        shutil.copy(LOCAL / D2B_VIDEO.name, D2B_VIDEO)
    print('Video (H.264):', D2B_VIDEO)
except Exception as e:
    shutil.copy(_raw_vid, D2B_VIDEO)
    print('ffmpeg dönüşümü yapılamadı, mp4v kaydedildi:', D2B_VIDEO, repr(e))

# Yakınlaştırılmış örnekler: cNMS'nin sildiği kutulardan rastgele 6 tanesi (tohum 0), öncesi / sonrası
CROP_B, ZOOM_B = 40, 6
_rng_b = np.random.default_rng(0)
_pool = [(s, r) for s in VAL_SEQS for r in _rm[s]]
_pick = [_pool[i] for i in _rng_b.permutation(len(_pool))[:6]]

def _crop(seq, f, cx, cy, dets, removed):
    img = cv2.imread(sorted(glob.glob(f'{FRAMES_ROOT}/{seq}/*.jpg'))[f - 1])
    H, W = img.shape[:2]
    x0 = int(np.clip(cx - CROP_B / 2, 0, W - CROP_B)); y0 = int(np.clip(cy - CROP_B / 2, 0, H - CROP_B))
    p = cv2.resize(img[y0:y0 + CROP_B, x0:x0 + CROP_B], None, fx=ZOOM_B, fy=ZOOM_B, interpolation=cv2.INTER_NEAREST)
    def _rect(b, col):
        cv2.rectangle(p, (int((b[0] - x0) * ZOOM_B), int((b[1] - y0) * ZOOM_B)), (int((b[2] - x0) * ZOOM_B), int((b[3] - y0) * ZOOM_B)), col, 1)
    for r, col in [(r, (0, 0, 255)) for r in dets] + [(r, (0, 255, 255)) for r in removed]:
        if r[0] == f and r[3] >= x0 and r[1] <= x0 + CROP_B and r[4] >= y0 and r[2] <= y0 + CROP_B:
            _rect(r[1:5], col)
            cv2.putText(p, f'{r[5]:.2f}', (max(int((r[1] - x0) * ZOOM_B), 0), max(int((r[2] - y0) * ZOOM_B) - 3, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, col, 1)
    g = load_mot(DV1 / 'gt' / f'{seq}.txt'); g = g[(g[:, 6] == 1) & (g[:, 0] == f)]
    for r in g:                                                    # GT en son: tespitle çakışsa da görünsün
        _rect(xywh2xyxy(r[None, 2:6])[0], (0, 255, 0))
    return p[..., ::-1]

D2B_EX_FIG = None
if _pick:
    fig, axs = plt.subplots(len(_pick), 2, figsize=(6, 3 * len(_pick)), squeeze=False)
    for i, (s, r) in enumerate(_pick):
        f = int(r[0]); cx, cy = (r[1] + r[3]) / 2, (r[2] + r[4]) / 2
        d0 = load_dets(DET_DIRS['p2'] / f'{s}.txt'); d0 = d0[(d0[:, 0] == f) & (d0[:, 5] >= 0.09)]
        d1 = load_dets(DET_DIRS_CNMS['p2'] / f'{s}.txt'); d1 = d1[(d1[:, 0] == f) & (d1[:, 5] >= 0.09)]
        rmf = _rm[s][_rm[s][:, 0] == f]
        kept0 = d0[~removed_mask(d0, d1)] if len(d0) else d0
        axs[i][0].imshow(_crop(s, f, cx, cy, kept0, rmf)); axs[i][1].imshow(_crop(s, f, cx, cy, d1, np.zeros((0, 6))))
        axs[i][0].set_ylabel(f'{s} kare {f}', fontsize=8)
        for ax in axs[i]:
            ax.set_xticks([]); ax.set_yticks([])
    axs[0][0].set_title('p2 (sarı: silinen)'); axs[0][1].set_title('p2 + cNMS 3 px')
    fig.suptitle('Yeşil GT, kırmızı tespit, sarı cNMS ile silinen (skor ≥ 0,09); 40×40 px, 6× büyütme', fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, 0.98)); D2B_EX_FIG = D2_DIR / 'D2b_cnms_examples.png'; fig.savefig(D2B_EX_FIG, dpi=120); plt.show()

# MLflow: D2b_cnms çalışmasına ekle
if MLFLOW_OK:
    try:
        _r = mlflow.search_runs(experiment_names=['TIR-JDE-D2'], filter_string="tags.mlflow.runName = 'D2b_cnms'",
                                order_by=['start_time DESC'], max_results=1)
        _cl = mlflow.tracking.MlflowClient()
        for f in (D2B_VIDEO, D2B_EX_FIG):
            if f is not None:
                _cl.log_artifact(_r.run_id.iloc[0], str(f))
        shutil.copy(LOCAL / 'mlflow_d2.db', D2_DIR / 'mlflow_d2.db')
        print('MLflow: video ve örnek görseli D2b_cnms çalışmasına eklendi.')
    except Exception as e:
        print('MLflow eki başarısız (dosyalar Drive\'da):', repr(e))
print('Dosyalar:', D2B_VIDEO, D2B_EX_FIG)

In [ ]:
# D2b.4) Confluence raporu (sonuçlardan otomatik)
def _f(x, n=4):
    return f'{float(x):.{n}f}'.replace('.', ',')

def _ci(c):
    return f'{_f(c["fark"])}, CI95 ({_f(c["CI95"][0])}; {_f(c["CI95"][1])}), p {_f(c["p"])}'

_rows = '\n'.join(f'| {k} | {_f(tab_b_fz.loc[k, "MOTA_seq"])} | {_f(tab_b_fz.loc[k, "HOTA"])} | {_f(tab_b_fz.loc[k, "DetA"])} | '
                  f'{_f(tab_b_fz.loc[k, "AssA"])} | {int(tab_b_fz.loc[k, "IDSW"])} | {int(tab_b_fz.loc[k, "FP"])} | {int(tab_b_fz.loc[k, "FN"])} |'
                  for k in ['ctrl', 'ctrl_cnms', 'p2', 'p2_cnms'])
_rows_sec = '\n'.join(f'| {r.metrik} | {_f(r.fark)} | ({_f(r.CI95[0])}; {_f(r.CI95[1])}) | {_f(r.p_holm)} |' for r in B_SEC_DF.itertuples())
_rows_dup = '\n'.join(f'| {k} | {B_DUP[k]["tespit"]} | {B_DUP[k]["yinelenen_cift"]} | {_f(B_DUP[k]["binde"], 1)} |' for k in B_DUP)
_rows_rec = '\n'.join(f'| {b} px | ' + ' | '.join(_f(B_REC.loc[b, k], 3) for k in ['ctrl', 'ctrl_cnms', 'p2', 'p2_cnms']) + ' |'
                      for b in B_REC.index)

D2B_REPORT = f"""## D2b: Merkez uzaklığına dayalı NMS, sonuç raporu

**Durum:** Geliştirme seti (val, 16 sekans), tek seed. **Karar: {B_DECISION}.**

**Ne:** D2'de p2'nin yinelenen tespit oranı üç kattı. Kaydedilmiş tespitlere ek bir NMS uygulandı: her karede, tutulan kutuya merkezi {_f(CNMS_TAU, 0)} px'ten yakın olan daha düşük skorlu kutular silindi. Eğitim ve yeniden tespit yok. Kural çalıştırmadan önce kilitlendi (D2b_prereg.json, sha256 {D2B_PREREG_SHA256[:16]}…).

**Silinen tespit:** ctrl %{100 * N_REMOVED['ctrl']['silinen_pay']:.1f} ({N_REMOVED['ctrl']['silinen']}), p2 %{100 * N_REMOVED['p2']['silinen_pay']:.1f} ({N_REMOVED['p2']['silinen']}). GT'de merkezleri {_f(CNMS_TAU, 0)} px'ten yakın gerçek çift payı: %{100 * B_GT_CLOSE['pay']:.2f} (yanlışlıkla silinebilecek gerçek hedeflerin üst sınırı).

### Sonuç (dondurulmuş tracker: {FROZEN_ARGS['cmc']} + {FROZEN_ARGS['patches']} + K{FROZEN_ARGS['coast_k']})

| Kol | MOTA_seq | HOTA | DetA | AssA | IDSW | FP | FN |
| --- | --- | --- | --- | --- | --- | --- | --- |
{_rows}

**Birincil:** HOTA (p2+cNMS − p2) = {_ci(B_PRIMARY)}.
**Koruma:** MOTA_seq (p2+cNMS − p2) = {_ci(B_GUARD)}.
**Karar:** {B_DECISION}.

**İkincil (Holm, kararı değiştirmez):**

| Metrik | Fark | CI95 | p (Holm) |
| --- | --- | --- | --- |
{_rows_sec}

### Bilgi

- ctrl+cNMS − ctrl, HOTA (dondurulmuş): {_ci(B_INFO['ctrl+cNMS - ctrl HOTA (dondurulmuş)'])}
- D1 protokolü (IoU, K = 0), p2+cNMS − p2: HOTA {_ci(B_INFO['D1 protokolü p2+cNMS - p2 HOTA'])}; DetA {_ci(B_INFO['D1 protokolü p2+cNMS - p2 DetA'])}

| Kol | Tespit | Yinelenen çift | Binde |
| --- | --- | --- | --- |
{_rows_dup}

| Boyut | ctrl | ctrl+cNMS | p2 | p2+cNMS |
| --- | --- | --- | --- | --- |
{_rows_rec}

**Sınırlar:** Motivasyon D2 sonuçlarından geldi; kural D2b sonuçlarından önce sabitlendi. Tek τ değeri (ızgara yok), tek seed, val 16 sekans.

**Dosyalar:** WORK/D2/D2b_results.json, D2b_prereg.json, D2b_cnms.png; tespitler {{ctrl,p2}}_cnms3/dets. MLflow: TIR-JDE-D2 / D2b_cnms.
"""
(D2_DIR / 'D2b_confluence_raporu.md').write_text(D2B_REPORT, encoding='utf-8')
print(D2B_REPORT)

In [ ]:
# E5a.0) Ön teşhis: p2 + dondurulmuş tracker'da IDSW nereden geliyor, uzun boşluk sonrası yeniden bağlanma ne durumda?
_rows, _ev = [], []
for s in VAL_SEQS:
    g = load_mot(DV1 / 'gt' / f'{s}.txt'); g = g[g[:, 6] == 1]
    p = load_mot(Path(TRK_FROZEN['p2']) / f'{s}.txt')
    ev = idsw_groups(g, p)                                   # yakın çift / uzun boşluk / diğer
    ev['seq'] = s; _ev.append(ev)
    A = assignments(g, p)
    gaps = []
    for gid, gg in A.sort_values('f').groupby('gid'):
        fs, ps = gg.f.values, gg.pid.values
        for k in range(1, len(fs)):
            gl = int(fs[k] - fs[k - 1] - 1)
            if gl > 10:
                gaps.append((gl, int(ps[k] == ps[k - 1])))
    for gl, ok in gaps:
        _rows.append({'seq': s, 'gap': gl, 'ayni_kimlik': ok})
E5_EV = pd.concat(_ev, ignore_index=True)
E5_GAPS = pd.DataFrame(_rows, columns=['seq', 'gap', 'ayni_kimlik'])

print('Toplam IDSW (yaklaşık, kendi eşleştirmemiz):', len(E5_EV), '| TrackEval IDSW:', int(tab_fz.loc['p2', 'IDSW']))
print(E5_EV.grup.value_counts().rename('IDSW').to_frame().assign(pay=lambda d: (d.IDSW / d.IDSW.sum()).round(3)))

E5_GAPS['aralik'] = pd.cut(E5_GAPS.gap, [10, 30, 60, 10**6], labels=['11-30', '31-60', '>60'])
print('\nUzun boşluklar (>10 kare) ve aynı kimlikle dönüş oranı:')
print(E5_GAPS.groupby('aralik', observed=False).ayni_kimlik.agg(['count', 'mean']).rename(
    columns={'count': 'bosluk', 'mean': 'ayni_kimlik_orani'}).round(3))
print('Genel:', len(E5_GAPS), 'boşluk, aynı kimlik oranı', round(E5_GAPS.ayni_kimlik.mean(), 3) if len(E5_GAPS) else 'yok')
print('\nNot: tracker kayıp izi 60 kare tutar (track_buffer); >60 karelik boşlukta yeniden bağlanma mümkün değil.')

## E3: Kimlik başlığı sondajı (head-only)

**Soru:** Termal görüntüde 4–8 piksellik İHA'ların görünüşü (appearance), onları birbirinden ayırmaya yetiyor mu? Yetiyorsa hangi loss (CE ya da CE + triplet) ve hangi embedding boyutu (64 ya da 128)?

**Neden:** E1b ve E5a.0'a göre IDSW'nin %84'ü yakın çift ve kısa süreli karışmalardan geliyor; bunlar konumla çözülemiyor. Tezin ana iddiası (r ile görünüş/konum ağırlığı, E7) görünüşün bilgi taşımasına dayanıyor. E3 bu riski ucuzca ölçer. Bu bir sondajdır: nihai embedding sonra ortak eğitilen başlıktan (7a / B2) gelir; E3 yalnızca yönü seçer.

**Ne yapacağız:**
1. **detector-v1-P2 dondurulur.** Her GT kutusu için neck çıktılarından feature alınır: P2 (katman 23, stride 4) ve P3 (katman 14, stride 8), RoIAlign 2×2, birleştirilmiş vektör.
2. Bu feature'ların üstüne yalnızca **küçük bir kimlik başlığı** (MLP: D → 512 → E) eğitilir. Dört kol: CE-64, CE-128, CE+triplet-64, CE+triplet-128. Aynı örnekleyici (PK: 32 kimlik × 4 örnek), aynı iterasyon (3000), seed 0. Val'e bakarak erken durdurma yok.
3. **Eğitim:** 84 train sekansı (val ve hariç tutulan T_13, BB2P_01 yok), her 5. kare, kimlik başına en çok 300 kutu. Kimlikler `TrainLabels` (Train.zip) dosyalarından.
4. **Değerlendirme (val, 16 sekans):** Sorgu = t karesinde, en yakın başka GT'ye ≤ 12,7 px olan (yakın çift) GT kutusu. Galeri = t − d karesindeki GT kutuları (en az 2). Doğru = en benzer galeri kutusu aynı kimlik. Birincil d = 10, ikincil d = 1. Ayrıca tüm kutular (yakın çift şartı olmadan).

**Referanslar:** Konum referansı (galeride merkezi en yakın kutu, hareket modeli yok) ve şans düzeyi (1 / galeri boyutu). "Görünüş %X doğru" ancak konuma göre anlamlıdır.

**Önceden yazılmış karar kuralı** (`E3_DIR/E3_prereg.json`, çalıştırmadan önce kilitlenir; metrik: d = 10 yakın çift top-1 doğruluk, sekans başına, eşleştirilmiş bootstrap B = 10000):
- **Loss:** 128 boyutta (CE+triplet − CE) CI alt sınırı > 0 ise CE+triplet, değilse CE (daha basit olan).
- **Boyut:** Seçilen loss ile (128 − 64) CI alt sınırı > 0 ise 128, değilse 64.
- **Görünüş ve konum:** Seçilen kol − konum referansı. CI alt sınırı > 0 → "görünüş konumdan fazla bilgi taşıyor"; CI üst sınırı < 0 → "görünüş konumdan zayıf"; aksi halde "fark yok". Bu, E7'nin beklentisini belirler; E7'yi iptal etmez.
- Üç p-değeri Holm ile raporlanır.

**Sınırlar:** Head-only (detector feature'ları kimlik için eğitilmedi; ortak eğitim B2'de). Galeri tek kareden; gerçek tracker'da iz geçmişi var. Tek seed. Val 16 sekans.

**Süre:** Feature çıkarma ~10–15 dk (train ~15–20 bin kare, val 11.868 kare); 4 başlık eğitimi ~1–2 dk/kol; değerlendirme ~2 dk.

**Ön koşul:** D2 notebook'unun açık oturumunda: 1.1–1.5, 3.0 ve `P2_RUN`/`P2_LAST` hücresi. 4.x hücreleri gerekmez.

In [ ]:
# E3.1) Ön kayıt: karar kuralı sonuçlardan ÖNCE kilitlenir
E3_DIR = WORK / 'E3'
E3_LOCAL = LOCAL / 'E3'
for _p in (E3_DIR, E3_LOCAL):
    _p.mkdir(parents=True, exist_ok=True)
E3_ARMS = {'ce64': ('ce', 64), 'ce128': ('ce', 128), 'cetri64': ('ce+tri', 64), 'cetri128': ('ce+tri', 128)}
E3_CFG = dict(feat_layers={'P2': 23, 'P3': 14}, roi=2, train_step=5, cap_per_id=300, min_side=2.0,
              hidden=512, P=32, K=4, iters=3000, lr=1e-3, wd=5e-4, ce_scale=16.0, tri_margin=0.3, tri_weight=1.0,
              seed=0, close_px=12.7, gaps=[1, 10], primary_gap=10, imgsz=IMGSZ,
              exclude=['T_13', 'BB2P_01'], detector=str(P2_LAST))
E3_PREREG = E3_DIR / 'E3_prereg.json'
e3_prereg = {
    'deney': 'E3: kimlik başlığı sondajı (head-only), detector-v1-P2 donuk',
    'kollar': {k: {'loss': v[0], 'boyut': v[1]} for k, v in E3_ARMS.items()},
    'ayarlar': E3_CFG,
    'metrik': 'd=10 yakın çift (en yakın başka GT <= 12.7 px) top-1 kimlik doğruluğu; galeri t-d karesindeki GT kutuları (>=2); '
              'sekans başına doğruluk, eşleştirilmiş bootstrap B=10000.',
    'karar_loss': '128 boyutta (CE+triplet - CE) CI alt > 0 ise CE+triplet, değilse CE.',
    'karar_boyut': 'Seçilen loss ile (128 - 64) CI alt > 0 ise 128, değilse 64.',
    'karar_gorunus': 'Seçilen kol - konum referansı: CI alt > 0 -> gorunus_ustun; CI üst < 0 -> gorunus_zayif; aksi -> fark_yok. '
                     'E7 beklentisini belirler, E7 iptal edilmez.',
    'holm': 'Üç p-değeri (loss, boyut, görünüş-konum) Holm ile raporlanır.',
    'ikincil': ['d=1 yakın çift', 'd=10 ve d=1 tüm kutular', 'şans düzeyi'],
}
_e3_sha = hashlib.sha1(json.dumps(e3_prereg, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
if E3_PREREG.exists():
    if json.load(open(E3_PREREG)).get('kural_sha1') != _e3_sha:
        raise RuntimeError(f'E3 ön kaydı değişti; kural sonradan değiştirilemez: {E3_PREREG}')
    print('E3 ön kaydı mevcut ve değişmedi.')
else:
    with open(E3_PREREG, 'w') as fh:
        json.dump({**e3_prereg, 'kural_sha1': _e3_sha, 'olusturma': time.strftime('%Y-%m-%d %H:%M:%S')}, fh, indent=1, ensure_ascii=False)
    print('E3 ön kaydı yazıldı:', E3_PREREG)
E3_PREREG_SHA256 = hashlib.sha256(E3_PREREG.read_bytes()).hexdigest()
print('sha256:', E3_PREREG_SHA256)

In [ ]:
# E3.2) Kutu tabloları (train: TrainLabels + YOLO train kareleri; val: DV1 GT + val kareleri) ve feature çıkarma
import torch, torchvision
from torchvision.ops import roi_align
from ultralytics import YOLO
import pandas as pd

# ---- (a) train kutuları ----
_VAL = set(VAL_SEQS)
_tr_imgs = {}
for p in SPLITS['train']['images']:
    seq, idx = Path(p).stem.rsplit('_', 1)
    _tr_imgs.setdefault(seq, {})[int(idx)] = p
assert not (set(_tr_imgs) & _VAL), f'Train ve val sekansları kesişiyor: {set(_tr_imgs) & _VAL}'
E3_TRAIN_SEQS = sorted(s for s in _tr_imgs if s not in set(E3_CFG['exclude']))
print('Train sekansı:', len(E3_TRAIN_SEQS), '| hariç:', sorted(set(_tr_imgs) & set(E3_CFG['exclude'])))

def read_mot_text(txt):
    """tir_jde.dataset.read_mot ile aynı ayrıştırma."""
    rows = []
    for line in txt.splitlines():
        parts = line.strip().replace(' ', '').split(',')
        if len(parts) < 6 or not parts[0]:
            continue
        v = [float(x) for x in parts[:9]]
        rows.append(v + [1.0] * (9 - len(v)))
    return np.asarray(rows, float).reshape(-1, 9)

TRAIN_ZIP = next((p for p in TRAIN_ZIP_CANDIDATES if p.exists()), None)
assert TRAIN_ZIP is not None, f'Train.zip bulunamadı: {TRAIN_ZIP_CANDIDATES}'
_lab_pat = re.compile(r'(?:^|/)TrainLabels/([^/]+)\.txt$')
TRAIN_GT = {}
with zipfile.ZipFile(TRAIN_ZIP) as zf:
    for n in zf.namelist():
        m = _lab_pat.search(n)
        if m and m.group(1) in E3_TRAIN_SEQS:
            TRAIN_GT[m.group(1)] = read_mot_text(zf.read(n).decode('utf-8'))
_miss = sorted(set(E3_TRAIN_SEQS) - set(TRAIN_GT))
assert not _miss, f'TrainLabels içinde bulunamayan sekanslar: {_miss}'

def select_train_boxes(gt_by_seq, imgs_by_seq, step, cap, min_side):
    """Her step. kare; kenarı < min_side kutular atılır; kimlik başına en çok cap kutu (zamana eşit aralıklı)."""
    rows, gid_map = [], {}
    for s in sorted(gt_by_seq):
        g = gt_by_seq[s]
        g = g[((g[:, 0] - 1) % step == 0) & (g[:, 4] >= min_side) & (g[:, 5] >= min_side)]
        g = g[np.isin(g[:, 0].astype(int), list(imgs_by_seq[s].keys()))]
        for tid in np.unique(g[:, 1]):
            gg = g[g[:, 1] == tid]
            gg = gg[np.argsort(gg[:, 0], kind='stable')]
            if len(gg) > cap:
                gg = gg[np.unique(np.linspace(0, len(gg) - 1, cap).round().astype(int))]
            gid = gid_map.setdefault((s, int(tid)), len(gid_map))
            for r in gg:
                rows.append((s, int(r[0]), gid, r[2], r[3], r[4], r[5], imgs_by_seq[s][int(r[0])]))
    return pd.DataFrame(rows, columns=['seq', 'f', 'gid', 'x', 'y', 'w', 'h', 'img'])

E3_TR = select_train_boxes(TRAIN_GT, _tr_imgs, E3_CFG['train_step'], E3_CFG['cap_per_id'], E3_CFG['min_side'])
print(f'Train: {len(E3_TR)} kutu, {E3_TR.gid.nunique()} kimlik, {E3_TR.groupby(["seq", "f"]).ngroups} kare')

# ---- (b) val kutuları: tüm kareler, tüm GT ----
_rows = []
for s in VAL_SEQS:
    g = load_mot(DV1 / 'gt' / f'{s}.txt'); g = g[g[:, 6] == 1]
    files = sorted(glob.glob(f'{FRAMES_ROOT}/{s}/*.jpg'))
    for r in g:
        _rows.append((s, int(r[0]), int(r[1]), r[2], r[3], r[4], r[5], files[int(r[0]) - 1]))
E3_VA = pd.DataFrame(_rows, columns=['seq', 'f', 'gid', 'x', 'y', 'w', 'h', 'img'])
print(f'Val: {len(E3_VA)} kutu, {E3_VA.groupby(["seq", "f"]).ngroups} kare')

# ---- (c) feature çıkarma: donuk detector-v1-P2, P2 (stride 4) + P3 (stride 8), RoIAlign 2x2 ----
E3_MODEL = YOLO(str(P2_LAST)).model.to('cuda').eval()
assert list(E3_MODEL.model[-1].f) == [23, 14, 17, 20], f'Detect girdileri beklenenden farklı: {E3_MODEL.model[-1].f}'
_FEATS = {}
_hooks = [E3_MODEL.model[i].register_forward_hook(lambda m, inp, out, k=k: _FEATS.__setitem__(k, out))
          for k, i in E3_CFG['feat_layers'].items()]

def letterbox_tl(img, size):
    """Sol üstte hizalı letterbox (dolgu sağ/alt, 114): kutu koordinatı yalnız r ile ölçeklenir."""
    h, w = img.shape[:2]
    r = size / max(h, w)
    nh, nw = round(h * r), round(w * r)
    out = np.full((math.ceil(nh / 32) * 32, math.ceil(nw / 32) * 32, 3), 114, np.uint8)
    out[:nh, :nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return out, r

@torch.no_grad()
def extract_features(df, batch=32):
    """df satır sırasıyla aynı sırada (N, D) float16 feature döner. Kareler sekans içinde aynı boyutta."""
    out = [None] * len(df)
    groups = list(df.groupby(['seq', 'f'], sort=True))
    t0 = time.time()
    for c0 in range(0, len(groups), batch):
        chunk = groups[c0:c0 + batch]
        ims, rs = zip(*[letterbox_tl(cv2.imread(g.img.iloc[0]), E3_CFG['imgsz']) for _, g in chunk])
        if len({im.shape for im in ims}) > 1:                     # farklı çözünürlük: tek tek işle
            sub = [(i, [i]) for i in range(len(chunk))]
        else:
            sub = [(0, list(range(len(chunk))))]
        for _, idxs in sub:
            x = torch.from_numpy(np.stack([ims[i] for i in idxs])[..., ::-1].copy()).permute(0, 3, 1, 2)
            x = x.cuda().float().div_(255)
            with torch.autocast('cuda', dtype=torch.float16):
                E3_MODEL(x)
            rois = np.concatenate([np.c_[np.full(len(chunk[i][1]), b),
                                         xywh2xyxy(chunk[i][1][['x', 'y', 'w', 'h']].values) * rs[i]]
                                   for b, i in enumerate(idxs)])
            rois = torch.tensor(rois, dtype=torch.float32, device='cuda')
            parts = []
            for k in ('P2', 'P3'):
                fm = _FEATS[k].float()
                stride = x.shape[-1] / fm.shape[-1]
                assert stride == {'P2': 4, 'P3': 8}[k], f'{k} stride {stride}'
                parts.append(roi_align(fm, rois, output_size=E3_CFG['roi'], spatial_scale=1 / stride,
                                       sampling_ratio=2, aligned=True).flatten(1))
            F_ = torch.cat(parts, 1).half().cpu().numpy()
            k0 = 0
            for i in idxs:
                ix = chunk[i][1].index.values
                for j, row_id in enumerate(ix):
                    out[row_id] = F_[k0 + j]
                k0 += len(ix)
        if (c0 // batch) % 100 == 0:
            print(f'  {min(c0 + batch, len(groups))}/{len(groups)} kare ({time.time() - t0:.0f} s)')
    return np.stack(out)

def cached_features(df, name):
    key = hashlib.sha1(pd.util.hash_pandas_object(df[['seq', 'f', 'gid', 'x', 'y', 'w', 'h']], index=False).values.tobytes()
                       + json.dumps(E3_CFG, sort_keys=True).encode() + _md5(P2_LAST).encode()).hexdigest()[:12]
    path = E3_LOCAL / f'{name}_{key}.npy'
    if path.exists():
        print(f'{name}: önbellekten ({path.name})')
        return np.load(path)
    F_ = extract_features(df.reset_index(drop=True))
    np.save(path, F_)
    return F_

E3_TR = E3_TR.reset_index(drop=True); E3_VA = E3_VA.reset_index(drop=True)
_t0 = time.time()
E3_FTR = cached_features(E3_TR, 'train')
E3_FVA = cached_features(E3_VA, 'val')
assert E3_FTR.shape[0] == len(E3_TR) and E3_FVA.shape[0] == len(E3_VA) and E3_FTR.shape[1] == E3_FVA.shape[1]
assert np.isfinite(E3_FTR.astype(np.float32)).all() and np.isfinite(E3_FVA.astype(np.float32)).all()
print(f'Feature: train {E3_FTR.shape}, val {E3_FVA.shape} ({time.time() - _t0:.0f} s)')
for h in _hooks:
    h.remove()
del E3_MODEL; torch.cuda.empty_cache()

In [ ]:
# E3.3) Başlık eğitimi (4 kol), değerlendirme, karar, kayıt
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ---- değerlendirme protokolü (numpy) ----
def build_queries(va, gap, close_px, close_only=True):
    """Sorgu: t karesindeki GT (yakın çift şartı: en yakın başka GT <= close_px). Galeri: t-gap karesindeki GT (>=2).
    Sorgunun kimliği galeride olmalı. Dönüş: liste (seq, q_row, gal_rows ndarray, doğru_index)."""
    Q = []
    for s, g in va.groupby('seq', sort=True):
        by_f = {f: gg for f, gg in g.groupby('f')}
        for f, cur in by_f.items():
            prev = by_f.get(f - gap)
            if prev is None or len(prev) < 2:
                continue
            arr = np.c_[cur.f.values, cur.gid.values, cur[['x', 'y', 'w', 'h']].values]
            nn_d = nearest_other_gt(arr)
            pg = prev.gid.values
            for j, (row_id, gid) in enumerate(zip(cur.index.values, cur.gid.values)):
                if close_only and not nn_d[j] <= close_px:
                    continue
                hit = np.where(pg == gid)[0]
                if len(hit) != 1:
                    continue
                Q.append((s, row_id, prev.index.values, int(hit[0])))
    return Q

def centers_df(df, rows):
    b = df.loc[rows, ['x', 'y', 'w', 'h']].values
    return np.c_[b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2]

def score_queries(Q, va, emb=None):
    """Sekans başına doğruluk. emb None ise konum referansı (en yakın merkez). Ayrıca şans düzeyi."""
    res = {}
    for s, q, gal, k in Q:
        if emb is None:
            d = np.linalg.norm(centers_df(va, gal) - centers_df(va, [q]), axis=1)
            pick = int(np.argmin(d))
        else:
            pick = int(np.argmax(emb[gal] @ emb[q]))
        r = res.setdefault(s, [0, 0, 0.0])
        r[0] += int(pick == k); r[1] += 1; r[2] += 1.0 / len(gal)
    return pd.DataFrame({s: {'dogru': v[0], 'n': v[1], 'acc': v[0] / v[1], 'sans': v[2] / v[1]} for s, v in res.items()}).T.astype(float)

# ---- başlık ve kayıplar ----
class Head(nn.Module):
    def __init__(self, d_in, d_out, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(inplace=True), nn.Linear(hidden, d_out))
    def forward(self, x):
        return F.normalize(self.net(x), dim=1)

class CosineClassifier(nn.Module):
    def __init__(self, d, n, s):
        super().__init__()
        self.W = nn.Parameter(torch.randn(n, d) * 0.01); self.s = s
    def forward(self, z):
        return self.s * z @ F.normalize(self.W, dim=1).t()

def batch_hard_triplet(z, y, margin):
    d = (2 - 2 * z @ z.t()).clamp_min(1e-12).sqrt()        # z birim vektör; cdist'in sıfırdaki NaN gradyanından kaçınır
    same = y[:, None] == y[None]
    pos = (d * same).max(1).values
    neg = d.masked_fill(same, float('inf')).min(1).values
    return F.relu(pos - neg + margin).mean()

def pk_indices(rng, by_id, P, K):
    ids = rng.choice(len(by_id), P, replace=False)
    return np.concatenate([rng.choice(by_id[i], K, replace=len(by_id[i]) < K) for i in ids])

_mu = E3_FTR.astype(np.float32).mean(0); _sd = E3_FTR.astype(np.float32).std(0) + 1e-6
XTR = torch.from_numpy((E3_FTR.astype(np.float32) - _mu) / _sd).cuda()
XVA = torch.from_numpy((E3_FVA.astype(np.float32) - _mu) / _sd).cuda()
YTR = torch.from_numpy(E3_TR.gid.values).long().cuda()
_by_id = [np.where(E3_TR.gid.values == i)[0] for i in range(E3_TR.gid.max() + 1)]
assert all(len(v) for v in _by_id)
np.savez(E3_DIR / 'feat_norm.npz', mu=_mu, sd=_sd)

def train_arm(loss, dim):
    torch.manual_seed(E3_CFG['seed']); rng = np.random.default_rng(E3_CFG['seed'])
    head = Head(XTR.shape[1], dim, E3_CFG['hidden']).cuda()
    cls = CosineClassifier(dim, len(_by_id), E3_CFG['ce_scale']).cuda()
    opt = torch.optim.AdamW(list(head.parameters()) + list(cls.parameters()), lr=E3_CFG['lr'], weight_decay=E3_CFG['wd'])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E3_CFG['iters'])
    log = []
    head.train()
    for it in range(E3_CFG['iters']):
        idx = torch.from_numpy(pk_indices(rng, _by_id, E3_CFG['P'], E3_CFG['K'])).cuda()
        z = head(XTR[idx]); y = YTR[idx]
        l_ce = F.cross_entropy(cls(z), y)
        l_tri = batch_hard_triplet(z, y, E3_CFG['tri_margin']) if loss == 'ce+tri' else torch.zeros((), device='cuda')
        l = l_ce + E3_CFG['tri_weight'] * l_tri
        opt.zero_grad(set_to_none=True); l.backward(); opt.step(); sch.step()
        if it % 500 == 0 or it == E3_CFG['iters'] - 1:
            log.append((it, float(l_ce), float(l_tri)))
    head.eval()
    with torch.no_grad():
        emb = torch.cat([head(XVA[i:i + 65536]) for i in range(0, len(XVA), 65536)]).cpu().numpy()
    return head, emb, log

# ---- sorgular (bir kez) ----
E3_Q = {(gap, co): build_queries(E3_VA, gap, E3_CFG['close_px'], co) for gap in E3_CFG['gaps'] for co in (True, False)}
for (gap, co), Q in E3_Q.items():
    print(f'Sorgu d={gap}, {"yakın çift" if co else "tümü"}: {len(Q)} ({len({q[0] for q in Q})} sekans)')

E3_RES, E3_EMB, E3_LOG = {}, {}, {}
for (gap, co), Q in E3_Q.items():
    E3_RES[('konum', gap, co)] = score_queries(Q, E3_VA)
(E3_DIR / 'heads').mkdir(exist_ok=True)
for arm, (loss, dim) in E3_ARMS.items():
    _t0 = time.time()
    head, emb, log = train_arm(loss, dim)
    torch.save(head.state_dict(), E3_DIR / 'heads' / f'{arm}.pt')
    E3_EMB[arm], E3_LOG[arm] = emb.astype(np.float16), log
    for (gap, co), Q in E3_Q.items():
        E3_RES[(arm, gap, co)] = score_queries(Q, E3_VA, emb)
    print(f'{arm}: {time.time() - _t0:.0f} s | son kayıp CE {log[-1][1]:.3f} triplet {log[-1][2]:.3f}')

# ---- özet tablo ----
_cols = []
for gap in E3_CFG['gaps']:
    for co in (True, False):
        _cols.append((gap, co))
E3_TAB = pd.DataFrame({f'd={g} {"yakın" if c else "tümü"}': {k: round(float((E3_RES[(k, g, c)].dogru.sum()) / E3_RES[(k, g, c)].n.sum()), 4)
                                                               for k in list(E3_ARMS) + ['konum']}
                       for g, c in _cols})
E3_TAB.loc['şans'] = [round(float((E3_RES[('konum', g, c)].sans * E3_RES[('konum', g, c)].n).sum() / E3_RES[('konum', g, c)].n.sum()), 4)
                      for g, c in _cols]
print('\nTop-1 kimlik doğruluğu (havuzlanmış):'); print(E3_TAB)

# ---- karar (önceden kayıtlı) ----
_pg = E3_CFG['primary_gap']
def _acc(k):
    return E3_RES[(k, _pg, True)].sort_index().acc.astype(float)
def e3_compare(a, b, name):
    x, y = _acc(a), _acc(b)
    assert list(x.index) == list(y.index)
    m, ci, p = bootstrap_diff(x.values, y.values)
    return {'karsilastirma': name, 'fark': round(m, 4), 'CI95': (round(ci[0], 4), round(ci[1], 4)), 'p': round(p, 4)}

E3_C_LOSS = e3_compare('ce128', 'cetri128', 'CE+triplet - CE (128)')
E3_LOSS = 'ce+tri' if E3_C_LOSS['CI95'][0] > 0 else 'ce'
_l = 'cetri' if E3_LOSS == 'ce+tri' else 'ce'
E3_C_DIM = e3_compare(f'{_l}64', f'{_l}128', f'128 - 64 ({E3_LOSS})')
E3_DIM = 128 if E3_C_DIM['CI95'][0] > 0 else 64
E3_CHOSEN = f'{_l}{E3_DIM}'
E3_C_POS = e3_compare('konum', E3_CHOSEN, f'{E3_CHOSEN} - konum')
E3_APPEAR = ('gorunus_ustun' if E3_C_POS['CI95'][0] > 0 else 'gorunus_zayif' if E3_C_POS['CI95'][1] < 0 else 'fark_yok')
E3_DEC = pd.DataFrame([E3_C_LOSS, E3_C_DIM, E3_C_POS])
E3_DEC['p_holm'] = holm(E3_DEC.p.values).round(4)
print(f'\nKARAR (d={_pg}, yakın çift):'); print(E3_DEC.to_string(index=False))
print(f'Seçilen: loss = {E3_LOSS}, boyut = {E3_DIM} ({E3_CHOSEN}) | görünüş ve konum: {E3_APPEAR}')

# ---- görsel: doğruluk çubukları ----
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
_order = list(E3_ARMS) + ['konum', 'şans']
for ax, gap in zip(axs, E3_CFG['gaps']):
    v = E3_TAB.loc[_order, f'd={gap} yakın'].values
    ax.bar(_order, v, color=['#4c72b0'] * 4 + ['#dd8452', '#bbbbbb'])
    for i, x in enumerate(v):
        ax.text(i, x + 0.01, f'{x:.2f}', ha='center', fontsize=8)
    ax.set_title(f'Yakın çift, d = {gap} kare'); ax.set_ylim(0, 1.05); ax.tick_params(axis='x', rotation=30)
axs[0].set_ylabel('top-1 kimlik doğruluğu')
fig.tight_layout(); E3_FIG = E3_DIR / 'E3_accuracy.png'; fig.savefig(E3_FIG, dpi=130); plt.show()

# ---- kayıt ----
np.save(E3_DIR / f'val_emb_{E3_CHOSEN}.npy', E3_EMB[E3_CHOSEN])
E3_VA[['seq', 'f', 'gid', 'x', 'y', 'w', 'h']].to_csv(E3_DIR / 'val_boxes.csv', index=False)
E3_RESULTS = {'secilen': E3_CHOSEN, 'loss': E3_LOSS, 'boyut': E3_DIM, 'gorunus_konum': E3_APPEAR,
              'kararlar': E3_DEC.to_dict(orient='records'), 'tablo': E3_TAB.to_dict(),
              'sekans_basina': {k: E3_RES[(k, _pg, True)].acc.astype(float).round(4).to_dict() for k in list(E3_ARMS) + ['konum']},
              'sorgu_sayisi': {f'd={g} {"yakın" if c else "tümü"}': len(Q) for (g, c), Q in E3_Q.items()},
              'egitim': {'kutu': len(E3_TR), 'kimlik': int(E3_TR.gid.nunique()), 'sekans': len(E3_TRAIN_SEQS)},
              'kayip_log': E3_LOG, 'on_kayit_sha256': E3_PREREG_SHA256, 'tarih': time.strftime('%Y-%m-%d %H:%M:%S')}
with open(E3_DIR / 'E3_results.json', 'w') as fh:
    json.dump(E3_RESULTS, fh, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
print('Kaydedildi:', E3_DIR / 'E3_results.json')

if MLFLOW_OK:
    try:
        if mlflow.get_experiment_by_name('TIR-JDE-E3') is None:
            mlflow.create_experiment('TIR-JDE-E3', artifact_location=(E3_DIR / 'mlartifacts').as_uri())
        mlflow.set_experiment('TIR-JDE-E3')
        for arm, (loss, dim) in E3_ARMS.items():
            with mlflow.start_run(run_name=f'E3_{arm}'):
                mlflow.log_params({'loss': loss, 'dim': dim, **{k: str(v) for k, v in E3_CFG.items()}})
                mlflow.log_metrics({f'acc_d{g}_{"yakin" if c else "tumu"}': float(E3_TAB.loc[arm, f'd={g} {"yakın" if c else "tümü"}'])
                                    for g, c in _cols})
                mlflow.log_metric('secildi', float(arm == E3_CHOSEN))
        with mlflow.start_run(run_name='E3_karar'):
            mlflow.log_params({'secilen': E3_CHOSEN, 'gorunus_konum': E3_APPEAR})
            mlflow.log_metrics({'pos_acc_d10_yakin': float(E3_TAB.loc['konum', f'd={_pg} yakın']),
                                'fark_secilen_konum': E3_C_POS['fark']})
            for f in (E3_FIG, E3_DIR / 'E3_results.json', E3_PREREG):
                mlflow.log_artifact(str(f))
        shutil.copy(LOCAL / 'mlflow_d2.db', D2_DIR / 'mlflow_d2.db')
        print('MLflow: TIR-JDE-E3 kaydedildi, veritabanı Drive\'a kopyalandı.')
    except Exception as e:
        print('MLflow kaydı başarısız (sonuçlar dosyada):', repr(e))

In [ ]:
# E3.4) Görsel kayıt: yakın çift örnekleri (d = 10). Sorgu ve galeri kırpıntıları; görünüş ve konum seçimi
_Q = E3_Q[(E3_CFG['primary_gap'], True)]
_emb = E3_EMB[E3_CHOSEN].astype(np.float32)
_rng = np.random.default_rng(0)
_cat = {'ikisi doğru': [], 'görünüş doğru, konum yanlış': [], 'konum doğru, görünüş yanlış': [], 'ikisi yanlış': []}
for q in _Q:
    s, qr, gal, k = q
    pa = int(np.argmax(_emb[gal] @ _emb[qr]))
    pp = int(np.argmin(np.linalg.norm(centers_df(E3_VA, gal) - centers_df(E3_VA, [qr]), axis=1)))
    key = ('ikisi doğru' if pa == k else 'konum doğru, görünüş yanlış') if pp == k else \
          ('görünüş doğru, konum yanlış' if pa == k else 'ikisi yanlış')
    _cat[key].append((q, pa, pp))
print({k: len(v) for k, v in _cat.items()})
_ex = []
for key in _cat:
    lst = _cat[key]
    _ex += [(key,) + lst[i] for i in _rng.permutation(len(lst))[:2]]

CROP_E, ZOOM_E, MAXG = 32, 4, 4
def crop_box(row):
    r = E3_VA.loc[row]
    img = cv2.imread(r.img)
    H, W = img.shape[:2]
    cx, cy = r.x + r.w / 2, r.y + r.h / 2
    x0 = int(np.clip(cx - CROP_E / 2, 0, W - CROP_E)); y0 = int(np.clip(cy - CROP_E / 2, 0, H - CROP_E))
    p = cv2.resize(img[y0:y0 + CROP_E, x0:x0 + CROP_E], None, fx=ZOOM_E, fy=ZOOM_E, interpolation=cv2.INTER_NEAREST)
    cv2.rectangle(p, (int((r.x - x0) * ZOOM_E), int((r.y - y0) * ZOOM_E)),
                  (int((r.x + r.w - x0) * ZOOM_E), int((r.y + r.h - y0) * ZOOM_E)), (0, 255, 0), 1)
    return p[..., ::-1]

E3_EX_FIG = None
if _ex:
    fig, axs = plt.subplots(len(_ex), 1 + MAXG, figsize=(2.2 * (1 + MAXG), 2.4 * len(_ex)), squeeze=False)
    for i, (key, (s, qr, gal, k), pa, pp) in enumerate(_ex):
        axs[i][0].imshow(crop_box(qr)); axs[i][0].set_title(f'sorgu (t)\n{s} kare {E3_VA.loc[qr, "f"]}', fontsize=7)
        axs[i][0].set_ylabel(key, fontsize=7)
        sims = _emb[gal] @ _emb[qr]
        order = list(dict.fromkeys([k, pp, pa] + [int(j) for j in np.argsort(-sims)]))[:MAXG]   # doğru, [K], [G] hep görünür
        order = sorted(order, key=lambda j: -sims[j])
        for j in range(MAXG):
            ax = axs[i][1 + j]; ax.set_xticks([]); ax.set_yticks([])
            if j >= len(order) or j >= len(gal):
                ax.axis('off'); continue
            gj = order[j]
            ax.imshow(crop_box(gal[gj]))
            tags = ('✓ doğru ' if gj == k else '') + ('[G] ' if gj == pa else '') + ('[K]' if gj == pp else '')
            ax.set_title(f't−10 | cos {sims[gj]:.2f}\n{tags}', fontsize=7, color='green' if gj == k else 'black')
            for sp in ax.spines.values():
                sp.set_edgecolor('green' if gj == k else '#999999'); sp.set_linewidth(2 if gj == k else 1)
        axs[i][0].set_xticks([]); axs[i][0].set_yticks([])
    fig.suptitle(f'E3 ({E3_CHOSEN}): yakın çift sorguları, d = 10. [G] görünüş seçimi, [K] konum seçimi; 32×32 px, 4× büyütme',
                 fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, 0.98)); E3_EX_FIG = E3_DIR / 'E3_examples.png'; fig.savefig(E3_EX_FIG, dpi=120); plt.show()

if MLFLOW_OK and E3_EX_FIG is not None:
    try:
        _r = mlflow.search_runs(experiment_names=['TIR-JDE-E3'], filter_string="tags.mlflow.runName = 'E3_karar'",
                                order_by=['start_time DESC'], max_results=1)
        mlflow.tracking.MlflowClient().log_artifact(_r.run_id.iloc[0], str(E3_EX_FIG))
        shutil.copy(LOCAL / 'mlflow_d2.db', D2_DIR / 'mlflow_d2.db')
    except Exception as e:
        print('MLflow eki başarısız:', repr(e))
print('Görsel:', E3_EX_FIG)

In [ ]:
# E3.5) Confluence raporu (sonuçlardan otomatik)
def _f(x, n=3):
    return f'{float(x):.{n}f}'.replace('.', ',')
def _ci(c):
    return f'{_f(c["fark"])}, CI95 ({_f(c["CI95"][0])}; {_f(c["CI95"][1])}), p {_f(c["p"])}'
_APP_TXT = {'gorunus_ustun': 'görünüş konumdan fazla bilgi taşıyor', 'gorunus_zayif': 'görünüş konumdan zayıf',
            'fark_yok': 'görünüş ile konum arasında anlamlı fark yok'}
_hdr = ' | '.join(E3_TAB.columns)
_rows = '\n'.join(f'| {k} | ' + ' | '.join(_f(E3_TAB.loc[k, c]) for c in E3_TAB.columns) + ' |' for k in E3_TAB.index)
_rows_dec = '\n'.join(f'| {r.karsilastirma} | {_f(r.fark)} | ({_f(r.CI95[0])}; {_f(r.CI95[1])}) | {_f(r.p_holm)} |'
                      for r in E3_DEC.itertuples())
E3_REPORT = f"""## E3: Kimlik başlığı sondajı (head-only), sonuç raporu

**Durum:** Geliştirme seti (val, 16 sekans), tek seed. **Seçilen: {E3_CHOSEN} (loss {E3_LOSS}, boyut {E3_DIM}). Görünüş ve konum: {_APP_TXT[E3_APPEAR]}.**

**Ne:** detector-v1-P2 donuk; P2 ve P3 neck feature'ları (RoIAlign 2×2) üstüne küçük bir kimlik başlığı eğitildi. Dört kol: CE-64, CE-128, CE+triplet-64, CE+triplet-128. Eğitim: {len(E3_TRAIN_SEQS)} train sekansı, {E3_RESULTS['egitim']['kutu']} kutu, {E3_RESULTS['egitim']['kimlik']} kimlik. Kural çalıştırmadan önce kilitlendi (E3_prereg.json, sha256 {E3_PREREG_SHA256[:16]}…).

**Ölçü:** Sorgu, t karesinde başka bir GT'ye ≤ 12,7 px olan GT kutusu (yakın çift); galeri, t − d karesindeki GT kutuları. Top-1 kimlik doğruluğu. Konum referansı: galeride merkezi en yakın kutu (hareket modeli yok).

### Top-1 kimlik doğruluğu (havuzlanmış)

| Kol | {_hdr} |
| --- | {' | '.join('---' for _ in E3_TAB.columns)} |
{_rows}

### Karar (d = {E3_CFG['primary_gap']}, yakın çift, sekans başına bootstrap)

| Karşılaştırma | Fark | CI95 | p (Holm) |
| --- | --- | --- | --- |
{_rows_dec}

- Loss: {_ci(E3_C_LOSS)} → **{E3_LOSS}**
- Boyut: {_ci(E3_C_DIM)} → **{E3_DIM}**
- Görünüş − konum: {_ci(E3_C_POS)} → **{_APP_TXT[E3_APPEAR]}**

**E7 için anlamı:** {'Görünüş, yakın çiftte konumun ötesinde bilgi taşıyor; r görünüşe ağırlık verebilir.' if E3_APPEAR == 'gorunus_ustun' else 'Head-only görünüş konumu geçemiyor; E7 de r konuma yaslanmalı ve ortak eğitimin (B2) görünüşü güçlendirip güçlendirmediği ayrıca test edilmeli.' if E3_APPEAR == 'gorunus_zayif' else 'Görünüş ve konum yakın çiftte benzer; ikisinin birleşimi (E7) tek başına her ikisinden iyi olabilir, bu E7 de ölçülür.'}

**Sınırlar:** Head-only (detector feature'ları kimlik için eğitilmedi); galeri tek kareden (gerçek tracker'da iz geçmişi var); konum referansı hareket modeli içermiyor; tek seed; val 16 sekans.

**Dosyalar:** WORK/E3/E3_results.json, E3_prereg.json, E3_accuracy.png, E3_examples.png, heads/*.pt, feat_norm.npz, val_emb_{E3_CHOSEN}.npy. MLflow: TIR-JDE-E3.
"""
(E3_DIR / 'E3_confluence_raporu.md').write_text(E3_REPORT, encoding='utf-8')
print(E3_REPORT)

## E7a: Görünüş ve konumun birleştirilmesi (sabit ağırlık)

**Soru:** E3'teki kimlik başlığı (ce128) tracker'a eklendiğinde takip iyileşiyor mu? Görünüş ve konum nasıl birleştirilmeli: SB'nin yöntemi (ikisinin **minimumu**) mı, **ağırlıklı toplam** mı?

**Neden:** E3'e göre görünüş tek başına konumdan zayıf (yakın çift, d = 10: %78 / %86), ama konumun yanıldığı sorguların %53'ünde doğru; ideal birleşimin tavanı %93. Bu deney, sabit bir ağırlıkla bu tavanın ne kadarının gerçek tracker'da alınabildiğini ölçer. Öğrenilmiş, tespit başına ağırlık (r) E7b'de.

**Ne yapacağız:**
1. p2 tespitlerinin (4.1, `DET_DIRS['p2']`) her biri için E3'ün ce128 başlığıyla 128 boyutlu embedding çıkarılır (E3 ile aynı feature: P2 + P3, RoIAlign 2×2). 1. karedeki GT kutuları için de aynısı yapılır (tracker 1. karede GT'yi kullanır).
2. Dondurulmuş tracker (sparseOptFlow + NWD + K10) ReID açık olarak koşulur. Tracker iz başına embedding geçmişinin üstel ortalamasını tutar (α = 0,9).
3. **Kollar:**
   - `noreid`: görünüş yok (D2'deki dondurulmuş koşu, referans).
   - `min`: SB'nin yöntemi, `min(konum, görünüş)`.
   - `w25`, `w50`, `w75`: ağırlıklı toplam, `(1 − w) · konum + w · görünüş`, w ∈ {0,25; 0,5; 0,75}.
   SB'nin kapıları tüm ReID kollarında aynı: konum maliyeti 0,5'ten büyükse görünüş = 1; görünüş uzaklığı 0,25'ten büyükse görünüş = 1. Yalnızca birleştirme satırı değişir.

**Önceden yazılmış karar kuralı** (`E7_DIR/E7a_prereg.json`, çalıştırmadan önce kilitlenir):
- **Birincil:** Her ReID kolu − `noreid`, HOTA; eşleştirilmiş sekans bootstrap (B = 10.000); 4 karşılaştırmaya Holm.
- **Koruma:** MOTA_seq farkının CI üst sınırı ≥ 0.
- **Karar:** Holm p < 0,05, fark > 0 ve korumayı geçen kollar arasında HOTA'sı en yüksek olan E7a yapılandırması olur. Hiçbiri geçmezse görünüş sabit ağırlıkla eklenmez (E7b yine yapılır).
- **İkincil (Holm, kararı değiştirmez):** Ağırlıklı kollar − `min` (HOTA, 3 karşılaştırma); yakın çift IDSW, toplam IDSW, AssA, DetA (seçilen ya da w50 − `noreid`).

**Sınırlar:** Head-only embedding (detector kimlik için eğitilmedi); w ızgarası kaba (3 değer); tek seed; val 16 sekans.

**Süre:** Embedding çıkarma ~5 dk; 4 tracker koşusu ~20 dk; değerlendirme ~5 dk; video ~2 dk.

**Ön koşul (aynı oturum):** 4.1 ve 4.2 (`DET_DIRS`, `TRK_FROZEN`, `PS_FZ`), E3.1 ve E3.2 (feature fonksiyonları, `E3_VA`, `E3_FVA`). E3.3 gerekmez; başlık Drive'dan yüklenir.

In [ ]:
# E7a.1) Ön kayıt: karar kuralı sonuçlardan ÖNCE kilitlenir
E7_DIR = WORK / 'E7'
E7_DIR.mkdir(parents=True, exist_ok=True)
E7_ARMS = {'min': None, 'w25': 0.25, 'w50': 0.5, 'w75': 0.75}
E7_HEAD = E3_DIR / 'heads' / 'ce128.pt'
assert E7_HEAD.exists() and (E3_DIR / 'feat_norm.npz').exists(), 'E3 başlığı veya feat_norm.npz yok'
E7_CFG = {'detector': str(P2_LAST), 'head': str(E7_HEAD), 'head_md5': _md5(E7_HEAD), 'emb_dim': 128,
          'tracker': dict(FROZEN_ARGS), 'proximity_thresh': 0.5, 'appearance_thresh': 0.25, 'ema_alpha': 0.9,
          'kollar': {k: ('min' if v is None else f'(1-{v})*konum + {v}*gorunus') for k, v in E7_ARMS.items()}}
E7_PREREG = E7_DIR / 'E7a_prereg.json'
e7_prereg = {
    'deney': 'E7a: görünüş (E3 ce128) + konum birleştirme, sabit ağırlık; p2 tespitleri, dondurulmuş tracker',
    'ayarlar': E7_CFG,
    'birincil': 'Her ReID kolu - noreid, HOTA, eşleştirilmiş sekans bootstrap B=10000; 4 karşılaştırmaya Holm.',
    'koruma': 'MOTA_seq farkının CI üst sınırı >= 0.',
    'karar': 'Holm p < 0.05, fark > 0 ve korumayı geçen kollar arasında HOTA en yüksek olan seçilir; hiçbiri yoksa eklenmez.',
    'ikincil_holm': ['ağırlıklı kollar - min (HOTA, 3 karşılaştırma)'],
    'ikincil_bilgi': ['yakın çift IDSW', 'toplam IDSW', 'AssA', 'DetA (seçilen ya da w50 - noreid)'],
    'video': 'noreid ve seçilen kol (yoksa w50); yakın çift sorgusu en çok olan sekansın en yoğun 300 karesi',
}
_e7_sha = hashlib.sha1(json.dumps(e7_prereg, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
if E7_PREREG.exists():
    if json.load(open(E7_PREREG)).get('kural_sha1') != _e7_sha:
        raise RuntimeError(f'E7a ön kaydı değişti; kural sonradan değiştirilemez: {E7_PREREG}')
    print('E7a ön kaydı mevcut ve değişmedi.')
else:
    with open(E7_PREREG, 'w') as fh:
        json.dump({**e7_prereg, 'kural_sha1': _e7_sha, 'olusturma': time.strftime('%Y-%m-%d %H:%M:%S')}, fh, indent=1, ensure_ascii=False)
    print('E7a ön kaydı yazıldı:', E7_PREREG)
E7_PREREG_SHA256 = hashlib.sha256(E7_PREREG.read_bytes()).hexdigest()
print('sha256:', E7_PREREG_SHA256)

In [ ]:
# E7a.2) p2 tespitleri ve 1. kare GT için embedding (ce128) -> tracker'ın okuyacağı emb_dir/<seq>.npz
import torch, torch.nn as nn, torch.nn.functional as F
from ultralytics import YOLO

class HeadE7(nn.Module):                     # E3.3'teki Head ile aynı yapı (state_dict anahtarları aynı)
    def __init__(self, d_in, d_out, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(inplace=True), nn.Linear(hidden, d_out))
    def forward(self, x):
        return F.normalize(self.net(x), dim=1)

_nrm = np.load(E3_DIR / 'feat_norm.npz')
_e7_head = HeadE7(E3_FVA.shape[1], 128, E3_CFG['hidden']).cuda()
_e7_head.load_state_dict(torch.load(E7_HEAD, map_location='cuda'))
_e7_head.eval()

@torch.no_grad()
def embed(F16):
    X = torch.from_numpy((F16.astype(np.float32) - _nrm['mu']) / _nrm['sd']).cuda()
    return torch.cat([_e7_head(X[i:i + 65536]) for i in range(0, len(X), 65536)]).cpu().numpy().astype(np.float32)

# Doğrulama: başlık E3'teki val embedding'ini birebir üretiyor mu?
_chk = embed(E3_FVA[:2000])
_ref = np.load(E3_DIR / 'val_emb_ce128.npy')[:2000].astype(np.float32)
assert np.abs(_chk - _ref).max() < 1e-2, f'Başlık E3 ile tutarsız: {np.abs(_chk - _ref).max()}'
print('Başlık E3 ile tutarlı (max fark', float(np.abs(_chk - _ref).max()), ')')

# Tespit tablosu: dets dosyasının satır sırasıyla birebir (tracker satır indeksine göre eşler)
_rows = []
for s in VAL_SEQS:
    d = load_dets(DET_DIRS['p2'] / f'{s}.txt')
    files = sorted(glob.glob(f'{FRAMES_ROOT}/{s}/*.jpg'))
    for i, r in enumerate(d):
        _rows.append((s, int(r[0]), -1, r[1], r[2], r[3] - r[1], r[4] - r[2], files[int(r[0]) - 1], i))
E7_DETS = pd.DataFrame(_rows, columns=['seq', 'f', 'gid', 'x', 'y', 'w', 'h', 'img', 'row'])
print(f'p2 tespiti: {len(E7_DETS)} ({E7_DETS.groupby(["seq", "f"]).ngroups} kare)')

E3_MODEL = YOLO(str(P2_LAST)).model.to('cuda').eval()
_FEATS.clear()
_hooks = [E3_MODEL.model[i].register_forward_hook(lambda m, inp, out, k=k: _FEATS.__setitem__(k, out))
          for k, i in E3_CFG['feat_layers'].items()]
try:
    _t0 = time.time()
    E7_FDET = cached_features(E7_DETS, 'p2dets')
finally:
    for h in _hooks:
        h.remove()
    del E3_MODEL; torch.cuda.empty_cache()
assert E7_FDET.shape == (len(E7_DETS), E3_FVA.shape[1]) and np.isfinite(E7_FDET.astype(np.float32)).all()
E7_EDET = embed(E7_FDET)
print(f'Tespit embedding: {E7_EDET.shape} ({time.time() - _t0:.0f} s)')

# 1. kare GT (tracker'ın 'prior' satırları: DV1 gt dosyasındaki sırayla)
E7_EMB_DIR = LOCAL / 'E7' / 'emb_ce128'
E7_EMB_DIR.mkdir(parents=True, exist_ok=True)
for s in VAL_SEQS:
    g = load_mot(DV1 / 'gt' / f'{s}.txt')
    g1 = g[g[:, 0] == 1]
    va1 = E3_VA[(E3_VA.seq == s) & (E3_VA.f == 1)]
    assert len(va1) == len(g1) and np.allclose(va1[['x', 'y', 'w', 'h']].values, g1[:, 2:6]), f'{s}: 1. kare GT sırası uyuşmuyor'
    gt1 = embed(E3_FVA[va1.index.values])
    m = (E7_DETS.seq == s).values
    det = E7_EDET[m][np.argsort(E7_DETS.row.values[m])]
    assert len(det) == len(load_dets(DET_DIRS['p2'] / f'{s}.txt'))
    np.savez(E7_EMB_DIR / f'{s}.npz', det_emb=det, gt1_emb=gt1)
print('emb_dir hazır:', E7_EMB_DIR, '|', len(list(E7_EMB_DIR.glob('*.npz'))), 'sekans')

In [ ]:
# E7a.3) Tracker koşuları: min (SB) ve ağırlıklı toplam (w25, w50, w75). Yama çalışma anında, sayaçla doğrulanır.
E7_BLOCK = r"""
# --- E7a: görünüş + konum birleştirme ---
import inspect, textwrap
STATS.update({'emb_calls': 0, 'fuse_calls': 0})
# Bu runner mc_bot_sort'u ReID kapalı taklitle içe aktarıyor; track_7a'nın vektör bankası (install_stubs) sonra kuruluyor.
# Aracı sınıf, BoTSORT kurulduğu anda sys.modules'taki güncel (track_7a) arayüzü kullanır.
class _E7FRProxy:
    def __init__(self, *a, **k):
        self._impl = sys.modules['fast_reid.fast_reid_interfece'].FastReIDInterface(*a, **k)
    def inference(self, img, dets):
        return self._impl.inference(img, dets)
MB.FastReIDInterface = _E7FRProxy
_emb_orig = M.embedding_distance
def _emb_count(*a, **k):
    STATS['emb_calls'] += 1
    return _emb_orig(*a, **k)
M.embedding_distance = _emb_count
_W = os.environ.get('E7_FUSE_W', '')
if _W:
    MB._E7_W = float(_W)
    def _e7_fuse(pos, emb):
        STATS['fuse_calls'] += 1
        return (1.0 - MB._E7_W) * pos + MB._E7_W * emb
    MB._e7_fuse = _e7_fuse
    _src = textwrap.dedent(inspect.getsource(_upd_orig))
    _old = 'dists = np.minimum(ious_dists, emb_dists)'
    assert _src.count(_old) == 1, 'birleştirme satırı bulunamadı'
    _ns = {}
    exec(compile(_src.replace(_old, 'dists = _e7_fuse(ious_dists, emb_dists)'), MB.__file__, 'exec'), MB.__dict__, _ns)
    _upd_orig = _ns['update']
"""
_anchor = 'sys.argv = [TRACK] + sys.argv[1:]'
assert PATCH_SRC.count(_anchor) == 1
E7_PATCH_FILE = LOCAL / 'run_patched_e7.py'
E7_PATCH_FILE.write_text(PATCH_SRC.replace(_anchor, E7_BLOCK + '\n' + _anchor))

def run_e7(out_dir, w):
    global PATCH_FILE
    _old_pf = PATCH_FILE
    PATCH_FILE = E7_PATCH_FILE
    os.environ.pop('E7_FUSE_W', None)
    if w is not None:
        os.environ['E7_FUSE_W'] = str(w)
    try:
        st = run_tracker_patched(DET_DIRS['p2'], out_dir, **FROZEN_ARGS,
                                 extra=('--emb_dir', str(E7_EMB_DIR), '--proximity_thresh', '0.5', '--appearance_thresh', '0.25'))
    finally:
        PATCH_FILE = _old_pf
        os.environ.pop('E7_FUSE_W', None)
    assert st.get('emb_calls', 0) > 0, f'ReID kullanılmadı: {st}'
    assert (st.get('fuse_calls', 0) > 0) == (w is not None), f'Birleştirme yaması beklendiği gibi çalışmadı: {st}'
    return st

E7_TRK, E7_STATS = {'noreid': TRK_FROZEN['p2']}, {}
for arm, w in E7_ARMS.items():
    _t0 = time.time()
    E7_TRK[arm] = str(E7_DIR / f'track_{arm}')
    E7_STATS[arm] = run_e7(E7_TRK[arm], w)
    print(f'{arm}: {time.time() - _t0:.0f} s')
print('E7a tracker koşuları tamam.')

In [ ]:
# E7a.4) Değerlendirme, karar, kayıt
import matplotlib.pyplot as plt
_cols = ['MOTA', 'MOTA_seq', 'HOTA', 'DetA', 'AssA', 'IDF1', 'IDSW', 'Frag', 'FP', 'FN']
E7_TAB = summary_table(E7_TRK, 'E7a')[_cols].astype(float)
print(E7_TAB.round(4))
E7_PS = {a: per_seq_table(d, f'E7a_{a}') for a, d in E7_TRK.items()}

# --- Birincil + koruma ---
_ra = list(E7_ARMS)
E7_PRIM = pd.DataFrame([compare(E7_PS['noreid'], E7_PS[a], 'HOTA') | {'kol': a} for a in _ra])
E7_PRIM['p_holm'] = holm(E7_PRIM.p.values).round(4)
E7_PRIM['koruma_MOTAseq_ust'] = [compare(E7_PS['noreid'], E7_PS[a], 'MOTA')['CI95'][1] for a in _ra]
E7_PRIM['gecti'] = (E7_PRIM.fark > 0) & (E7_PRIM.p_holm < 0.05) & (E7_PRIM.koruma_MOTAseq_ust >= 0)
print('\nBİRİNCİL (kol - noreid, HOTA; Holm):'); print(E7_PRIM.to_string(index=False))
_pass = E7_PRIM[E7_PRIM.gecti].kol.tolist()
E7_CHOSEN = max(_pass, key=lambda a: E7_TAB.loc[a, 'HOTA']) if _pass else None
E7_DECISION = f'{E7_CHOSEN} eklenir' if E7_CHOSEN else 'görünüş sabit ağırlıkla eklenmez'
print('E7a KARAR:', E7_DECISION)

# --- İkincil ---
E7_VS_MIN = pd.DataFrame([compare(E7_PS['min'], E7_PS[a], 'HOTA') | {'kol': f'{a} - min'} for a in ('w25', 'w50', 'w75')])
E7_VS_MIN['p_holm'] = holm(E7_VS_MIN.p.values).round(4)
print('\nİKİNCİL (ağırlıklı - min, HOTA; Holm):'); print(E7_VS_MIN.to_string(index=False))

_ref_arm = E7_CHOSEN or 'w50'
def close_idsw(trk_dir):
    out = {}
    for s in VAL_SEQS:
        g = load_mot(DV1 / 'gt' / f'{s}.txt'); g = g[g[:, 6] == 1]
        ev = idsw_groups(g, load_mot(Path(trk_dir) / f'{s}.txt'))
        out[s] = {grp: int((ev.grup == grp).sum()) if len(ev) else 0 for grp in ('yakın çift', 'uzun boşluk', 'diğer')}
    return pd.DataFrame(out).T
E7_IDG = {a: close_idsw(E7_TRK[a]) for a in ('noreid', _ref_arm, 'min')}
_cp0, _cp1 = E7_IDG['noreid']['yakın çift'].loc[VAL_SEQS], E7_IDG[_ref_arm]['yakın çift'].loc[VAL_SEQS]
_m, _ci, _p = bootstrap_diff(_cp0.values, _cp1.values)
E7_SEC = [{'metrik': 'yakın çift IDSW (sekans başına)', 'fark': round(_m, 3), 'CI95': (round(_ci[0], 3), round(_ci[1], 3)), 'p': round(_p, 4)}]
E7_SEC += [compare(E7_PS['noreid'], E7_PS[_ref_arm], m) for m in ('IDSW', 'AssA', 'DetA')]
E7_SEC = pd.DataFrame(E7_SEC)
print(f'\nBİLGİ ({_ref_arm} - noreid):'); print(E7_SEC.to_string(index=False))
print('\nIDSW grupları (toplam):'); print(pd.DataFrame({a: v.sum() for a, v in E7_IDG.items()}).T)

# --- Görsel ---
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
_lab = ['noreid'] + _ra
axs[0].bar(_lab, [E7_TAB.loc[a, 'HOTA'] for a in _lab], color=['#999999', '#dd8452'] + ['#4c72b0'] * 3)
axs[0].set_ylim(min(E7_TAB.HOTA) - 0.02, max(E7_TAB.HOTA) + 0.02); axs[0].set_title('HOTA (dondurulmuş tracker, p2)')
for i, a in enumerate(_lab):
    axs[0].text(i, E7_TAB.loc[a, 'HOTA'] + 0.002, f"{E7_TAB.loc[a, 'HOTA']:.3f}", ha='center', fontsize=8)
axs[1].bar(_lab, [E7_TAB.loc[a, 'IDSW'] for a in _lab], color=['#999999', '#dd8452'] + ['#4c72b0'] * 3)
axs[1].set_title('IDSW (TrackEval)')
fig.tight_layout(); E7_FIG = E7_DIR / 'E7a_hota_idsw.png'; fig.savefig(E7_FIG, dpi=130); plt.show()

# --- Kayıt ---
E7_RESULTS = {'karar': E7_DECISION, 'secilen': E7_CHOSEN, 'tablo': E7_TAB.round(4).to_dict(orient='index'),
              'birincil': E7_PRIM.to_dict(orient='records'), 'agirlikli_vs_min': E7_VS_MIN.to_dict(orient='records'),
              'bilgi': E7_SEC.to_dict(orient='records'), 'bilgi_kolu': _ref_arm,
              'idsw_gruplari': {a: v.sum().to_dict() for a, v in E7_IDG.items()}, 'yama_sayaclari': E7_STATS,
              'on_kayit_sha256': E7_PREREG_SHA256, 'tarih': time.strftime('%Y-%m-%d %H:%M:%S')}
with open(E7_DIR / 'E7a_results.json', 'w') as fh:
    json.dump(E7_RESULTS, fh, indent=1, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
print('\nKaydedildi:', E7_DIR / 'E7a_results.json')

if MLFLOW_OK:
    try:
        if mlflow.get_experiment_by_name('TIR-JDE-E7') is None:
            mlflow.create_experiment('TIR-JDE-E7', artifact_location=(E7_DIR / 'mlartifacts').as_uri())
        mlflow.set_experiment('TIR-JDE-E7')
        for a in _lab:
            with mlflow.start_run(run_name=f'E7a_{a}'):
                mlflow.log_params({'kol': a, 'birlestirme': E7_CFG['kollar'].get(a, 'gorunus yok'),
                                   'tracker': json.dumps(FROZEN_ARGS), 'head': E7_CFG['head_md5']})
                mlflow.log_metrics({k: float(E7_TAB.loc[a, k]) for k in _cols} | {'secildi': float(a == E7_CHOSEN)})
        with mlflow.start_run(run_name='E7a_karar'):
            mlflow.log_params({'karar': E7_DECISION})
            for f in (E7_FIG, E7_DIR / 'E7a_results.json', E7_PREREG):
                mlflow.log_artifact(str(f))
        shutil.copy(LOCAL / 'mlflow_d2.db', D2_DIR / 'mlflow_d2.db')
        print('MLflow: TIR-JDE-E7 kaydedildi, veritabanı Drive\'a kopyalandı.')
    except Exception as e:
        print('MLflow kaydı başarısız (sonuçlar dosyada):', repr(e))

In [ ]:
# E7a.5) Video: noreid ve seçilen kol (yoksa w50) yan yana. Sekans/pencere sonuca göre değil, yakın çift yoğunluğuna göre seçilir.
import subprocess
_va_arm = E7_CHOSEN or 'w50'
_Qc = E3_Q[(10, True)] if 'E3_Q' in globals() else None
if _Qc is not None:
    _qd = pd.DataFrame([(q[0], int(E3_VA.loc[q[1], 'f'])) for q in _Qc], columns=['seq', 'f'])
else:                                                   # E3.3 oturumda yoksa: yakın çift GT sayısı doğrudan
    _r = []
    for s in VAL_SEQS:
        g = load_mot(DV1 / 'gt' / f'{s}.txt'); g = g[g[:, 6] == 1]
        for f in np.unique(g[:, 0]):
            gg = g[g[:, 0] == f]
            _r += [(s, int(f))] * int((nearest_other_gt(gg) <= 12.7).sum())
    _qd = pd.DataFrame(_r, columns=['seq', 'f'])
_best = (None, 1, -1)
for s, g in _qd.groupby('seq'):
    n_fr = len(glob.glob(f'{FRAMES_ROOT}/{s}/*.jpg'))
    cnt = np.bincount(g.f.values, minlength=n_fr + 1)[1:n_fr + 1]
    win = np.convolve(cnt, np.ones(min(300, n_fr)), 'valid')
    if win.max() > _best[2]:
        _best = (s, int(win.argmax()) + 1, int(win.max()))
E7_VSEQ, E7_F0, _n = _best
print(f'Video: {E7_VSEQ}, kare {E7_F0}-{E7_F0 + 299} (yakın çift sorgusu {_n})')

_gt = load_mot(DV1 / 'gt' / f'{E7_VSEQ}.txt'); _gt = _gt[_gt[:, 6] == 1]
_tr = {a: load_mot(Path(E7_TRK[a]) / f'{E7_VSEQ}.txt') for a in ('noreid', _va_arm)}
_files = sorted(glob.glob(f'{FRAMES_ROOT}/{E7_VSEQ}/*.jpg'))
_S = 2
def _color(i):
    rng = np.random.default_rng(int(i) * 7919)
    return tuple(int(c) for c in rng.integers(60, 256, 3))
def _panel(img, f, tr, label):
    img = cv2.resize(img, None, fx=_S, fy=_S, interpolation=cv2.INTER_NEAREST)
    for r in tr[tr[:, 0] == f]:
        x, y, w, h = r[2:6] * _S
        c = _color(r[1])
        cv2.rectangle(img, (int(x), int(y)), (int(x + w), int(y + h)), c, 1)
        cv2.putText(img, str(int(r[1])), (int(x), max(int(y) - 3, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, c, 1)
    for r in _gt[_gt[:, 0] == f]:
        x, y, w, h = r[2:6] * _S
        cv2.rectangle(img, (int(x) - 2, int(y) - 2), (int(x + w) + 2, int(y + h) + 2), (0, 255, 0), 1)
    cv2.putText(img, f'{label} | kare {f}', (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    return img
_raw = LOCAL / f'E7a_{E7_VSEQ}_mp4v.mp4'
_vw = None
for f in range(E7_F0, min(E7_F0 + 299, len(_files)) + 1):
    im = cv2.imread(_files[f - 1])
    fr = np.hstack([_panel(im.copy(), f, _tr['noreid'], 'gorunus yok'), _panel(im.copy(), f, _tr[_va_arm], f'gorunus + konum ({_va_arm})')])
    if _vw is None:
        _vw = cv2.VideoWriter(str(_raw), cv2.VideoWriter_fourcc(*'mp4v'), 10, (fr.shape[1], fr.shape[0]))
    _vw.write(fr)
_vw.release()
E7_VIDEO = E7_DIR / f'E7a_{E7_VSEQ}_noreid_vs_{_va_arm}.mp4'
try:
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(_raw), '-c:v', 'libx264', '-preset', 'veryfast',
                    '-crf', '23', '-pix_fmt', 'yuv420p', str(LOCAL / E7_VIDEO.name)], check=True)
    shutil.copy(LOCAL / E7_VIDEO.name, E7_VIDEO)
except Exception as e:
    shutil.copy(_raw, E7_VIDEO); print('ffmpeg yok, mp4v kaydedildi:', repr(e))
print('Video:', E7_VIDEO, '| renkler kimliğe göre; yeşil çerçeve GT')
if MLFLOW_OK:
    try:
        _r = mlflow.search_runs(experiment_names=['TIR-JDE-E7'], filter_string="tags.mlflow.runName = 'E7a_karar'",
                                order_by=['start_time DESC'], max_results=1)
        mlflow.tracking.MlflowClient().log_artifact(_r.run_id.iloc[0], str(E7_VIDEO))
        shutil.copy(LOCAL / 'mlflow_d2.db', D2_DIR / 'mlflow_d2.db')
    except Exception as e:
        print('MLflow eki başarısız:', repr(e))